# mT5-base Answer-Aware Question Generation: Fase 3 (Two-Stage Curriculum, DDP T4 x 2)

Notebook ini adalah refactoring total Fase 2 menjadi 12 cell kode yang saling berurutan. Jalankan dari Cell 1 sampai Cell 12 tanpa melompat.

### Enam blind spot Fase 2 dan perlakuannya

| Blind spot | Perbaikan di notebook ini |
|---|---|
| 1. Jawaban rumus dan angka menghasilkan pertanyaan identik | Cell 6 membuat Contrastive Triplets (rumus, hasil, variabel atau hukum pada konteks yang sama) dari 39 template dengan angka acak, sekitar 345 konteks unik per bahasa. Fase 2 hanya punya sekitar 90 konteks. Cell 10 menguji langsung dengan template yang sengaja tidak pernah dilihat saat training |
| 2. Pertanyaan buntung dan kata kerja halusinasi | Cell 5 menyediakan detektor kualitas pertanyaan (terlalu pendek, tidak berpijak pada konteks, berakhir menggantung, membocorkan jawaban). Detektor yang sama dipakai membersihkan label training (Cell 6), mengukur prediksi (Cell 10), dan memilih kandidat terbaik dari n-best (Cell 10 dan 11). Dekode juga diubah |
| 3. Regex formula rakus | Cell 5 mengganti regex dengan pemindai token. `p = 8 cm dan lebar l = 5 cm` kini terbagi menjadi `p = 8 cm` dan `l = 5 cm`. Ada 32 unit test |
| 4. Slide-style merusak data | Cell 6 membuat salinan non-destruktif, pemisah kalimat sadar singkatan, dan header yang mengikuti bahasa baris |
| 5. Tanpa EDA dan preprocessing menumpuk | Cell 3 sampai 7 dipisah per tahap. Cell 4 berisi EDA lengkap dan Cell 7 berisi EDA pasca-preprocessing |
| 6. Single-stage dan tanpa ZIP | Cell 8 dan 9 melatih dua tahap. Cell 12 membuat ZIP otomatis |

### Penyimpangan dari spesifikasi yang sengaja saya ambil karena lebih efektif

1. Dev-set pemilih checkpoint Stage 2 adalah campuran data umum, formula, dan slide. Jika dipilih dengan validasi umum saja, checkpoint yang paling mirip Stage 1 akan selalu menang dan tujuan Stage 2 tidak pernah tercapai.
2. Evaluasi formula dibagi dua. Template yang sama dengan training dengan angka baru (`seen`), dan 7 template yang tidak pernah dilihat (`unseen`). Hanya `unseen` yang benar-benar mengukur generalisasi.
3. Ada test set umum tambahan sebesar 3 persen yang tidak dipakai memilih checkpoint.
4. Label training yang buntung atau tidak berpijak pada konteks dibuang. Validasi dan test tidak disentuh.
5. Dekode memakai `length_penalty` 1.3, `min_new_tokens` 5, dan `no_repeat_ngram_size` 3, ditambah reranking n-best. Cell 10 memuat tabel ablasi supaya efeknya bisa dinilai, bukan dipercaya begitu saja.
6. Embedding empat token baru diinisialisasi dari rata-rata embedding kosakata asli. Baris tambahan milik mT5 asli tidak pernah dilatih.
7. Tag ditulis dengan spasi (`<answer> teks </answer>`), sehingga token pertama jawaban identik dengan token pada baris `answer: ...`.
8. Checkpoint disimpan tanpa optimizer state dan dihapus setiap akhir tahap. Tanpa ini dua tahap bisa melewati batas penyimpanan 20 GB Kaggle.

### Hal yang belum terverifikasi

Seluruh logika non-GPU (detektor, generator data, slide-style, ekstraksi kandidat, distractor, audit) sudah diuji dengan data tiruan. Cell training dan evaluasi model (Cell 7 sampai 10) belum pernah dijalankan pada GPU. Jika ada error di sana, kirim traceback-nya apa adanya.

## Cell 1. Instalasi dependensi

Memakai `subprocess` dan `sys.executable` sehingga pip memasang paket ke interpreter yang sama dengan kernel. Versi `transformers` dibatasi ke seri 5.10 sampai 5.17. Jika pip mengganti paket yang sudah terimpor, restart session lalu mulai dari Cell 2.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "transformers>=5.10.0,<5.18.0",
    "datasets>=4.0.0,<5.0.0",
    "accelerate>=1.5.0,<2.0.0",
    "evaluate>=0.4.5,<0.5.0",
    "rouge_score>=0.1.2",
    "sacrebleu>=2.5.0",
    "sentencepiece>=0.2.0",
    "protobuf>=4.25.0",
    "packaging>=24.0",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--disable-pip-version-check",
    *PACKAGES,
]

result = subprocess.run(command, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-2000:])
    print(result.stderr[-4000:])
    raise RuntimeError(
        "Instalasi dependensi gagal. Pastikan Internet Kaggle aktif, lalu jalankan ulang cell ini."
    )

print("Dependensi selesai dipasang.")
print("Jika pip memperbarui paket yang sudah terimpor, restart session Kaggle lalu jalankan dari Cell 2.")

try:
    import sentencepiece
    print(f"sentencepiece siap: versi {sentencepiece.__version__}")
except ImportError:
    print("[PERINGATAN] sentencepiece belum terbaca di interpreter ini. Restart kernel Kaggle jika diperlukan.")

## Cell 2. Import, seed aman DDP, verifikasi 2x T4, dan konfigurasi global

Variabel lingkungan dipasang sebelum `import torch`. GPU diverifikasi lewat `nvidia-smi` sehingga `torch.cuda` tidak disentuh. Seed hanya untuk CPU di proses utama karena seed CUDA diatur di dalam worker. Semua hyperparameter dan ukuran data berada di sini agar mudah diubah. Cell ini harus mencetak `CUDA aktif proses utama: False`.

In [ ]:
import os
import sys
import gc
import re
import json
import math
import time
import random
import shutil
import zipfile
import dataclasses
import subprocess
from collections import Counter, defaultdict

# Environment harus dipasang sebelum import torch agar allocator dan tokenizers mengikutinya.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import torch
import datasets
import evaluate
import transformers
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from packaging import version
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    from IPython.display import display, FileLink
except ImportError:  # pragma: no cover
    def display(obj):
        print(obj)
    FileLink = None

sns.set_theme(style="whitegrid")

# ------------------------------- Seed aman untuk DDP -------------------------------
# Hanya seed CPU di proses utama. torch.cuda.manual_seed_all dipanggil di dalam worker.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------- Direktori kerja -------------------------------
MODEL_NAME = "google/mt5-base"
STAGE1_DIR = "/kaggle/working/mt5-base-aqg-stage1"
STAGE2_DIR = "/kaggle/working/mt5-base-aqg-stage2"
FINAL_DIR = "/kaggle/working/mt5-base-aqg-final"
EDA_DIR = "/kaggle/working/eda_and_eval_plots"
ZIP_PATH = "/kaggle/working/mt5_base_aqg_complete_bundle.zip"
CHECKPOINT_DIR_STAGE1 = "/kaggle/working/ckpt-stage1"
CHECKPOINT_DIR_STAGE2 = "/kaggle/working/ckpt-stage2"
for directory in (STAGE1_DIR, STAGE2_DIR, FINAL_DIR, EDA_DIR):
    os.makedirs(directory, exist_ok=True)

# ------------------------------- Panjang dan split -------------------------------
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64
MAX_ANSWER_TOKENS = 64
VAL_RATIO = 0.10
TEST_RATIO = 0.03   # hold-out terpisah agar angka akhir tidak dihitung pada set yang memilih checkpoint

# ------------------------------- Hyperparameter DDP (T4 x 2) -------------------------------
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 8
TARGET_EFFECTIVE_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LABEL_SMOOTHING = 0.10

STAGE1_EPOCHS = 2
STAGE1_LR = 3e-4
STAGE1_EVAL_STEPS = 250
STAGE2_EPOCHS = 2
STAGE2_LR = 1e-4
STAGE2_EVAL_STEPS = 100

# ------------------------------- Ukuran data sintetis dan kurikulum -------------------------------
FORMULA_TRAIN_PER_LANG = 1200
FORMULA_EVAL_SEEN_PER_LANG = 120      # template sama dengan training, angka dan konteks baru
FORMULA_EVAL_UNSEEN_PER_LANG = 60     # template yang tidak pernah dilihat saat training
SLIDE_FROM_GENERAL_RATIO = 0.20
SLIDE_FROM_FORMULA_RATIO = 0.15
REPLAY_RATIO = 0.35
SLIDE_EVAL_PER_LANG = 150
SELECTION_GENERAL_ROWS = 300          # porsi val umum pada dev-set pemilih checkpoint Stage 2

# ------------------------------- Dekode -------------------------------
# length_penalty > 1 dan min_new_tokens mengurangi kecenderungan beam search memilih pertanyaan generik yang pendek.
DECODE_CONFIG = {
    "num_beams": 5,
    "length_penalty": 1.3,
    "no_repeat_ngram_size": 3,
    "min_new_tokens": 5,
    "max_new_tokens": 48,
    "early_stopping": True,
}
PHASE2_DECODE_CONFIG = {"num_beams": 4, "length_penalty": 1.0, "no_repeat_ngram_size": 2, "max_length": MAX_TARGET_LENGTH}

# ------------------------------- Verifikasi 2x T4 tanpa menyentuh torch.cuda -------------------------------
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, timeout=30,
)
gpu_lines = [line.strip() for line in smi.stdout.strip().splitlines() if line.strip()] if smi.returncode == 0 else []
NUM_GPUS = len(gpu_lines)
if NUM_GPUS != 2 or not all("T4" in line for line in gpu_lines):
    raise RuntimeError(
        f"Notebook ini ditargetkan untuk 2x Tesla T4, tetapi terdeteksi {NUM_GPUS} GPU: {gpu_lines}. "
        "Ubah Accelerator Kaggle menjadi GPU T4 x2."
    )
if version.parse(transformers.__version__) < version.parse("5.10.0"):
    raise RuntimeError(f"Transformers {transformers.__version__} terlalu lama. Jalankan ulang Cell 1 lalu restart session.")

GRADIENT_ACCUMULATION_STEPS = TARGET_EFFECTIVE_BATCH_SIZE // (PER_DEVICE_TRAIN_BATCH_SIZE * NUM_GPUS)
assert PER_DEVICE_TRAIN_BATCH_SIZE * NUM_GPUS * GRADIENT_ACCUMULATION_STEPS == TARGET_EFFECTIVE_BATCH_SIZE

print("Python           :", sys.version.split()[0])
print("PyTorch          :", torch.__version__)
print("Transformers     :", transformers.__version__)
print("Datasets         :", datasets.__version__)
print("GPU              :", " | ".join(gpu_lines))
print("Batch/GPU        :", PER_DEVICE_TRAIN_BATCH_SIZE, "| grad accumulation:", GRADIENT_ACCUMULATION_STEPS,
      "| effective batch:", TARGET_EFFECTIVE_BATCH_SIZE)
print(f"Stage 1          : {STAGE1_EPOCHS} epoch, lr={STAGE1_LR}, cosine")
print(f"Stage 2          : {STAGE2_EPOCHS} epoch, lr={STAGE2_LR}, cosine")
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())


## Cell 3. Data ingestion dan raw parsing

Memuat IDK-MRC dan SQuAD dengan fallback `refs/convert/parquet`. Konteks dinormalisasi spasinya dan offset jawaban dihitung ulang, sehingga lebih sedikit baris yang terbuang hanya karena beda spasi. Tabel ringkasan pembersihan dicetak per dataset. Setelah balancing 1:1, split dilakukan per konteks (90:10) dengan assertion nol irisan. Test hold-out 3 persen diambil dari porsi train.

In [ ]:
# =============================================================================
# Utilitas parsing. Konteks dinormalisasi spasinya dan offset answer dihitung ulang,
# sehingga lebih sedikit baris yang terbuang hanya karena perbedaan whitespace.
# =============================================================================
def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def as_records(obj):
    if obj is None:
        return []
    if isinstance(obj, dict):
        keys = list(obj.keys())
        if not keys:
            return []
        first = obj[keys[0]]
        if isinstance(first, (list, tuple, np.ndarray)):
            return [{key: obj[key][i] for key in keys} for i in range(len(first))]
        return [obj]
    if isinstance(obj, (list, tuple, np.ndarray)):
        return list(obj)
    return []


def to_int(value, default=-1):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            value = value[0] if len(value) else default
        return int(value)
    except (TypeError, ValueError):
        return default


def to_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    return bool(value)


def first_answer(answers):
    for record in as_records(answers):
        if not isinstance(record, dict):
            continue
        text = record.get("text")
        if isinstance(text, (list, tuple, np.ndarray)):
            text = text[0] if len(text) else None
        if text is None or not str(text).strip():
            continue
        return str(text), to_int(record.get("answer_start", -1))
    return None, -1


def normalized_offset(raw_context, raw_start):
    """Petakan offset pada konteks mentah ke offset pada konteks yang sudah dinormalisasi spasinya."""
    if raw_start is None or raw_start < 0:
        return -1
    return len(re.sub(r"\s+", " ", raw_context[:raw_start]).lstrip())


def resolve_answer(context, answer, start_hint):
    """Cari answer di context: offset asli, pencarian eksak, lalu case-insensitive. None bila tidak ada."""
    if 0 <= start_hint <= len(context) - len(answer) and context[start_hint:start_hint + len(answer)] == answer:
        return start_hint, answer
    position = context.find(answer)
    if position >= 0:
        return position, answer
    match = re.search(re.escape(answer), context, flags=re.IGNORECASE)
    if match:
        return match.start(), context[match.start():match.end()]
    return None


def _build_row(lang, context, raw_answer, raw_question, raw_start, raw_context, source, stats, seen):
    clean_answer = normalize_space(raw_answer)
    clean_question_text = normalize_space(raw_question)
    if not clean_answer or not clean_question_text:
        stats["empty_removed"] += 1
        return None
    resolved = resolve_answer(context, clean_answer, normalized_offset(raw_context, raw_start))
    if resolved is None:
        stats["answer_not_in_context"] += 1
        return None
    start, answer_text = resolved
    key = (context, clean_question_text, answer_text)
    if key in seen:
        stats["duplicate_removed"] += 1
        return None
    seen.add(key)
    return {"lang": lang, "context": context, "answer": answer_text, "question": clean_question_text,
            "answer_start": start, "source": source}


def parse_idk_mrc(dataset_obj):
    stats, rows, seen, contexts_seen = Counter(), [], set(), set()
    for split_name in dataset_obj.keys():
        for example in tqdm(dataset_obj[split_name], desc=f"Parsing IDK-MRC ({split_name})"):
            raw_context = example.get("context")
            if not raw_context:
                continue
            raw_context = str(raw_context)
            context = normalize_space(raw_context)
            contexts_seen.add(context)
            for qa in as_records(example.get("qas")):
                stats["total_qas"] += 1
                if not isinstance(qa, dict):
                    continue
                if to_bool(qa.get("is_impossible", False)):
                    stats["impossible_removed"] += 1
                    continue
                answer, start = first_answer(qa.get("answers"))
                if answer is None or qa.get("question") is None:
                    stats["empty_removed"] += 1
                    continue
                row = _build_row("id", context, answer, qa.get("question"), start, raw_context, "idk_mrc", stats, seen)
                if row:
                    rows.append(row)
    stats["unique_contexts"] = len(contexts_seen)
    stats["kept"] = len(rows)
    return rows, stats


def parse_squad(dataset_obj):
    stats, rows, seen, contexts_seen = Counter(), [], set(), set()
    for example in tqdm(dataset_obj, desc="Parsing SQuAD (en)"):
        stats["total_qas"] += 1
        raw_context = str(example.get("context", "") or "")
        if not raw_context.strip():
            stats["empty_removed"] += 1
            continue
        context = normalize_space(raw_context)
        contexts_seen.add(context)
        answer, start = first_answer(example.get("answers"))
        if answer is None:
            stats["empty_removed"] += 1
            continue
        row = _build_row("en", context, answer, example.get("question", ""), start, raw_context, "squad", stats, seen)
        if row:
            rows.append(row)
    stats["unique_contexts"] = len(contexts_seen)
    stats["kept"] = len(rows)
    return rows, stats


def load_dataset_safe(repo_id, **kwargs):
    errors = []
    attempts = [kwargs, {**kwargs, "revision": "refs/convert/parquet"}]
    for attempt_index, attempt in enumerate(attempts, start=1):
        print(f"    [LOAD {attempt_index}/{len(attempts)}] Mencoba {repo_id} dengan {attempt}", flush=True)
        started = time.time()
        try:
            dataset = load_dataset(repo_id, **attempt)
            print(f"    [LOAD OK] {repo_id} dalam {time.time() - started:.1f} detik.", flush=True)
            return dataset
        except Exception as exc:
            errors.append(f"{repo_id} {attempt}: {type(exc).__name__}: {str(exc)[:500]}")
            print(f"    [LOAD GAGAL] percobaan {attempt_index}: {str(exc)[:150]}", flush=True)
    raise RuntimeError("Gagal memuat dataset.\n" + "\n".join(errors))


def split_by_context(records, ratio, seed):
    """Pisahkan per konteks (bukan per baris) agar tidak ada paragraf yang muncul di dua split."""
    groups = defaultdict(list)
    for row in records:
        groups[row["context"]].append(row)
    contexts = list(groups)
    random.Random(seed).shuffle(contexts)
    target = int(round(len(records) * ratio))
    held, rest = [], []
    for context in contexts:
        (held if len(held) < target else rest).extend(groups[context])
    return rest, held

# =============================================================================
# Eksekusi: muat dataset, parsing, tabel statistik pembersihan, balancing, dan split
# =============================================================================
print("=" * 78)
print("[1/4] MEMUAT DATASET")
print("=" * 78)
raw_idk = load_dataset_safe("rifkiaputri/idk-mrc")
if not isinstance(raw_idk, dict) and not hasattr(raw_idk, "keys"):
    raw_idk = {"train": raw_idk}
raw_squad = load_dataset_safe("rajpurkar/squad", split="train")
for split_name in raw_idk.keys():
    print(f"  IDK-MRC {split_name}: {len(raw_idk[split_name]):,} konteks, kolom={raw_idk[split_name].column_names}")
print(f"  SQuAD train: {len(raw_squad):,} baris, kolom={raw_squad.column_names}")

print("\n" + "=" * 78)
print("[2/4] PARSING DAN PEMBERSIHAN")
print("=" * 78)
idk_rows, idk_stats = parse_idk_mrc(raw_idk)
squad_rows, squad_stats = parse_squad(raw_squad)

STAT_COLUMNS = ["total_qas", "impossible_removed", "empty_removed", "answer_not_in_context", "duplicate_removed", "unique_contexts", "kept"]
cleaning_table = pd.DataFrame(
    [{col: int(stats.get(col, 0)) for col in STAT_COLUMNS} for stats in (idk_stats, squad_stats)],
    index=["IDK-MRC (id)", "SQuAD (en)"],
)
print("\nRingkasan pembersihan:")
display(cleaning_table)
cleaning_table.to_csv(os.path.join(EDA_DIR, "cleaning_summary.csv"))

print("\n" + "=" * 78)
print("[3/4] BALANCING 1:1 ANTARA INDONESIA DAN INGGRIS")
print("=" * 78)
rng_balance = random.Random(SEED)
n_per_lang = min(len(idk_rows), len(squad_rows))
idk_selected = rng_balance.sample(idk_rows, n_per_lang)
squad_selected = rng_balance.sample(squad_rows, n_per_lang)
all_general = idk_selected + squad_selected
print(f"IDK-MRC dipakai: {len(idk_selected):,} | SQuAD dipakai: {len(squad_selected):,} | total: {len(all_general):,}")

print("\n" + "=" * 78)
print("[4/4] SPLIT BERDASARKAN KONTEKS: TRAIN / VAL (90:10) + TEST HOLD-OUT")
print("=" * 78)
train_general, val_general, test_general = [], [], []
for lang in ("id", "en"):
    lang_rows = [row for row in all_general if row["lang"] == lang]
    lang_train_full, lang_val = split_by_context(lang_rows, VAL_RATIO, SEED)
    lang_train, lang_test = split_by_context(lang_train_full, TEST_RATIO / (1 - VAL_RATIO), SEED + 1)
    train_general.extend(lang_train)
    val_general.extend(lang_val)
    test_general.extend(lang_test)

contexts_train = {r["context"] for r in train_general}
contexts_val = {r["context"] for r in val_general}
contexts_test = {r["context"] for r in test_general}
assert contexts_train.isdisjoint(contexts_val), "Konteks bocor antara train dan val."
assert contexts_train.isdisjoint(contexts_test), "Konteks bocor antara train dan test."
assert contexts_val.isdisjoint(contexts_test), "Konteks bocor antara val dan test."
print(f"Overlap konteks train/val/test: 0 (train={len(contexts_train):,}, val={len(contexts_val):,}, test={len(contexts_test):,})")

split_table = pd.DataFrame(
    {name: {lang: sum(r["lang"] == lang for r in rows) for lang in ("id", "en")}
     for name, rows in (("train", train_general), ("val", val_general), ("test", test_general))}
)
split_table["total"] = split_table.sum(axis=1)
display(split_table)
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())


## Cell 4. EDA lengkap pada data mentah

Statistik deskriptif panjang karakter dan kata untuk context, question, dan answer per bahasa. Audit kata tanya, distribusi posisi jawaban (histogram dan KDE), serta inspeksi baris terpendek, terpanjang, dan kebocoran jawaban pada pertanyaan ground truth. Semua grafik disimpan ke `EDA_DIR` lalu ditampilkan.

In [ ]:
# =============================================================================
# CELL 4. EDA data mentah. Semua grafik disimpan ke EDA_DIR lalu ditampilkan.
# =============================================================================
INTERROGATIVE_PATTERNS = {
    "id": [
        ("apa", r"\bapa(?:kah)?\b"), ("siapa", r"\bsiapa(?:kah)?\b"), ("kapan", r"\bkapan(?:kah)?\b"),
        ("di mana", r"\bdi\s*mana(?:kah)?\b|\bdimana(?:kah)?\b"),
        ("mengapa", r"\bmengapa(?:kah)?\b|\bkenapa(?:kah)?\b"),
        ("bagaimana", r"\bbagaimana(?:kah)?\b"), ("berapa", r"\bberapa(?:kah)?\b"),
        ("mana", r"\bmana(?:kah)?\b"),
    ],
    "en": [
        ("how many/much", r"\bhow\s+(?:many|much)\b"), ("what", r"\bwhat\b"), ("who", r"\bwh(?:o|om|ose)\b"),
        ("when", r"\bwhen\b"), ("where", r"\bwhere\b"), ("why", r"\bwhy\b"), ("how", r"\bhow\b"),
        ("which", r"\bwhich\b"),
    ],
}



def interrogative_of(question, lang):
    """Kata tanya paling awal pada pertanyaan; kembalikan label atau 'lainnya'."""
    text = str(question).casefold()
    best = None
    for label, pattern in INTERROGATIVE_PATTERNS.get(lang, []):
        match = re.search(pattern, text)
        if match and (best is None or match.start() < best[0] or (match.start() == best[0] and match.end() > best[1])):
            best = (match.start(), match.end(), label)
    return best[2] if best else "lainnya"




def save_and_show(figure, filename):
    figure.savefig(os.path.join(EDA_DIR, filename), dpi=150, bbox_inches="tight")
    plt.show()


eda_df = pd.DataFrame(train_general + val_general + test_general)[["lang", "context", "question", "answer", "answer_start", "source"]].copy()
eda_df["ctx_chars"] = eda_df["context"].str.len()
eda_df["ctx_words"] = eda_df["context"].str.split().str.len()
eda_df["q_chars"] = eda_df["question"].str.len()
eda_df["q_words"] = eda_df["question"].str.split().str.len()
eda_df["a_chars"] = eda_df["answer"].str.len()
eda_df["a_words"] = eda_df["answer"].str.split().str.len()
eda_df["answer_position"] = eda_df["answer_start"] / eda_df["context"].str.len().clip(lower=1)
eda_df["interrogative"] = [interrogative_of(q, lang) for q, lang in zip(eda_df["question"], eda_df["lang"])]

# ------------------------------ 1. Statistik deskriptif ------------------------------
print("=" * 78)
print("1. STATISTIK DESKRIPTIF (mean, median, std, min, P25, P75, P95, max)")
print("=" * 78)
descriptive_records = []
for lang in ("id", "en"):
    subset = eda_df[eda_df["lang"] == lang]
    for column in ("ctx_chars", "ctx_words", "q_chars", "q_words", "a_chars", "a_words"):
        values = subset[column]
        descriptive_records.append({
            "lang": lang, "metric": column, "mean": values.mean(), "median": values.median(), "std": values.std(),
            "min": values.min(), "p25": values.quantile(0.25), "p75": values.quantile(0.75),
            "p95": values.quantile(0.95), "max": values.max(),
        })
descriptive_table = pd.DataFrame(descriptive_records).set_index(["lang", "metric"]).round(2)
display(descriptive_table)
descriptive_table.to_csv(os.path.join(EDA_DIR, "eda_descriptive_stats.csv"))

figure, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, (column, title) in zip(axes, (("ctx_words", "Jumlah kata context"), ("q_words", "Jumlah kata pertanyaan"), ("a_words", "Jumlah kata jawaban"))):
    upper = eda_df[column].quantile(0.99)
    sns.histplot(data=eda_df[eda_df[column] <= upper], x=column, hue="lang", bins=40, element="step", stat="density", common_norm=False, ax=axis)
    axis.set_title(title)
figure.suptitle("Distribusi panjang (dipotong pada persentil 99)")
save_and_show(figure, "eda_length_distributions.png")

# ------------------------------ 2. Kata tanya ------------------------------
print("\n" + "=" * 78)
print("2. AUDIT KATA TANYA")
print("=" * 78)
interrogative_table = pd.crosstab(eda_df["interrogative"], eda_df["lang"])
display(interrogative_table)
interrogative_table.to_csv(os.path.join(EDA_DIR, "eda_interrogative_counts.csv"))
figure, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for axis, lang, title in zip(axes, ("id", "en"), ("Bahasa Indonesia", "English")):
    counts = eda_df[eda_df["lang"] == lang]["interrogative"].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axis, color="#3b7dd8")
    axis.set_title(f"Kata tanya: {title}")
    axis.set_xlabel("")
    axis.set_ylabel("jumlah pertanyaan")
    axis.tick_params(axis="x", rotation=35)
    for position, value in enumerate(counts.values):
        axis.text(position, value, f"{value / counts.sum():.0%}", ha="center", va="bottom", fontsize=8)
save_and_show(figure, "eda_interrogative_words.png")

# ------------------------------ 3. Posisi jawaban ------------------------------
print("\n" + "=" * 78)
print("3. POSISI JAWABAN DI DALAM KONTEKS (answer_start / panjang context)")
print("=" * 78)
position_table = eda_df.groupby("lang")["answer_position"].describe().round(3)
display(position_table)
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(data=eda_df, x="answer_position", hue="lang", bins=30, element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Histogram posisi jawaban")
sns.kdeplot(data=eda_df, x="answer_position", hue="lang", common_norm=False, clip=(0, 1), fill=True, ax=axes[1])
axes[1].set_title("KDE posisi jawaban")
save_and_show(figure, "eda_answer_position.png")
thirds = pd.cut(eda_df["answer_position"], [-0.001, 1 / 3, 2 / 3, 1.001], labels=["awal", "tengah", "akhir"])
display(pd.crosstab(eda_df["lang"], thirds, normalize="index").round(3))

# ------------------------------ 4. Inspeksi kualitas ------------------------------
print("\n" + "=" * 78)
print("4. INSPEKSI KUALITAS SAMPEL DAN ANOMALI")
print("=" * 78)
show_columns = ["lang", "ctx_words", "question", "answer"]
print("\nBaris dengan context terpendek:")
display(eda_df.nsmallest(3, "ctx_words")[show_columns])
print("Baris dengan context terpanjang:")
display(eda_df.nlargest(3, "ctx_words")[show_columns])
print("Pertanyaan terpendek:")
display(eda_df.nsmallest(5, "q_words")[show_columns])


def ground_truth_leaks(question, answer):
    q, a = question.casefold(), answer.casefold()
    return len(a) >= 3 and re.search(r"(?<!\w)" + re.escape(a) + r"(?!\w)", q) is not None


eda_df["gt_leak"] = [ground_truth_leaks(q, a) for q, a in zip(eda_df["question"], eda_df["answer"])]
leak_table = eda_df.groupby("lang")["gt_leak"].agg(["sum", "mean"]).rename(columns={"sum": "jumlah", "mean": "proporsi"})
print("\nKebocoran jawaban pada pertanyaan ground truth (answer tertulis mentah di dalam question):")
display(leak_table.round(4))
display(eda_df[eda_df["gt_leak"]].head(5)[show_columns])
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())


## Cell 5. Precision formula detector, answer type classifier, dan detektor kualitas pertanyaan

Detektor persamaan tidak lagi memakai satu regex yang menelan teks setelah tanda `=`. Setiap tanda relasi dicari dahulu, sisi kiri dicocokkan mundur, lalu sisi kanan dipindai token demi token dan berhenti pada kata biasa. Satuan seperti `cm` dan `m/s` ikut terambil, sedangkan `dan`, `atau`, `jika`, `and`, `or`, `if` menghentikan pemindaian. Waktu eksekusi linear terhadap panjang teks.

Pemisah kalimat mengenali singkatan (`Dr.`, `Prof.`, `U.S.`, `No.`), inisial nama, dan angka desimal, tetapi tidak menganggap variabel di akhir rumus (`... × R.`) sebagai inisial. Bagian akhir cell menjalankan 32 unit test dan menerapkan anotasi ke dataset asli beserta audit false positive.

In [ ]:
# =============================================================================
# BAGIAN A. Detektor formula presisi tinggi (tanpa regex rakus)
# =============================================================================
# Prinsip desain:
# 1. Persamaan TIDAK dicari dengan satu regex "= lalu telan sampai titik/koma".
#    Setiap tanda relasi (=, ≈, ≤, ≥, ≠) dicari lebih dulu, lalu sisi kiri (LHS)
#    dicocokkan mundur, dan sisi kanan (RHS) dipindai token demi token.
# 2. Pemindai RHS berhenti begitu bertemu kata biasa ("dan", "atau", "jika", dst.),
#    sehingga "p = 8 cm dan lebar l = 5 cm" menjadi dua span terpisah.
# 3. Semua langkah linear terhadap panjang teks, sehingga tidak ada risiko ReDoS.

GREEK_RANGE = "Α-Ωα-ω"
_LETTER_CLASS = "A-Za-z" + GREEK_RANGE
_SUBSCRIPT = r"(?:_\{?[A-Za-z0-9]{1,3}\}?|[₀-₉]+)"
_VARIABLE = "[" + _LETTER_CLASS + "]" + _SUBSCRIPT + "?"

_LHS_ALTERNATIVES = [
    r"pH|pOH",
    r"[A-Z]{2,4}",
    r"[A-Z][a-z]",
    r"d\s*\([^()\n]{1,30}\)\s*/\s*d[a-z]",
    r"d[a-z]?/d[a-z](?:\s*\([^()\n]{1,30}\))?",
    r"∫[^=\n]{1,40}?",
    _VARIABLE + r"(?:\([A-Za-z0-9, ]{1,10}\))?['′]*[²³]?(?:\^\{?-?[0-9A-Za-z]{1,3}\}?)?",
]
_LHS_RE = re.compile(r"(?<![\w'’])(?:" + "|".join(_LHS_ALTERNATIVES) + r")\s*$")
_LHS_PRODUCT_RE = re.compile(r"(?<![\w'’])" + _VARIABLE + r"[²³]?\s*[×·*]\s*$")

_TOKEN_NUMBER = re.compile(r"\d+(?:[.,]\d+)*%?")
_TOKEN_ALPHA = re.compile("[" + _LETTER_CLASS + "]+")
_TOKEN_SUBSCRIPT = re.compile(_SUBSCRIPT)
_TOKEN_SUPERSCRIPT = re.compile(r"[²³]|\^\{[^{}\n]{1,12}\}|\^-?[0-9A-Za-z]{1,3}")
_TOKEN_OPERATOR = re.compile(r"[-+*/×÷^·−]")
_TOKEN_RELATION = re.compile(r"[=≈≤≥≠]")

_FUNCTION_NAMES = {
    "sin", "cos", "tan", "cot", "sec", "csc", "log", "ln", "exp", "lim",
    "max", "min", "sqrt", "arcsin", "arccos", "arctan", "det", "mod", "sinh", "cosh", "tanh",
}

# Kata pendek bahasa Indonesia dan Inggris yang tidak boleh dianggap perkalian variabel.
_SHORT_WORDS = {
    "di", "ke", "ya", "ia", "dan", "atau", "jika", "bila", "maka", "dari", "pada", "oleh", "yang",
    "itu", "ini", "dgn", "dng", "tak", "ada", "apa", "bagi", "para", "per", "pun", "lah", "kan",
    "kah", "nya", "akan", "juga", "saat", "bisa", "lalu", "tapi", "sbg", "dll", "dsb", "dst",
    "is", "if", "or", "to", "of", "in", "on", "at", "by", "as", "an", "it", "be", "we", "he",
    "me", "my", "no", "so", "do", "up", "us", "am", "and", "the", "for", "not", "can", "may",
    "has", "had", "how", "who", "why", "but", "are", "was", "its", "all", "any", "one", "two",
    "then", "than", "that", "this", "with", "when", "where", "which", "while",
}

_UNIT_BASE = (
    r"(?:km|cm|mm|dm|nm|μm|µm|kg|mg|μg|µg|ton|ms|μs|µs|kN|kJ|kW|MW|kV|mV|mA|kΩ|MΩ|kHz|MHz|GHz|Hz|"
    r"kPa|MPa|Pa|atm|bar|mol|mmol|mL|ml|kcal|cal|keV|eV|rad|dB|menit|detik|jam|hari|bulan|tahun|"
    r"minggu|persen|rupiah|triliun|miliar|juta|ribu|dollars?|USD|IDR|euros?|seconds?|minutes?|hours?|"
    r"days?|years?|months?|meters?|metres?|grams?|liters?|litres?|units?|unit|Wb|"
    r"m|g|s|h|N|J|W|V|A|Ω|K|L|l|C|F|H|T)"
)
_UNIT_AFTER_NUMBER = re.compile(
    r"\s?(?:°[CFK]?|%|" + _UNIT_BASE + r"[²³]?(?:[/·]" + _UNIT_BASE + r"[²³]?)?)(?![A-Za-zΑ-Ωα-ω0-9_])"
)
_JUXTAPOSITION_AHEAD = re.compile(
    r"\s+[" + _LETTER_CLASS + r"](?:[²³^_]|\s+[" + _LETTER_CLASS + r"](?![" + _LETTER_CLASS + r"]))"
)
_RELATION_AHEAD = re.compile(r"\s*[=≈≤≥≠]")


def _looks_like_variable_token(word):
    """Apakah rangkaian huruf ini lebih mirip variabel/singkatan daripada kata biasa."""
    if len(word) == 1:
        return True
    if word.casefold() in _SHORT_WORDS:
        return False
    if 2 <= len(word) <= 3:
        if word[0].isupper() and word[1:].islower():
            return False
        return True
    return False


def _scan_rhs(text, start):
    """Pindai sisi kanan persamaan token demi token. Kembalikan indeks akhir yang valid atau None."""
    n = len(text)
    pos = start
    depth = 0
    previous = "start"
    previous_kind = ""
    best_end = None
    while pos < n:
        k = pos
        while k < n and text[k] in " \t":
            k += 1
        spaced = k > pos
        if k >= n:
            break
        ch = text[k]

        if ch.isdecimal():
            if previous == "operand" and spaced:
                break
            match = _TOKEN_NUMBER.match(text, k)
            pos = match.end()
            previous, previous_kind = "operand", "num"
            unit_match = _UNIT_AFTER_NUMBER.match(text, pos)
            if unit_match:
                unit_text = unit_match.group(0).strip()
                single_letter_unit = len(unit_text) == 1 and unit_text.isalpha()
                if not (single_letter_unit and _JUXTAPOSITION_AHEAD.match(text, unit_match.end())):
                    pos = unit_match.end()
                    previous_kind = "unit"
            if depth == 0:
                best_end = pos
            continue

        if ch.isalpha() or ch == "∞":
            match = _TOKEN_ALPHA.match(text, k) if ch.isalpha() else None
            word = match.group(0) if match else ch
            end_of_word = match.end() if match else k + 1
            if word.casefold() in _FUNCTION_NAMES:
                if previous == "operand":
                    break
                pos = end_of_word
                previous, previous_kind = "op", "func"
                continue
            accepted = False
            if previous == "operand":
                if not spaced:
                    accepted = len(word) <= 3 and _looks_like_variable_token(word)
                elif len(word) == 1 and not _RELATION_AHEAD.match(text, end_of_word):
                    if previous_kind in ("var", "sup", "sub", "rp"):
                        accepted = True
                    elif previous_kind in ("num", "unit"):
                        accepted = bool(_JUXTAPOSITION_AHEAD.match(text, end_of_word))
            else:
                accepted = _looks_like_variable_token(word)
            if not accepted:
                break
            pos = end_of_word
            previous, previous_kind = "operand", "var"
            if depth == 0:
                best_end = pos
            continue

        if previous == "operand" and not spaced:
            sup_match = _TOKEN_SUPERSCRIPT.match(text, k) if ch in "²³^" else None
            if sup_match:
                pos = sup_match.end()
                previous_kind = "sup"
                if depth == 0:
                    best_end = pos
                continue
            sub_match = _TOKEN_SUBSCRIPT.match(text, k) if ch in "_₀₁₂₃₄₅₆₇₈₉" else None
            if sub_match:
                pos = sub_match.end()
                previous_kind = "sub"
                if depth == 0:
                    best_end = pos
                continue

        if _TOKEN_RELATION.match(text, k):
            if previous == "operand" and depth == 0:
                pos = k + 1
                previous, previous_kind = "op", "relation"
                continue
            break

        if ch in "√∑∏∫":
            if previous == "operand":
                break
            pos = k + 1
            previous, previous_kind = "op", "func"
            continue

        if _TOKEN_OPERATOR.match(text, k):
            if ch in "+-−" and previous == "operand" and depth > 0 and k + 1 < n and text[k + 1] in ")]}":
                pos = k + 1
                previous_kind = "charge"
                continue
            if previous == "operand":
                pos = k + 1
                previous, previous_kind = "op", "op"
                continue
            if previous in ("start", "op", "lp") and ch in "+-−":
                pos = k + 1
                previous, previous_kind = "op", "unary"
                continue
            break

        if ch in "([{":
            if previous == "operand" and spaced:
                break
            depth += 1
            pos = k + 1
            previous, previous_kind = "lp", "lp"
            continue

        if ch in ")]}":
            if depth == 0 or previous != "operand":
                break
            depth -= 1
            pos = k + 1
            previous, previous_kind = "operand", "rp"
            if depth == 0:
                best_end = pos
            continue

        break
    return best_end


def _extend_lhs(text, start):
    """Perluas LHS ke kiri untuk bentuk perkalian seperti 'P × V = n × R × T'."""
    for _ in range(3):
        head = text[:start].rstrip()
        match = _LHS_PRODUCT_RE.search(head)
        if not match:
            break
        start = match.start()
    return start


def _scan_equations(text):
    spans = []
    occupied_until = -1
    for match in re.finditer(r"[=≈≤≥≠]", text):
        i = match.start()
        if i < occupied_until:
            continue
        previous_char = text[i - 1] if i > 0 else ""
        next_char = text[i + 1] if i + 1 < len(text) else ""
        if text[i] == "=" and (previous_char in "=<>!:" or next_char in "=>"):
            continue
        window_start = max(0, i - 80)
        left = text[window_start:i].rstrip()
        lhs = _LHS_RE.search(left)
        if not lhs:
            continue
        lhs_start = _extend_lhs(text, window_start + lhs.start())
        rhs_end = _scan_rhs(text, i + 1)
        if rhs_end is None:
            continue
        spans.append((lhs_start, rhs_end))
        occupied_until = rhs_end
    return spans


_LATEX_PATTERNS = [
    re.compile(r"\$\$[\s\S]{1,400}?\$\$"),
    re.compile(r"(?<![\w$])\$(?=[^\s$])[^$\n]{1,200}?(?<=[^\s$])\$(?![\w$])"),
    re.compile(r"\\\([^\n]{1,200}?\\\)"),
    re.compile(r"\\\[[\s\S]{1,300}?\\\]"),
    re.compile(
        r"\\(?:d?frac|sqrt|sum|int|prod|lim|partial|nabla|oint)\b(?:\s*[_^]\s*(?:\{[^{}]*\}|\w))*"
        r"(?:\s*\[[^\]]*\])?(?:\s*\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}){0,2}"
    ),
    re.compile(
        r"\\(?:alpha|beta|gamma|delta|theta|lambda|mu|pi|sigma|omega|rho|phi|Delta|Sigma|Omega|"
        r"cdot|times|div|pm|leq|geq|neq|approx|infty)\b"
    ),
]

_SYMBOL_PATTERNS = [
    re.compile(r"√\s*(?:\([^()\n]{1,30}\)|[A-Za-z0-9" + GREEK_RANGE + r"²³.,]{1,12})"),
    re.compile(r"[∑∏]\s*\(?[A-Za-z][^\s,;.]{0,15}"),
    re.compile(r"∫[^=\n]{0,40}?\bd[a-z]\b"),
    re.compile(r"(?<![\w])d[a-z]?/d[a-z](?:\s*\([^()\n]{1,30}\))?"),
    re.compile(r"(?<![\w])[A-Za-z]\s*\^\s*\{?-?\d+\}?"),
]

_CHEM_SPECIES = (
    r"(?:\d*(?:[A-Z][a-z]?(?:[₀-₉]+|\d*)|\([A-Za-z0-9]{1,6}\)(?:[₀-₉]+|\d*))+(?:[⁺⁻]|\^?\d*[+-])?)"
)
_CHEM_RE = re.compile(
    _CHEM_SPECIES + r"(?:\s*\+\s*" + _CHEM_SPECIES + r"){0,3}\s*(?:→|⟶|⇌|⇄|-->|->|=>)\s*"
    + _CHEM_SPECIES + r"(?:\s*\+\s*" + _CHEM_SPECIES + r"){0,3}"
)

_ARITHMETIC_RE = re.compile(
    r"(?<![\d/.,\w])(?!(?:1[5-9]|20)\d{2}\b)\d+(?:[.,]\d+)?[²³]?"
    r"(?:\s*[+*×÷^]\s*\d+(?:[.,]\d+)?[²³]?|\s+[-−]\s+\d+(?:[.,]\d+)?[²³]?)+(?![\d/\w])"
)


def _valid_chemical_match(value):
    return bool(re.search(r"[0-9₀-₉]|[A-Z][a-z]|[A-Z]{2}", value))


def detect_formula_spans(text):
    """Kembalikan daftar (start, end, teks) ekspresi matematika yang valid dan sudah digabung."""
    text = str(text)
    raw = []
    for pattern in _LATEX_PATTERNS + _SYMBOL_PATTERNS:
        for match in pattern.finditer(text):
            if match.end() > match.start():
                raw.append((match.start(), match.end()))
    for match in _CHEM_RE.finditer(text):
        if _valid_chemical_match(match.group(0)):
            raw.append((match.start(), match.end()))
    for match in _ARITHMETIC_RE.finditer(text):
        raw.append((match.start(), match.end()))
    raw.extend(_scan_equations(text))

    raw.sort()
    merged = []
    for start, end in raw:
        if merged and start < merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    spans = []
    for start, end in merged:
        while end > start and text[end - 1] in ".,;:":
            end -= 1
        if end - start >= 2:
            spans.append((start, end, text[start:end]))
    return spans


def detect_formula_content(text):
    spans = detect_formula_spans(text)
    return bool(spans), [span[2] for span in spans]


# =============================================================================
# BAGIAN B. Utilitas teks: pemisah kalimat aman singkatan dan pencari span jawaban
# =============================================================================
ABBREVIATIONS = {
    "dr", "drs", "dra", "prof", "ir", "mr", "mrs", "ms", "mss", "jr", "sr", "st", "no", "nr", "vs",
    "dll", "dsb", "dst", "dkk", "yg", "dgn", "hlm", "hal", "tgl", "gb", "fig", "eq", "al", "etc", "cf",
    "ca", "approx", "kec", "kab", "jl", "bpk", "sdr", "sdri", "hj", "kh", "rp", "vol", "ed", "inc",
    "ltd", "corp", "co", "mt", "jan", "feb", "mar", "apr", "jun", "jul", "agu", "aug", "sep", "sept",
    "okt", "oct", "nov", "des", "dec", "gen", "col", "lt", "sgt", "capt", "gov", "sen", "rep", "rev",
}

_BOUNDARY_RE = re.compile(r"([.!?]+)([\"'”’)\]]*)\s+(?=[\"'“‘(\[]?[A-ZÀ-ÝΑ-Ω0-9])")
_TOKEN_BEFORE_PERIOD_RE = re.compile(r"([A-Za-zÀ-ÿ]+(?:\.[A-Za-zÀ-ÿ]+)*|\d+)$")


def _period_is_abbreviation(prefix):
    match = _TOKEN_BEFORE_PERIOD_RE.search(prefix)
    if not match:
        return False
    token = match.group(1)
    lowered = token.casefold()
    if lowered in ABBREVIATIONS:
        return True
    if re.fullmatch(r"(?:[A-Za-z]\.)+[A-Za-z]", token):
        return True
    if len(token) == 1 and token.isalpha() and token.isupper():
        # Huruf kapital tunggal adalah inisial nama (J. K. Rowling), kecuali didahului operator/angka
        # seperti pada kalimat rumus "... = I × R. Jika ..." (R adalah variabel, bukan inisial).
        before = prefix[:match.start(1)].rstrip()
        if before and (before[-1] in "=+-−×÷*/^(<>≤≥≈" or before[-1].isdigit()):
            return False
        return True
    return False


def sentence_spans(text):
    """Pisahkan teks menjadi span kalimat tanpa memotong singkatan, inisial, atau angka desimal."""
    text = str(text)
    spans = []
    start = 0
    for match in _BOUNDARY_RE.finditer(text):
        punct = match.group(1)
        if punct[0] == "." and _period_is_abbreviation(text[start:match.start(1)]):
            continue
        end = match.end(2)
        if end > start:
            spans.append((start, end))
        start = match.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans


def split_sentences(text):
    text = str(text)
    return [text[s:e].strip() for s, e in sentence_spans(text) if text[s:e].strip()]


def ends_with_abbreviation(sentence):
    sentence = str(sentence).rstrip()
    if not sentence.endswith("."):
        return False
    return _period_is_abbreviation(sentence[:-1])


def find_exact_spans(context, needle):
    """Semua kemunculan needle yang tidak menempel pada huruf/angka lain (batas kata sadar simbol)."""
    if not needle:
        return []
    pattern = re.compile(r"(?<![\w,.])" + re.escape(needle) + r"(?![\w²³])")
    return [(m.start(), m.end()) for m in pattern.finditer(context)]


def locate_answer(context, answer, answer_start=-1):
    """Cari (start, end) answer di context: offset asli, lalu eksak, lalu case-insensitive."""
    context, answer = str(context), str(answer)
    if not answer:
        return None
    try:
        start = int(answer_start)
    except (TypeError, ValueError):
        start = -1
    if 0 <= start <= len(context) - len(answer) and context[start:start + len(answer)] == answer:
        return start, start + len(answer)
    position = context.find(answer)
    if position >= 0:
        return position, position + len(answer)
    match = re.search(re.escape(answer), context, flags=re.IGNORECASE)
    if match:
        return match.start(), match.end()
    return None


# =============================================================================
# BAGIAN C. Klasifikasi tipe jawaban: date, number, formula, text
# =============================================================================
_MONTHS = (
    "januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember|"
    "january|february|march|may|june|july|august|october|december|"
    "jan|feb|mar|apr|jun|jul|agu|aug|sep|sept|okt|oct|nov|des|dec"
)
_YEAR = r"(?:1[0-9]\d{2}|20\d{2})"
_DATE_FULL_PATTERNS = [
    re.compile(r"(?:0?[1-9]|[12]\d|3[01])[-/.](?:0?[1-9]|1[0-2])[-/.]" + _YEAR),
    re.compile(_YEAR + r"[-/.](?:0?[1-9]|1[0-2])[-/.](?:0?[1-9]|[12]\d|3[01])"),
    re.compile(r"(?:\d{1,2}\s+)?(?:" + _MONTHS + r")\.?\s+(?:\d{1,2},\s*)?" + _YEAR, re.IGNORECASE),
    re.compile(_YEAR),
    re.compile(_YEAR + r"\s*[-–]\s*" + _YEAR),
    re.compile(r"abad\s+ke-?\s*(?:[IVXLC]+|\d+)", re.IGNORECASE),
    re.compile(r"\d{1,2}(?:st|nd|rd|th)\s+century", re.IGNORECASE),
]
_NUMBER_FIRST_TOKEN = re.compile(
    r"[-+−]?(?:\d+(?:[.,]\d+)*(?:[-–]\d+(?:[.,]\d+)*)?|\d+/\d+)(%|°[CFK]?|[A-Za-zµμΩ][A-Za-zµμΩ/²³·^\-0-9]{0,11})?"
)
_NUMBER_TRAILING_TOKEN = re.compile(r"[A-Za-zµμΩ°%][A-Za-zµμΩ/²³·^\-]{0,14}")
_ORDINAL = re.compile(r"\d+(?:st|nd|rd|th)", re.IGNORECASE)


def _looks_like_date(value):
    return any(pattern.fullmatch(value) for pattern in _DATE_FULL_PATTERNS)


def _looks_like_number(value):
    tokens = value.split()
    if not tokens or len(tokens) > 3:
        return False
    first = tokens[0]
    if _ORDINAL.fullmatch(first):
        return False
    if not _NUMBER_FIRST_TOKEN.fullmatch(first):
        return False
    return all(_NUMBER_TRAILING_TOKEN.fullmatch(token) for token in tokens[1:])


def _is_bare_expression(value):
    """Ekspresi tanpa tanda '=' seperti '1/2 × m × v²' atau 'm/V' (seluruh string valid sebagai RHS)."""
    if not re.search(r"[-+*/×÷^·−]", value):
        return False
    if not re.search(r"[0-9A-Z²³^" + GREEK_RANGE + r"×÷√∑]", value):
        return False
    return _scan_rhs(value, 0) == len(value)


def _looks_like_formula(value):
    if re.search(r"\\[A-Za-z]+|\$", value):
        return True
    spans = detect_formula_spans(value)
    if spans:
        covered = sum(end - start for start, end, _ in spans)
        if covered / max(1, len(value)) >= 0.60:
            return True
    return _is_bare_expression(value)


def answer_type(answer):
    """Klasifikasi tipe jawaban: date, number, formula, atau text."""
    value = normalize_space(answer)
    if not value:
        return "text"
    if _looks_like_date(value):
        return "date"
    if _looks_like_number(value):
        return "number"
    if _looks_like_formula(value):
        return "formula"
    return "text"


# =============================================================================
# BAGIAN D. Detektor kualitas pertanyaan (dipakai ulang di Cell 6, 10, dan 11)
# =============================================================================
# Detektor yang sama dipakai untuk tiga hal supaya definisi "pertanyaan buruk" konsisten:
# 1. membersihkan label training yang buntung atau tidak berpijak pada context,
# 2. mengukur kualitas hasil prediksi saat evaluasi,
# 3. memilih kandidat terbaik saat inferensi (rerank n-best).
STOPWORDS = {
    "id": set("""
        yang dan di ke dari untuk pada dengan adalah itu ini dalam tidak akan atau juga oleh sebagai karena
        dapat bisa merupakan jika maka saat yaitu sebuah suatu para antara lebih telah sudah ada mereka kita
        kami saya serta tetapi namun hanya setiap apa apakah siapa siapakah kapan kapankah dimana mana manakah
        mengapa kenapa bagaimana bagaimanakah berapa berapakah tersebut tersebutlah ialah sang si nya lah kah
        pun sama sekali sangat semua seluruh beberapa banyak dia ia pernah masih lagi sebelum sesudah setelah
        selama sejak hingga sampai bagi tentang terhadap melalui secara bukan belum harus perlu mampu boleh
    """.split()),
    "en": set("""
        the of and to in is are was were that for with as by on at from this which be or an it its has have had
        can will not if then when where than their these those what who whom whose why how which a do does did
        been being about into over after before during between under against while there here also such
        more most other some any each both few many much own same so too very just only
    """.split()),
}
DANGLING_ENDINGS = {
    "id": {"di", "ke", "dari", "pada", "untuk", "dengan", "yang", "adalah", "oleh", "dan", "atau", "sebagai",
           "dalam", "karena", "jika", "saat", "agar", "bahwa", "antara", "terhadap"},
    "en": {"of", "in", "to", "the", "a", "an", "for", "by", "was", "is", "are", "were", "and", "or", "on", "at",
           "with", "from", "as", "that", "which", "than", "between"},
}
SPECIAL_MARKERS = ("<answer>", "</answer>", "<formula>", "</formula>", "<extra_id", "<unk>", "<pad>", "</s>", "⁇")

_WORD_RE = re.compile(r"[0-9A-Za-zÀ-ÿΑ-Ωα-ω]+(?:[-'’][0-9A-Za-zÀ-ÿ]+)*")


def word_tokens(text):
    return _WORD_RE.findall(str(text).casefold())


def light_stem(word, lang):
    """Stemmer ringan seadanya: cukup untuk mencocokkan 'dibangun' dengan 'membangun'."""
    w = str(word)
    if lang == "id":
        for suffix in ("nya", "kah", "lah", "pun"):
            if w.endswith(suffix) and len(w) - len(suffix) >= 4:
                w = w[:-len(suffix)]
                break
        for suffix in ("kan", "an"):
            if w.endswith(suffix) and len(w) - len(suffix) >= 4:
                w = w[:-len(suffix)]
                break
        for prefix in ("meng", "meny", "peng", "peny", "mem", "men", "pem", "pen", "ber", "ter", "per", "me", "pe", "di", "ke", "se"):
            if w.startswith(prefix) and len(w) - len(prefix) >= 4:
                w = w[len(prefix):]
                break
        return w
    for suffix in ("ing", "ed", "es", "s", "ly"):
        if w.endswith(suffix) and len(w) - len(suffix) >= 4:
            return w[:-len(suffix)]
    return w


def content_tokens(text, lang):
    stop = STOPWORDS.get(lang, set())
    result = []
    for token in word_tokens(text):
        if token in stop:
            continue
        if len(token) < 3 and not any(c.isdigit() for c in token):
            continue
        result.append(token)
    return result


def grounding_report(question, context, lang):
    """Hitung berapa kata isi pada pertanyaan yang tidak berpijak pada context."""
    q_tokens = content_tokens(question, lang)
    context_tokens = set(word_tokens(context))
    context_stems = {light_stem(token, lang) for token in context_tokens}
    long_stems = [stem for stem in context_stems if len(stem) >= 4]
    ungrounded = []
    for token in q_tokens:
        stem = light_stem(token, lang)
        if token in context_tokens or stem in context_stems:
            continue
        if len(stem) >= 4 and any(cs.startswith(stem) or stem.startswith(cs) for cs in long_stems):
            continue
        ungrounded.append(token)
    return {
        "n_content": len(q_tokens), "n_ungrounded": len(ungrounded),
        "overlap": len(q_tokens) - len(ungrounded), "ungrounded": ungrounded,
    }


def answer_leaks(question, answer):
    """True bila string jawaban muncul mentah di dalam pertanyaan."""
    q = normalize_space(question).casefold()
    a = normalize_space(answer).casefold()
    if len(a) < 2:
        return False
    if re.search(r"(?<![\w])" + re.escape(a) + r"(?![\w])", q):
        return True
    if a.replace(" ", "") and a.replace(" ", "") in q.replace(" ", "") and len(a.replace(" ", "")) >= 5:
        return True
    a_tokens = word_tokens(a)
    if len(a_tokens) >= 4:
        q_tokens = set(word_tokens(q))
        if sum(token in q_tokens for token in a_tokens) / len(a_tokens) >= 0.8:
            return True
    return False


def clean_question(text):
    """Rapikan keluaran pertanyaan: spasi, tanda tanya, huruf kapital awal, marker khusus."""
    q = str(text)
    for marker in ("<answer>", "</answer>", "<formula>", "</formula>", "<unk>", "<pad>", "</s>", "⁇"):
        q = q.replace(marker, " ")
    q = re.sub(r"<extra_id_\d+>", " ", q)
    q = normalize_space(q)
    q = re.sub(r"\s+([?.!,;:])", r"\1", q)
    q = re.sub(r"\?{2,}", "?", q)
    if not q:
        return q
    if q[-1] in ".!":
        q = q[:-1] + "?"
    elif q[-1] != "?":
        q = q + "?"
    return q[0].upper() + q[1:]


def question_quality(question, context, answer, lang, strict=True):
    """Kembalikan (ok, alasan_gagal). strict=False hanya untuk membersihkan label training."""
    reasons = []
    q = str(question)
    if not normalize_space(q):
        return False, ["empty"]
    if any(marker in q for marker in SPECIAL_MARKERS) or re.search(r"<extra_id_\d+>", q):
        reasons.append("special_token")
    tokens = word_tokens(q)
    if len(tokens) < 3:
        reasons.append("too_short")
    grounding = grounding_report(q, context, lang)
    if grounding["n_content"] >= 1 and grounding["overlap"] == 0:
        reasons.append("no_grounding")
    if strict:
        if tokens and tokens[-1] in DANGLING_ENDINGS.get(lang, set()):
            reasons.append("dangling_end")
        if grounding["n_content"] >= 3 and grounding["n_ungrounded"] / grounding["n_content"] > 0.5:
            reasons.append("weak_grounding")
        if answer_leaks(q, answer):
            reasons.append("leak")
        bigrams = list(zip(tokens, tokens[1:]))
        if len(bigrams) != len(set(bigrams)) or (tokens and max(Counter(tokens).values()) >= 3):
            reasons.append("repetition")
        if len(tokens) > 40:
            reasons.append("too_long")
    return len(reasons) == 0, reasons


_ALIGNMENT_CUES = {
    "date": {
        "id": ["kapan", "tahun berapa", "pada tahun", "tanggal", "bulan apa", "abad", "tahun"],
        "en": ["when", "what year", "which year", "what date", "which century", "what century", "in which year"],
    },
    "number": {
        "id": ["berapa", "seberapa"],
        "en": ["how many", "how much", "how long", "how far", "how large", "how big", "how old", "how tall",
               "how fast", "how high", "how heavy", "what percentage", "what percent", "what number",
               "what value", "what amount", "value of"],
    },
    "formula": {
        "id": ["rumus", "persamaan", "formula", "dirumuskan", "dinyatakan", "bentuk matematis", "hubungan matematis"],
        "en": ["formula", "equation", "expression", "mathematical relationship", "is written", "expressed"],
    },
}
_UNIT_LIKE_ANSWER = re.compile(r"\d\s?(?:%|°|[A-Za-zµμΩ]{1,12}(?:[/·][A-Za-z]+)?[²³]?)\b|\d\s?[²³%°]")


def type_alignment(question, answer, lang, kind=None):
    """True/False bila bentuk pertanyaan konsisten dengan tipe jawaban; None bila tidak berlaku (text)."""
    q = " " + normalize_space(question).casefold() + " "
    kind = kind or answer_type(answer)
    if kind not in _ALIGNMENT_CUES:
        return None
    cues = _ALIGNMENT_CUES[kind].get(lang, [])
    if any(cue in q for cue in cues):
        return True
    if kind == "number" and lang == "en" and _UNIT_LIKE_ANSWER.search(str(answer)):
        return bool(re.search(r"\bwhat\b.*\b(?:is|was|are|were)\b", q))
    return False


# =============================================================================
# Unit test: kasus uji sulit untuk detektor formula dan klasifikasi tipe jawaban
# =============================================================================
FORMULA_TEST_CASES = [
    ("Jika panjang p = 8 cm dan lebar l = 5 cm, maka luasnya adalah 40 cm².", ["p = 8 cm", "l = 5 cm"]),
    ("Luas persegi panjang dihitung dengan rumus A = p × l.", ["A = p × l"]),
    ("Energi kinetik E_k = 1/2 × m × v².", ["E_k = 1/2 × m × v²"]),
    ("Kecepatan v = s/t dan massa jenis ρ = m/V.", ["v = s/t", "ρ = m/V"]),
    ("Teorema Pythagoras menyatakan c² = a² + b² pada segitiga siku-siku.", ["c² = a² + b²"]),
    ("Fungsi f(x) = x² + 2x + 1 memiliki turunan.", ["f(x) = x² + 2x + 1"]),
    ("Aturan pangkat: d/dx (x^n) = n × x^(n-1).", ["d/dx (x^n) = n × x^(n-1)"]),
    ("∫ x^n dx = x^(n+1) / (n + 1) + C", ["∫ x^n dx = x^(n+1) / (n + 1) + C"]),
    ("$\\frac{d}{dx} x^{n} = n x^{n-1}$", ["$\\frac{d}{dx} x^{n} = n x^{n-1}$"]),
    ("If current I = 2 A and resistance R = 6 Ω, the voltage is 12 V.", ["I = 2 A", "R = 6 Ω"]),
    ("Where x = 5 and y = 6, then z = x + y.", ["x = 5", "y = 6", "z = x + y"]),
    ("Pembelajaran berlangsung di ruang kelas pada pagi hari.", []),
    ("Candi Borobudur ditetapkan sebagai Situs Warisan Dunia UNESCO pada tahun 1991.", []),
    ("Perang berlangsung pada 1914-1918 dan menewaskan jutaan orang.", []),
    ("Harganya $5 dan $10 untuk dua barang.", []),
    ("pH = -log[H+]", ["pH = -log[H+]"]),
    ("2H₂ + O₂ → 2H₂O terjadi saat pembakaran.", ["2H₂ + O₂ → 2H₂O"]),
    ("σ = √(σ²) dan μ = ∑x / n", ["σ = √(σ²)", "μ = ∑x / n"]),
    ("P × V = n × R × T", ["P × V = n × R × T"]),
    ("Dr. Smith lahir pada 3.14 detik.", []),
]
ANSWER_TYPE_TEST_CASES = [
    ("1991", "date"), ("12 Agustus 1945", "date"), ("abad ke-9", "date"), ("40 cm²", "number"), ("12 V", "number"),
    ("10 m/s²", "number"), ("12th", "text"), ("A = p × l", "formula"), ("1/2 × m × v²", "formula"),
    ("p = 8 cm dan lebar l = 5 cm", "text"), ("Hukum Ohm", "text"), ("Dinasti Syailendra", "text"),
]

print("=" * 78)
print("UNIT TEST DETEKTOR FORMULA")
print("=" * 78)
failures = []
for index, (sentence, expected) in enumerate(FORMULA_TEST_CASES, start=1):
    found = [span[2] for span in detect_formula_spans(sentence)]
    status = "OK  " if found == expected else "GAGAL"
    if found != expected:
        failures.append((sentence, expected, found))
    print(f"[{status}] #{index:02d} {sentence}\n          -> {found}")
print("\n" + "=" * 78)
print("UNIT TEST answer_type()")
print("=" * 78)
for value, expected in ANSWER_TYPE_TEST_CASES:
    actual = answer_type(value)
    status = "OK  " if actual == expected else "GAGAL"
    if actual != expected:
        failures.append((value, expected, actual))
    print(f"[{status}] {value!r:34} -> {actual}")
assert not failures, f"{len(failures)} kasus uji gagal: {failures}"
print(f"\nSemua {len(FORMULA_TEST_CASES) + len(ANSWER_TYPE_TEST_CASES)} kasus uji lolos.")

# =============================================================================
# Terapkan ke dataset: anotasi answer_type, has_formula, formula_spans
# =============================================================================
def annotate_rows(rows):
    for row in rows:
        spans = [span[2] for span in detect_formula_spans(row["context"])]
        row["answer_type"] = answer_type(row["answer"])
        row["has_formula"] = bool(spans)
        row["formula_spans"] = json.dumps(spans[:12], ensure_ascii=False)
        row["ctx_format"] = "prose"
    return rows


for name, rows in (("train_general", train_general), ("val_general", val_general), ("test_general", test_general)):
    annotate_rows(rows)

print("\n" + "=" * 78)
print("STATISTIK KONTEKS BERFORMULA PADA DATASET ASLI")
print("=" * 78)
formula_stats = []
for source_name in ("idk_mrc", "squad"):
    unique_contexts = {}
    for row in train_general + val_general + test_general:
        if row["source"] == source_name:
            unique_contexts[row["context"]] = row["has_formula"]
    total = len(unique_contexts)
    with_formula = sum(unique_contexts.values())
    formula_stats.append({"dataset": source_name, "konteks_unik": total, "konteks_berformula": with_formula,
                          "persen": round(100 * with_formula / max(1, total), 2)})
display(pd.DataFrame(formula_stats))

all_spans_sample = []
for row in train_general:
    if row["has_formula"]:
        all_spans_sample.extend(json.loads(row["formula_spans"]))
rng_sample = random.Random(SEED)
print("\nAudit false positive: 15 span formula acak yang terdeteksi pada data asli")
for span_text in rng_sample.sample(all_spans_sample, min(15, len(all_spans_sample))):
    print("   ", repr(span_text))

type_table = pd.crosstab(pd.Series([r["answer_type"] for r in train_general], name="answer_type"),
                         pd.Series([r["lang"] for r in train_general], name="lang"))
print("\nKomposisi answer_type pada train_general:")
display(type_table)
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())


## Cell 6. Contrastive formula synthesis dan safe slide-style augmentation

39 template lintas geometri, aljabar, kalkulus (teks polos dan LaTeX), statistika, fisika, kimia, dan keuangan. Tujuh template disembunyikan dari training untuk menguji generalisasi. Setiap konteks menghasilkan beberapa target berbeda (rumus, hasil, nilai variabel, nama hukum) dengan pertanyaan yang spesifik pada target masing-masing. Angka dibangkitkan acak sehingga konteks tidak berulang.

Slide-style dibuat sebagai salinan baru. Jawaban harus tetap utuh dan offset dihitung ulang, bila gagal baris dilewati. Cell ini juga menyusun kurikulum: Stage 1 berisi prosa bersih, Stage 2 berisi formula kontras, slide-style, dan replay buffer 35 persen dari prosa.

In [ ]:
# =============================================================================
# BAGIAN A. Utilitas pemformat angka dan daftar spesifikasi 39 template formula
# =============================================================================
def fmt_num(value, lang):
    """Format angka: desimal koma untuk Indonesia, titik untuk Inggris, tanpa pemisah ribuan."""
    value = round(float(value), 6)
    if abs(value - round(value)) < 1e-9:
        text = str(int(round(value)))
    else:
        text = f"{value:.6f}".rstrip("0").rstrip(".")
    return text.replace(".", ",") if lang == "id" else text


def fmt_with_unit(value, unit, lang):
    number = fmt_num(value, lang)
    if not unit:
        return number
    return number + unit if unit == "%" else f"{number} {unit}"


def join_items(items, lang):
    items = list(items)
    if len(items) <= 1:
        return "".join(items)
    word = "dan" if lang == "id" else "and"
    if len(items) == 2:
        return f"{items[0]} {word} {items[1]}"
    separator = ", "
    return separator.join(items[:-1]) + f" {word} " + items[-1]


def _v(symbol, name_id, name_en, unit_id="", unit_en=None):
    return {"sym": symbol, "id": name_id, "en": name_en, "uid": unit_id, "uen": unit_id if unit_en is None else unit_en}


def _law(name_id, name_en, q_id, q_en):
    return {"id": name_id, "en": name_en, "q_id": q_id, "q_en": q_en}


FORMULA_SPECS = []


def _add(**kwargs):
    kwargs.setdefault("consts", [])
    kwargs.setdefault("law", None)
    kwargs.setdefault("style", "plain")
    kwargs.setdefault("held_out", False)
    kwargs.setdefault("obj_id", "")
    kwargs.setdefault("obj_en", "")
    kwargs.setdefault("result_unit_id", "")
    kwargs.setdefault("result_unit_en", None)
    if kwargs["result_unit_en"] is None:
        kwargs["result_unit_en"] = kwargs["result_unit_id"]
    FORMULA_SPECS.append(kwargs)


_LAW_OHM = _law("Hukum Ohm", "Ohm's law",
    ["Hukum apa yang menghubungkan tegangan, kuat arus, dan hambatan?",
     "Apa nama hukum yang dinyatakan oleh persamaan tersebut?",
     "Hukum manakah yang dipakai untuk menentukan tegangan pada rangkaian ini?"],
    ["Which law relates voltage, current, and resistance?",
     "What is the name of the law expressed by this equation?",
     "Which law is used to determine the voltage in this circuit?"])
_LAW_NEWTON = _law("Hukum II Newton", "Newton's second law",
    ["Hukum apa yang menghubungkan gaya, massa, dan percepatan?",
     "Apa nama hukum yang menjadi dasar persamaan gaya tersebut?",
     "Hukum manakah yang digunakan untuk menghitung gaya pada contoh ini?"],
    ["Which law relates force, mass, and acceleration?",
     "What is the name of the law behind this force equation?",
     "Which law is used to calculate the force in this example?"])
_LAW_PYTHAGORAS = _law("Teorema Pythagoras", "Pythagoras' theorem",
    ["Teorema apa yang menghubungkan sisi-sisi segitiga siku-siku?",
     "Apa nama teorema yang dinyatakan oleh persamaan tersebut?",
     "Teorema manakah yang digunakan untuk mencari sisi miring?"],
    ["Which theorem relates the sides of a right triangle?",
     "What is the name of the theorem expressed by this equation?",
     "Which theorem is used to find the hypotenuse?"])
_LAW_HOOKE = _law("Hukum Hooke", "Hooke's law",
    ["Hukum apa yang menjelaskan hubungan gaya pegas dengan pertambahan panjang?",
     "Apa nama hukum yang dinyatakan oleh persamaan tersebut?",
     "Hukum manakah yang dipakai untuk menghitung gaya pegas pada contoh ini?"],
    ["Which law describes the relationship between spring force and extension?",
     "What is the name of the law expressed by this equation?",
     "Which law is used to calculate the spring force in this example?"])
_LAW_GAS = _law("Hukum Gas Ideal", "Ideal Gas Law",
    ["Hukum apa yang menghubungkan tekanan, volume, jumlah mol, dan suhu gas?",
     "Apa nama hukum yang dinyatakan oleh persamaan tersebut?",
     "Hukum manakah yang digunakan untuk menghitung tekanan gas pada contoh ini?"],
    ["Which law relates the pressure, volume, amount, and temperature of a gas?",
     "What is the name of the law expressed by this equation?",
     "Which law is used to calculate the gas pressure in this example?"])
_LAW_POWER = _law("Aturan Pangkat", "Power Rule",
    ["Aturan apa yang digunakan untuk menurunkan x pangkat n?",
     "Apa nama aturan yang dinyatakan oleh persamaan tersebut?",
     "Aturan manakah yang dipakai untuk mencari turunan pada contoh ini?"],
    ["Which rule is used to differentiate x to the power n?",
     "What is the name of the rule expressed by this equation?",
     "Which rule is used to find the derivative in this example?"])
_LAW_POWER_INT = _law("Aturan Pangkat Integral", "Power Rule for Integration",
    ["Aturan apa yang digunakan untuk mengintegralkan x pangkat n?",
     "Apa nama aturan yang dinyatakan oleh persamaan tersebut?",
     "Aturan manakah yang dipakai untuk menghitung integral pada contoh ini?"],
    ["Which rule is used to integrate x to the power n?",
     "What is the name of the rule expressed by this equation?",
     "Which rule is used to compute the integral in this example?"])

_PYTHAGOREAN_PAIRS = [(3, 4), (5, 12), (8, 15), (7, 24), (20, 21), (6, 8), (9, 12), (12, 16), (15, 20), (9, 40)]


def _sample_pythagoras(r):
    a, b = r.choice(_PYTHAGOREAN_PAIRS)
    k = r.choice([1, 1, 2, 3]) if max(a, b) <= 24 else 1
    a, b = a * k, b * k
    return {"a": a, "b": b} if r.random() < 0.5 else {"a": b, "b": a}


def _sample_gradient(r):
    x1, dx, slope, y1 = r.randint(0, 5), r.randint(1, 6), r.randint(1, 8), r.randint(0, 9)
    return {"x_1": x1, "x_2": x1 + dx, "y_1": y1, "y_2": y1 + slope * dx}


def _sample_density(r):
    volume, density = r.choice([10, 20, 25, 40, 50, 100]), r.choice([0.5, 0.8, 1.2, 1.5, 2, 2.5, 2.7, 7.8])
    return {"m": round(density * volume, 2), "V": volume}


def _sample_speed(r):
    time_s, speed = r.choice([2, 4, 5, 8, 10, 20, 25]), r.randint(2, 30)
    return {"s": speed * time_s, "t": time_s}


def _sample_power(r):
    power, seconds = r.choice([5, 10, 20, 25, 40, 50, 100]), r.choice([2, 4, 5, 10, 20, 30])
    return {"W": power * seconds, "t": seconds}


def _sample_pressure(r):
    area, pressure = r.choice([0.5, 1, 2, 4, 5, 10]), r.choice([50, 100, 150, 200, 250, 300, 500])
    return {"F": round(pressure * area, 2), "A": area}


def _sample_moles(r):
    molar, moles = r.choice([18, 28, 32, 40, 44, 58, 98, 100]), r.choice([0.25, 0.5, 1, 1.5, 2, 3, 5])
    return {"m": round(moles * molar, 2), "M_r": molar}


def _sample_molarity(r):
    volume, molarity = r.choice([0.5, 1, 2, 2.5, 4, 5]), r.choice([0.1, 0.2, 0.5, 1, 2])
    return {"n": round(molarity * volume, 3), "V": volume}


def _sample_probability(r):
    total = r.choice([4, 5, 8, 10, 20, 25, 40, 50])
    return {"n(A)": r.randint(1, total - 1), "n(S)": total}


def _sample_mean(r):
    n, mean = r.randint(4, 15), r.randint(40, 95)
    return {"∑x": n * mean, "n": n}


def _sample_variance(r):
    n, variance = r.choice([4, 5, 8, 10, 20, 25]), r.randint(2, 15)
    return {"∑(x - μ)²": n * variance, "n": n}


def _sample_zscore(r):
    mean, sigma, k = r.randint(50, 80), r.randint(2, 10), r.choice([-2, -1.5, -1, -0.5, 0.5, 1, 1.5, 2, 2.5, 3])
    return {"x": round(mean + k * sigma, 2), "μ": mean, "σ": sigma}


def _sample_breakeven(r):
    margin, variable_cost, quantity = r.choice([500, 1000, 2000, 2500, 5000]), r.choice([1000, 1500, 2000, 3000, 4000]), r.randint(2, 20) * 10
    return {"FC": margin * quantity, "P": variable_cost + margin, "VC": variable_cost}


def _sample_roi(r):
    investment, percent = r.choice([1, 2, 4, 5, 8]) * 1000000, r.choice([5, 10, 15, 20, 25, 30, 40])
    return {"L": investment * percent // 100, "I": investment}


def _sample_ph(r):
    return {"[H+]": round(10 ** -r.randint(1, 5), 6)}


# ---- Geometri dan aljabar ----
_add(key="luas_persegi_panjang", domain="geometri", formula="A = p × l",
     concept_id="luas persegi panjang", concept_en="the area of a rectangle",
     target_id="luas persegi panjang", target_en="area of the rectangle",
     vars=[_v("p", "panjang", "length", "cm"), _v("l", "lebar", "width", "cm")],
     obj_id="persegi panjang tersebut", obj_en="of the rectangle", result_unit_id="cm²",
     sample=lambda r: {"p": r.randint(3, 25), "l": r.randint(2, 18)}, compute=lambda v: v["p"] * v["l"])
_add(key="luas_segitiga", domain="geometri", formula="A = 1/2 × a × t",
     concept_id="luas segitiga", concept_en="the area of a triangle",
     target_id="luas segitiga", target_en="area of the triangle",
     vars=[_v("a", "alas", "base", "cm"), _v("t", "tinggi", "height", "cm")],
     obj_id="segitiga tersebut", obj_en="of the triangle", result_unit_id="cm²",
     sample=lambda r: {"a": r.randint(2, 30), "t": r.randint(2, 20)}, compute=lambda v: 0.5 * v["a"] * v["t"])
_add(key="pythagoras", domain="geometri", formula="c² = a² + b²", law=_LAW_PYTHAGORAS,
     concept_id="panjang sisi miring segitiga siku-siku", concept_en="the length of the hypotenuse of a right triangle",
     target_id="panjang sisi miring", target_en="length of the hypotenuse",
     vars=[_v("a", "sisi tegak", "vertical leg", "cm"), _v("b", "sisi alas", "horizontal leg", "cm")],
     obj_id="segitiga siku-siku tersebut", obj_en="of the right triangle", result_unit_id="cm",
     sample=_sample_pythagoras, compute=lambda v: math.sqrt(v["a"] ** 2 + v["b"] ** 2))
_add(key="keliling_lingkaran", domain="geometri", formula="K = 2 × π × r",
     concept_id="keliling lingkaran", concept_en="the circumference of a circle",
     target_id="keliling lingkaran", target_en="circumference of the circle",
     vars=[_v("r", "jari-jari", "radius", "cm")], consts=[("π", "3,14", "3.14", "nilai pi", "the value of pi")],
     obj_id="lingkaran tersebut", obj_en="of the circle", result_unit_id="cm",
     sample=lambda r: {"r": r.choice([4, 5, 6, 7, 8, 10, 12, 14, 15, 20, 21, 25, 35, 50])},
     compute=lambda v: 2 * 3.14 * v["r"])
_add(key="luas_lingkaran", domain="geometri", formula="L = π × r²",
     concept_id="luas lingkaran", concept_en="the area of a circle",
     target_id="luas lingkaran", target_en="area of the circle",
     vars=[_v("r", "jari-jari", "radius", "cm")], consts=[("π", "3,14", "3.14", "nilai pi", "the value of pi")],
     obj_id="lingkaran tersebut", obj_en="of the circle", result_unit_id="cm²",
     sample=lambda r: {"r": r.choice([2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 15, 20, 21])},
     compute=lambda v: 3.14 * v["r"] ** 2)
_add(key="volume_balok", domain="geometri", formula="V = p × l × t",
     concept_id="volume balok", concept_en="the volume of a box",
     target_id="volume balok", target_en="volume of the box",
     vars=[_v("p", "panjang", "length", "cm"), _v("l", "lebar", "width", "cm"), _v("t", "tinggi", "height", "cm")],
     obj_id="balok tersebut", obj_en="of the box", result_unit_id="cm³",
     sample=lambda r: {"p": r.randint(3, 20), "l": r.randint(2, 15), "t": r.randint(2, 12)},
     compute=lambda v: v["p"] * v["l"] * v["t"])
_add(key="suku_aritmetika", domain="aljabar", formula="U_n = a + (n - 1) × b",
     concept_id="suku ke-n barisan aritmetika", concept_en="the n-th term of an arithmetic sequence",
     target_id="nilai suku ke-n", target_en="value of the n-th term",
     vars=[_v("a", "suku pertama", "first term"), _v("n", "urutan suku yang dicari", "position of the term"),
           _v("b", "beda", "common difference")],
     obj_id="barisan aritmetika tersebut", obj_en="of the arithmetic sequence",
     sample=lambda r: {"a": r.randint(1, 20), "n": r.randint(5, 20), "b": r.randint(1, 9)},
     compute=lambda v: v["a"] + (v["n"] - 1) * v["b"])
_add(key="diskriminan", domain="aljabar", formula="D = b² - 4 × a × c", held_out=True,
     concept_id="diskriminan persamaan kuadrat", concept_en="the discriminant of a quadratic equation",
     target_id="diskriminan", target_en="discriminant of the quadratic equation",
     vars=[_v("a", "koefisien x²", "coefficient of x²"), _v("b", "koefisien x", "coefficient of x"),
           _v("c", "konstanta", "constant term")],
     obj_id="persamaan kuadrat tersebut", obj_en="of the quadratic equation",
     sample=lambda r: {"a": r.randint(1, 4), "b": r.randint(2, 12), "c": r.randint(1, 6)},
     compute=lambda v: v["b"] ** 2 - 4 * v["a"] * v["c"])
_add(key="gradien_garis", domain="aljabar", formula="m = (y_2 - y_1) / (x_2 - x_1)",
     concept_id="gradien garis lurus", concept_en="the slope of a straight line",
     target_id="gradien garis", target_en="slope of the line",
     vars=[_v("y_2", "koordinat y titik kedua", "y-coordinate of the second point"),
           _v("y_1", "koordinat y titik pertama", "y-coordinate of the first point"),
           _v("x_2", "koordinat x titik kedua", "x-coordinate of the second point"),
           _v("x_1", "koordinat x titik pertama", "x-coordinate of the first point")],
     sample=_sample_gradient, compute=lambda v: (v["y_2"] - v["y_1"]) / (v["x_2"] - v["x_1"]))

# ---- Kalkulus dasar (teks polos dan LaTeX) ----
_add(key="turunan_pangkat", domain="kalkulus", formula="d/dx (x^n) = n × x^(n-1)", law=_LAW_POWER,
     concept_id="turunan dari x pangkat n", concept_en="the derivative of x to the power n",
     target_id="nilai turunan di titik tersebut", target_en="value of the derivative at that point",
     vars=[_v("n", "pangkat", "exponent"), _v("x", "nilai x", "value of x")],
     obj_id="pada fungsi tersebut", obj_en="in the function",
     sample=lambda r: {"n": r.randint(2, 6), "x": r.randint(1, 5)}, compute=lambda v: v["n"] * v["x"] ** (v["n"] - 1))
_add(key="turunan_pangkat_latex", domain="kalkulus", formula=r"\frac{d}{dx} x^{n} = n x^{n-1}", style="latex",
     law=_LAW_POWER, concept_id="turunan dari x pangkat n", concept_en="the derivative of x to the power n",
     target_id="nilai turunan di titik tersebut", target_en="value of the derivative at that point",
     vars=[_v("n", "pangkat", "exponent"), _v("x", "nilai x", "value of x")],
     obj_id="pada fungsi tersebut", obj_en="in the function",
     sample=lambda r: {"n": r.randint(2, 6), "x": r.randint(1, 5)}, compute=lambda v: v["n"] * v["x"] ** (v["n"] - 1))
_add(key="integral_pangkat", domain="kalkulus", formula="∫ x^n dx = x^(n+1) / (n + 1) + C", law=_LAW_POWER_INT,
     concept_id="integral dari x pangkat n", concept_en="the integral of x to the power n",
     target_id="nilai integral tentu dari 0 sampai batas atas", target_en="value of the definite integral from 0 to the upper limit",
     vars=[_v("n", "pangkat", "exponent"), _v("b", "batas atas", "upper limit")],
     obj_id="pada integral tersebut", obj_en="in the integral",
     sample=lambda r: {"n": r.randint(1, 3), "b": r.randint(1, 5)}, compute=lambda v: v["b"] ** (v["n"] + 1) / (v["n"] + 1))
_add(key="integral_pangkat_latex", domain="kalkulus", formula=r"\int x^{n} \, dx = \frac{x^{n+1}}{n+1} + C", style="latex",
     law=_LAW_POWER_INT, concept_id="integral dari x pangkat n", concept_en="the integral of x to the power n",
     target_id="nilai integral tentu dari 0 sampai batas atas", target_en="value of the definite integral from 0 to the upper limit",
     vars=[_v("n", "pangkat", "exponent"), _v("b", "batas atas", "upper limit")],
     obj_id="pada integral tersebut", obj_en="in the integral",
     sample=lambda r: {"n": r.randint(1, 3), "b": r.randint(1, 5)}, compute=lambda v: v["b"] ** (v["n"] + 1) / (v["n"] + 1))

# ---- Statistika ----
_add(key="rata_rata", domain="statistika", formula="μ = ∑x / n",
     concept_id="rata-rata data", concept_en="the mean of a data set",
     target_id="rata-rata data", target_en="mean of the data",
     vars=[_v("∑x", "jumlah seluruh data", "sum of all values"), _v("n", "banyak data", "number of values")], sample=_sample_mean, compute=lambda v: v["∑x"] / v["n"])
_add(key="varians", domain="statistika", formula="σ² = ∑(x - μ)² / n", held_out=True,
     concept_id="varians data", concept_en="the variance of a data set",
     target_id="varians data", target_en="variance of the data",
     vars=[_v("∑(x - μ)²", "jumlah kuadrat selisih", "sum of squared deviations"), _v("n", "banyak data", "number of values")], sample=_sample_variance, compute=lambda v: v["∑(x - μ)²"] / v["n"])
_add(key="simpangan_baku", domain="statistika", formula="σ = √(σ²)",
     concept_id="simpangan baku", concept_en="the standard deviation",
     target_id="simpangan baku", target_en="standard deviation",
     vars=[_v("σ²", "varians", "variance")], obj_id="data tersebut", obj_en="of the data",
     sample=lambda r: {"σ²": r.choice([4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144])}, compute=lambda v: math.sqrt(v["σ²"]))
_add(key="peluang", domain="statistika", formula="P(A) = n(A) / n(S)",
     concept_id="peluang kejadian A", concept_en="the probability of event A",
     target_id="peluang kejadian A", target_en="probability of event A",
     vars=[_v("n(A)", "banyak kejadian yang diharapkan", "number of favorable outcomes"),
           _v("n(S)", "banyak anggota ruang sampel", "size of the sample space")],
     sample=_sample_probability, compute=lambda v: v["n(A)"] / v["n(S)"])
_add(key="skor_z", domain="statistika", formula="z = (x - μ) / σ", held_out=True,
     concept_id="skor z", concept_en="the z-score",
     target_id="skor z", target_en="z-score",
     vars=[_v("x", "nilai data", "data value"), _v("μ", "rata-rata data", "mean of the data"), _v("σ", "simpangan baku data", "standard deviation of the data")], sample=_sample_zscore,
     compute=lambda v: (v["x"] - v["μ"]) / v["σ"])

# ---- Fisika ----
_add(key="massa_jenis", domain="fisika", formula="ρ = m / V",
     concept_id="massa jenis", concept_en="density",
     target_id="massa jenis benda", target_en="density of the object",
     vars=[_v("m", "massa", "mass", "g"), _v("V", "volume", "volume", "cm³")],
     obj_id="benda tersebut", obj_en="of the object", result_unit_id="g/cm³",
     sample=_sample_density, compute=lambda v: v["m"] / v["V"])
_add(key="kecepatan", domain="fisika", formula="v = s / t",
     concept_id="kecepatan", concept_en="speed",
     target_id="kecepatan benda", target_en="speed of the object",
     vars=[_v("s", "jarak tempuh", "distance traveled", "m"), _v("t", "waktu tempuh", "travel time", "s")],
     obj_id="benda tersebut", result_unit_id="m/s", sample=_sample_speed, compute=lambda v: v["s"] / v["t"])
_add(key="hukum_ohm", domain="fisika", formula="V = I × R", law=_LAW_OHM,
     concept_id="tegangan listrik", concept_en="the voltage in a circuit",
     target_id="tegangan", target_en="voltage",
     vars=[_v("I", "kuat arus", "current", "A"), _v("R", "hambatan", "resistance", "Ω")],
     obj_id="pada rangkaian tersebut", obj_en="in the circuit", result_unit_id="V",
     sample=lambda r: {"I": r.choice([0.5, 1, 1.5, 2, 2.5, 3, 4, 5]), "R": r.randint(2, 50)},
     compute=lambda v: v["I"] * v["R"])
_add(key="energi_kinetik", domain="fisika", formula="E_k = 1/2 × m × v²",
     concept_id="energi kinetik", concept_en="kinetic energy",
     target_id="energi kinetik benda", target_en="kinetic energy of the object",
     vars=[_v("m", "massa", "mass", "kg"), _v("v", "kecepatan", "speed", "m/s")],
     obj_id="benda tersebut", obj_en="of the object", result_unit_id="J",
     sample=lambda r: {"m": r.randint(1, 50), "v": r.randint(2, 30)}, compute=lambda v: 0.5 * v["m"] * v["v"] ** 2)
_add(key="energi_potensial", domain="fisika", formula="E_p = m × g × h",
     concept_id="energi potensial", concept_en="potential energy",
     target_id="energi potensial benda", target_en="potential energy of the object",
     vars=[_v("m", "massa", "mass", "kg"), _v("h", "ketinggian", "height", "m")],
     consts=[("g", "10 m/s²", "10 m/s²", "percepatan gravitasi", "the gravitational acceleration")],
     obj_id="benda tersebut", obj_en="of the object", result_unit_id="J",
     sample=lambda r: {"m": r.randint(1, 40), "h": r.randint(1, 30)}, compute=lambda v: v["m"] * 10 * v["h"])
_add(key="usaha", domain="fisika", formula="W = F × s",
     concept_id="usaha", concept_en="work",
     target_id="usaha yang dilakukan", target_en="work done",
     vars=[_v("F", "gaya", "force", "N"), _v("s", "perpindahan", "displacement", "m")],
     obj_id="benda tersebut", result_unit_id="J",
     sample=lambda r: {"F": r.randint(5, 80), "s": r.randint(2, 30)}, compute=lambda v: v["F"] * v["s"])
_add(key="daya", domain="fisika", formula="P = W / t",
     concept_id="daya", concept_en="power",
     target_id="daya", target_en="power",
     vars=[_v("W", "usaha", "work", "J"), _v("t", "waktu", "time", "s")],
     result_unit_id="W", sample=_sample_power, compute=lambda v: v["W"] / v["t"])
_add(key="hukum_newton_2", domain="fisika", formula="F = m × a", law=_LAW_NEWTON,
     concept_id="gaya yang bekerja pada benda", concept_en="the force acting on an object",
     target_id="gaya", target_en="force",
     vars=[_v("m", "massa", "mass", "kg"), _v("a", "percepatan", "acceleration", "m/s²")],
     obj_id="benda tersebut", obj_en="of the object", result_unit_id="N",
     sample=lambda r: {"m": r.randint(1, 60), "a": r.randint(1, 12)}, compute=lambda v: v["m"] * v["a"])
_add(key="tekanan", domain="fisika", formula="P = F / A", held_out=True,
     concept_id="tekanan", concept_en="pressure",
     target_id="tekanan", target_en="pressure",
     vars=[_v("F", "gaya", "force", "N"), _v("A", "luas bidang tekan", "contact area", "m²")],
     result_unit_id="Pa", sample=_sample_pressure, compute=lambda v: v["F"] / v["A"])
_add(key="glbb_jarak", domain="fisika", formula="s = v_0 × t + 1/2 × a × t²",
     concept_id="jarak tempuh pada gerak lurus berubah beraturan", concept_en="the distance traveled under uniform acceleration",
     target_id="jarak tempuh", target_en="distance traveled",
     vars=[_v("v_0", "kecepatan awal", "initial velocity", "m/s"), _v("t", "waktu", "time", "s"),
           _v("a", "percepatan", "acceleration", "m/s²")],
     result_unit_id="m", sample=lambda r: {"v_0": r.randint(0, 20), "t": r.randint(2, 12), "a": r.randint(1, 6)},
     compute=lambda v: v["v_0"] * v["t"] + 0.5 * v["a"] * v["t"] ** 2)
_add(key="hukum_hooke", domain="fisika", formula="F = k × x", law=_LAW_HOOKE,
     concept_id="gaya pegas", concept_en="the spring force",
     target_id="gaya pegas", target_en="spring force",
     vars=[_v("k", "konstanta pegas", "spring constant", "N/m"), _v("x", "pertambahan panjang", "extension", "m")],
     obj_id="pegas tersebut", obj_en="of the spring", result_unit_id="N",
     sample=lambda r: {"k": r.choice([50, 100, 150, 200, 250, 300, 400, 500]), "x": r.choice([0.1, 0.2, 0.25, 0.3, 0.4, 0.5])},
     compute=lambda v: v["k"] * v["x"])

# ---- Kimia dasar ----
_add(key="jumlah_mol", domain="kimia", formula="n = m / M_r",
     concept_id="jumlah mol zat", concept_en="the amount of substance in moles",
     target_id="jumlah mol zat", target_en="amount of substance",
     vars=[_v("m", "massa zat", "mass of the substance", "g"), _v("M_r", "massa molar", "molar mass", "g/mol")],
     result_unit_id="mol", sample=_sample_moles, compute=lambda v: v["m"] / v["M_r"])
_add(key="molaritas", domain="kimia", formula="M = n / V",
     concept_id="molaritas larutan", concept_en="the molarity of a solution",
     target_id="molaritas larutan", target_en="molarity of the solution",
     vars=[_v("n", "jumlah mol zat terlarut", "moles of solute", "mol"), _v("V", "volume larutan", "volume of the solution", "L")],
     result_unit_id="mol/L", sample=_sample_molarity, compute=lambda v: v["n"] / v["V"])
_add(key="gas_ideal", domain="kimia", formula="P × V = n × R × T", law=_LAW_GAS, held_out=True,
     concept_id="tekanan gas ideal", concept_en="the pressure of an ideal gas",
     target_id="tekanan gas", target_en="pressure of the gas",
     vars=[_v("n", "jumlah mol gas", "amount of gas", "mol"), _v("T", "suhu", "temperature", "K"), _v("V", "volume", "volume", "L")],
     consts=[("R", "0,082", "0.082", "tetapan gas", "the gas constant")],
     result_unit_id="atm",
     sample=lambda r: {"n": r.choice([1, 2, 3]), "T": r.choice([273, 300, 400, 500]), "V": r.choice([10, 20, 25, 50])},
     compute=lambda v: v["n"] * 0.082 * v["T"] / v["V"])
_add(key="ph_larutan", domain="kimia", formula="pH = -log[H+]",
     concept_id="pH larutan", concept_en="the pH of a solution",
     target_id="pH larutan", target_en="pH of the solution",
     vars=[_v("[H+]", "konsentrasi ion H+", "concentration of H+ ions", "M")],
     sample=_sample_ph, compute=lambda v: -math.log10(v["[H+]"]))

# ---- Ekonomi dan keuangan ----
_add(key="bunga_sederhana", domain="keuangan", formula="I = P × r × t",
     concept_id="bunga sederhana", concept_en="simple interest",
     target_id="bunga sederhana", target_en="simple interest",
     vars=[_v("P", "modal pokok", "principal", "rupiah", "dollars"),
           _v("r", "suku bunga per tahun", "annual interest rate"), _v("t", "lama pinjaman", "loan period", "tahun", "years")],
     result_unit_id="rupiah", result_unit_en="dollars",
     sample=lambda r: {"P": r.choice([1, 2, 3, 4, 5, 8, 10]) * 1000000, "r": r.choice([0.05, 0.06, 0.08, 0.1, 0.12]), "t": r.randint(1, 5)},
     compute=lambda v: v["P"] * v["r"] * v["t"])
_add(key="bunga_majemuk", domain="keuangan", formula="A = P × (1 + r)^n", held_out=True,
     concept_id="nilai akhir investasi dengan bunga majemuk", concept_en="the future value with compound interest",
     target_id="nilai akhir investasi", target_en="future value of the investment",
     vars=[_v("P", "modal awal", "initial principal", "rupiah", "dollars"),
           _v("r", "suku bunga per periode", "interest rate per period"), _v("n", "jumlah periode", "number of periods")],
     result_unit_id="rupiah", result_unit_en="dollars",
     sample=lambda r: {"P": r.choice([1000000, 2000000, 5000000]), "r": r.choice([0.05, 0.1]), "n": r.choice([2, 3])},
     compute=lambda v: v["P"] * (1 + v["r"]) ** v["n"])
_add(key="laba_usaha", domain="keuangan", formula="L = TR - TC",
     concept_id="laba usaha", concept_en="the profit of a business",
     target_id="laba usaha", target_en="profit of the business",
     vars=[_v("TR", "total pendapatan", "total revenue", "rupiah", "dollars"), _v("TC", "total biaya", "total cost", "rupiah", "dollars")],
     result_unit_id="rupiah", result_unit_en="dollars",
     sample=lambda r: {"TR": r.randint(10, 60) * 500000, "TC": r.randint(2, 9) * 500000},
     compute=lambda v: v["TR"] - v["TC"])
_add(key="roi", domain="keuangan", formula="ROI = (L / I) × 100%",
     concept_id="return on investment (ROI)", concept_en="the return on investment (ROI)",
     target_id="return on investment (ROI)", target_en="return on investment (ROI)",
     vars=[_v("L", "laba bersih", "net profit", "rupiah", "dollars"), _v("I", "nilai investasi", "investment value", "rupiah", "dollars")],
     result_unit_id="%", sample=_sample_roi, compute=lambda v: v["L"] / v["I"] * 100)
_add(key="titik_impas", domain="keuangan", formula="Q = FC / (P - VC)",
     concept_id="titik impas (BEP)", concept_en="the break-even point",
     target_id="titik impas (BEP)", target_en="break-even point",
     vars=[_v("FC", "biaya tetap", "fixed cost", "rupiah", "dollars"),
           _v("P", "harga jual per unit", "selling price per unit", "rupiah", "dollars"),
           _v("VC", "biaya variabel per unit", "variable cost per unit", "rupiah", "dollars")],
     result_unit_id="unit", result_unit_en="units", sample=_sample_breakeven,
     compute=lambda v: v["FC"] / (v["P"] - v["VC"]))
_add(key="pdb", domain="keuangan", formula="GDP = C + I + G + (X - M)", held_out=True,
     concept_id="produk domestik bruto (PDB)", concept_en="the gross domestic product (GDP)",
     target_id="produk domestik bruto (PDB)", target_en="gross domestic product (GDP)",
     vars=[_v("C", "konsumsi", "consumption", "triliun rupiah", "trillion dollars"),
           _v("I", "investasi", "investment", "triliun rupiah", "trillion dollars"),
           _v("G", "pengeluaran pemerintah", "government spending", "triliun rupiah", "trillion dollars"),
           _v("X", "ekspor", "exports", "triliun rupiah", "trillion dollars"),
           _v("M", "impor", "imports", "triliun rupiah", "trillion dollars")],
     result_unit_id="triliun rupiah", result_unit_en="trillion dollars",
     sample=lambda r: {"C": r.randint(300, 900), "I": r.randint(100, 400), "G": r.randint(100, 300), "X": r.randint(100, 400), "M": r.randint(50, 350)},
     compute=lambda v: v["C"] + v["I"] + v["G"] + (v["X"] - v["M"]))

assert len(FORMULA_SPECS) >= 30, "Template formula kurang dari 30."
assert len({spec["key"] for spec in FORMULA_SPECS}) == len(FORMULA_SPECS), "Ada key template yang duplikat."


# =============================================================================
# BAGIAN B. Generator Contrastive Triplets: satu konteks, banyak target berbeda
# =============================================================================
# Untuk SETIAP konteks sintetis dibuat minimal tiga baris dengan answer berbeda:
#   role "formula"  : answer = rumusnya                 -> pertanyaan tentang rumus
#   role "result"   : answer = hasil hitung + satuan    -> pertanyaan tentang besaran spesifik
#   role "variable" : answer = salah satu nilai diketahui (membaca posisi <answer> dengan teliti)
#   role "law"      : answer = nama hukum/teorema/aturan (bila template memilikinya)
# Angka dibangkitkan acak (bukan 3 varian tetap seperti Fase 2), sehingga ribuan konteks unik tercipta.
LEADS = {
    "id": {
        "law": ["{law} menyatakan bahwa {concept} dirumuskan sebagai {formula}.",
                "Menurut {law}, {concept} dinyatakan dengan persamaan {formula}.",
                "{law} dituliskan sebagai {formula}.",
                "Dalam {law}, hubungan tersebut dirumuskan {formula}."],
        "plain": ["{Concept} dihitung dengan rumus {formula}.",
                  "Rumus {concept} adalah {formula}.",
                  "{Concept} dirumuskan sebagai {formula}.",
                  "Untuk menentukan {concept}, digunakan persamaan {formula}."],
    },
    "en": {
        "law": ["{law} states that {concept} is given by {formula}.",
                "According to {law}, {concept} is expressed as {formula}.",
                "{law} is written as {formula}.",
                "In {law}, the relationship is expressed as {formula}."],
        "plain": ["{Concept} is calculated using the formula {formula}.",
                  "The formula for {concept} is {formula}.",
                  "{Concept} is given by {formula}.",
                  "To determine {concept}, the equation {formula} is used."],
    },
}
EXAMPLE_SENTENCES = {
    "id": ["Jika {given}, maka {target} adalah {res}.",
           "Diketahui {given}, sehingga {target} yang diperoleh sebesar {res}.",
           "Pada contoh soal, {given}. Dengan demikian, {target} bernilai {res}.",
           "Untuk {given}, hasilnya {target} sebesar {res}."],
    "en": ["If {given}, then the {target} is {res}.",
           "Given {given}, the {target} comes out to {res}.",
           "In this example, {given}. Therefore, the {target} equals {res}.",
           "With {given}, the resulting {target} is {res}."],
}
UNIT_NOTES = {
    "id": "Satuan {target} dalam contoh ini adalah {unit}.",
    "en": "The unit of the {target} here is {unit}.",
}
FORMULA_QUESTIONS = {
    "id": ["Apa rumus untuk menghitung {c}?", "Persamaan apa yang dipakai untuk menentukan {c}?",
           "Bagaimana bentuk rumus {c}?", "Rumus apa yang menyatakan {c}?", "Dengan persamaan apa {c} dihitung?",
           "Apa persamaan matematis dari {c}?", "Rumus manakah yang digunakan untuk {c}?"],
    "en": ["What is the formula for {c}?", "Which equation is used to calculate {c}?",
           "How is the formula for {c} written?", "What expression gives {c}?", "Which equation describes {c}?",
           "What is the mathematical relationship for {c}?", "Which formula is used to find {c}?"],
}
RESULT_QUESTIONS = {
    "id": ["Berapa {t} pada contoh tersebut?", "Berapakah besar {t} yang diperoleh?",
           "Berapa nilai {t} berdasarkan data yang diketahui?", "Jika {gn}, berapa {t}?",
           "Berapa {t} yang dihasilkan dari perhitungan tersebut?", "Berapakah {t} yang dihasilkan?"],
    "en_unit": ["What is the {t}?", "How much is the {t} in this example?", "What is the {t} obtained from the given data?",
                "If {gn}, what is the {t}?", "What is the calculated {t}?", "What is the resulting {t}?"],
    "en_bare": ["What is the value of the {t}?", "How much is the {t} in this example?",
                "What value is obtained for the {t} from the given data?", "If {gn}, what is the value of the {t}?",
                "What value does the calculated {t} take?", "What is the resulting value of the {t}?"],
}
VARIABLE_QUESTIONS = {
    "id": ["Berapa {vp}?", "Berapakah nilai {vp} yang diketahui?", "Berapa besar {vp} menurut data tersebut?"],
    "en_unit": ["What is the {vp}?", "What value is given for the {vp}?", "According to the data, what is the {vp}?"],
    "en_bare": ["How much is the {vp}?", "What number is given for the {vp}?", "According to the data, how much is the {vp}?"],
}
_EXPECTED_TYPE = {"formula": "formula", "result": "number", "variable": "number", "law": "text"}


def _cap(text):
    return text[:1].upper() + text[1:]


def _unit_of(var, lang):
    return var["uid"] if lang == "id" else var["uen"]


def build_context_rows(spec, lang, rng, split_name, used_contexts, group_counter):
    """Bangun satu konteks formula beserta barisnya. Kembalikan [] bila batasan kualitas tidak terpenuhi."""
    values = spec["sample"](rng)
    result_value = spec["compute"](values)
    result_unit = spec["result_unit_id"] if lang == "id" else spec["result_unit_en"]
    result_text = fmt_with_unit(result_value, result_unit, lang)

    name_key = "id" if lang == "id" else "en"
    formula = spec["formula"]
    formula_in_text = f"${formula}$" if spec["style"] == "latex" else formula
    concept = spec["concept_" + lang]
    target = spec["target_" + lang]
    obj = spec["obj_" + lang]

    use_law = bool(spec["law"]) and rng.random() < 0.8
    law_name = spec["law"][name_key] if use_law else None
    lead_template = rng.choice(LEADS[lang]["law" if use_law else "plain"])
    lead = lead_template.format(law=law_name or "", concept=concept, Concept=_cap(concept), formula=formula_in_text)

    value_texts = {var["sym"]: fmt_with_unit(values[var["sym"]], _unit_of(var, lang), lang) for var in spec["vars"]}
    const_symbol = [f"{c[0]} = {c[1] if lang == 'id' else c[2]}" for c in spec["consts"]]
    const_natural = [f"{c[3] if lang == 'id' else c[4]} {c[1] if lang == 'id' else c[2]}" for c in spec["consts"]]
    given_symbol = join_items([f"{var['sym']} = {value_texts[var['sym']]}" for var in spec["vars"]] + const_symbol, lang)
    given_natural = join_items([f"{var[name_key]} {value_texts[var['sym']]}" for var in spec["vars"]] + const_natural, lang)
    given_in_context = given_symbol if rng.random() < 0.5 else given_natural

    parts = [lead]
    if result_unit and result_unit != "%" and rng.random() < 0.30:
        parts.append(UNIT_NOTES[lang].format(target=target, unit=result_unit))
    example = rng.choice(EXAMPLE_SENTENCES[lang]).format(given=given_in_context, target=target, res=result_text)
    parts.append(example)
    context = normalize_space(" ".join(parts))
    if context in used_contexts:
        return []

    roles = [("formula", formula, None), ("result", result_text, None)]
    variables = list(spec["vars"])
    rng.shuffle(variables)
    for var in variables[: (2 if len(variables) >= 2 and rng.random() < 0.4 else 1)]:
        roles.append(("variable", value_texts[var["sym"]], var))
    if use_law:
        roles.append(("law", law_name, None))

    unit_free = not result_unit
    group_id = f"{split_name}-{spec['key']}-{lang}-{group_counter[0]}"
    group_counter[0] += 1
    rows = []
    for role, answer, var in roles:
        spans = find_exact_spans(context, answer)
        if len(spans) != 1:
            continue
        if answer_type(answer) != _EXPECTED_TYPE[role]:
            continue
        if role == "formula":
            question = rng.choice(FORMULA_QUESTIONS[lang]).format(c=concept)
        elif role == "result":
            key = "id" if lang == "id" else ("en_bare" if unit_free else "en_unit")
            question = rng.choice(RESULT_QUESTIONS[key]).format(t=target, gn=given_natural)
        elif role == "variable":
            key = "id" if lang == "id" else ("en_bare" if not _unit_of(var, lang) else "en_unit")
            phrase = f"{var[name_key]} {obj}".strip()
            question = rng.choice(VARIABLE_QUESTIONS[key]).format(vp=phrase)
        else:
            question = rng.choice(spec["law"]["q_" + lang])
        rows.append({
            "lang": lang, "context": context, "answer": answer, "question": clean_question(question),
            "answer_start": spans[0][0], "answer_type": _EXPECTED_TYPE[role],
            "has_formula": True, "formula_spans": json.dumps([s[2] for s in detect_formula_spans(context)], ensure_ascii=False),
            "source": f"synthetic_formula_{split_name}", "ctx_format": "formula", "role": role,
            "spec_key": spec["key"], "group_id": group_id,
        })
    present = {row["role"] for row in rows}
    if "formula" not in present or "result" not in present or not (present & {"variable", "law"}):
        return []
    if not detect_formula_content(context)[0]:
        return []
    used_contexts.add(context)
    return rows


def generate_formula_dataset(train_per_lang, eval_seen_per_lang, eval_unseen_per_lang, seed):
    """Hasilkan train, eval_seen (template sama, angka baru), dan eval_unseen (template tak pernah dilihat)."""
    rng = random.Random(seed)
    train_specs = [s for s in FORMULA_SPECS if not s["held_out"]]
    unseen_specs = [s for s in FORMULA_SPECS if s["held_out"]]
    used_contexts = set()
    group_counter = [0]
    output = {name: {"id": [], "en": []} for name in ("train", "eval_seen", "eval_unseen")}

    def fill(split_name, specs, target_rows, lang):
        rows, index, attempts = [], 0, 0
        while len(rows) < target_rows and attempts < target_rows * 60:
            attempts += 1
            spec = specs[index % len(specs)]
            index += 1
            rows.extend(build_context_rows(spec, lang, rng, split_name, used_contexts, group_counter))
        if len(rows) < target_rows:
            raise RuntimeError(f"Data formula {split_name}/{lang} hanya {len(rows)} baris dari target {target_rows}.")
        output[split_name][lang] = rows

    for lang in ("id", "en"):
        fill("train", train_specs, train_per_lang, lang)
    for lang in ("id", "en"):
        fill("eval_seen", train_specs, eval_seen_per_lang, lang)
        fill("eval_unseen", unseen_specs, eval_unseen_per_lang, lang)
    return output


# =============================================================================
# BAGIAN C. Safe Slide-Style Augmentation (non-destruktif)
# =============================================================================
# Perbedaan dari Fase 2: (1) membuat SALINAN baru, baris prosa asli tidak disentuh,
# (2) pemisah kalimat sadar singkatan, (3) label header mengikuti bahasa baris,
# (4) answer wajib utuh dan offset dihitung ulang, bila gagal baris dilewati.
SLIDE_HEADERS = {
    "id": [("Topik", "Poin Materi"), ("Topik", "Poin Penting"), ("Judul Slide", "Isi Slide"), ("Materi", "Poin Utama")],
    "en": [("Slide Topic", "Key Points"), ("Topic", "Key Points"), ("Slide Title", "Main Points"), ("Overview", "Highlights")],
}


def _nth_occurrence(text, needle, n):
    position = -1
    for _ in range(n + 1):
        position = text.find(needle, position + 1)
        if position < 0:
            return -1
    return position


def _strip_final_punctuation(sentence, protect_answer=None):
    sentence = sentence.rstrip()
    if protect_answer and sentence.endswith(protect_answer):
        return sentence
    if sentence and sentence[-1] in "!?":
        return sentence[:-1].rstrip()
    if sentence.endswith(".") and not ends_with_abbreviation(sentence):
        return sentence[:-1].rstrip()
    return sentence


def _split_long_bullet(bullet, answer, rng):
    """Pecah kalimat panjang di koma/titik koma bila aman: kedua sisi minimal 5 kata dan answer tidak terpotong."""
    words = bullet.split()
    if len(words) <= 20 or rng.random() > 0.5:
        return [bullet]
    answer_span = locate_answer(bullet, answer, -1)
    candidates = []
    for match in re.finditer(r"[,;]\s+", bullet):
        left, right = bullet[:match.start()], bullet[match.end():]
        if len(left.split()) < 5 or len(right.split()) < 5:
            continue
        if answer_span and answer_span[0] < match.end() and answer_span[1] > match.start():
            continue
        candidates.append((abs(len(left) - len(right)), left, right))
    if not candidates:
        return [bullet]
    _, left, right = min(candidates)
    right = re.sub(r"^(?:and|but|or|dan|tetapi|namun|serta|atau)\s+", "", right, flags=re.IGNORECASE)
    return [left, right]


def convert_to_slide_style(context, answer, lang="id", rng=None, answer_start=-1, max_sentences=5):
    """Ubah prosa menjadi format slide. Kembalikan (konteks_baru, answer_start_baru) atau None bila tidak aman."""
    rng = rng or random
    text = normalize_space(context)
    answer = normalize_space(answer)
    span = locate_answer(text, answer, answer_start)
    if span is None:
        return None
    sentence_list = sentence_spans(text)
    if len(sentence_list) < 2:
        return None
    holders = [i for i, (s, e) in enumerate(sentence_list) if s <= span[0] and span[1] <= e]
    if not holders:
        return None
    answer_index = holders[0]
    low = max(0, answer_index - (max_sentences - 1) // 2)
    high = min(len(sentence_list), low + max_sentences)
    low = max(0, high - max_sentences)
    window = [text[s:e].strip() for s, e in sentence_list[low:high]]
    if len(window) < 2:
        return None

    occurrence = sum(w.count(answer) for w in window[: answer_index - low])
    sentence_start = sentence_list[answer_index][0]
    occurrence += text[sentence_start:span[0]].count(answer)

    topic = _strip_final_punctuation(window[0], protect_answer=answer)
    bullets = []
    for sentence in window[1:]:
        bullets.extend(_split_long_bullet(sentence, answer, rng))
    bullets = [_strip_final_punctuation(b, protect_answer=answer) for b in bullets]
    header, bullet_header = rng.choice(SLIDE_HEADERS["id" if lang == "id" else "en"])
    new_context = normalize_space(f"{header}: {topic} | {bullet_header}: " + " ".join(f"- {b}" for b in bullets))
    new_start = _nth_occurrence(new_context, answer, occurrence)
    if new_start < 0:
        return None
    return new_context, new_start


def make_slide_rows(rows, target_count, seed, max_sentences=5):
    """Buat salinan slide-style dari rows. Baris asli tidak diubah."""
    rng = random.Random(seed)
    order = list(range(len(rows)))
    rng.shuffle(order)
    converted = []
    for index in order:
        if len(converted) >= target_count:
            break
        row = rows[index]
        result = convert_to_slide_style(row["context"], row["answer"], row["lang"], rng, row["answer_start"], max_sentences)
        if result is None:
            continue
        new_context, new_start = result
        copy = dict(row)
        copy.update({
            "context": new_context, "answer_start": new_start, "ctx_format": "slide",
            "has_formula": detect_formula_content(new_context)[0],
            "formula_spans": json.dumps([s[2] for s in detect_formula_spans(new_context)], ensure_ascii=False),
            "source": str(row.get("source", "unknown")) + "_slide",
        })
        converted.append(copy)
    return converted


# =============================================================================
# BAGIAN D. Pembersihan label training dan penyusunan kurikulum
# =============================================================================
def filter_training_rows(rows, name):
    """Buang label buntung atau tidak berpijak pada context (kriteria longgar, validasi tidak disentuh)."""
    kept, dropped = [], Counter()
    for row in rows:
        ok, reasons = question_quality(row["question"], row["context"], row["answer"], row["lang"], strict=False)
        if ok:
            kept.append(row)
        else:
            dropped.update(reasons)
    removed = len(rows) - len(kept)
    print(f"[bersih-label] {name}: {len(rows):,} -> {len(kept):,} (dibuang {removed:,}, {removed / max(1, len(rows)):.2%}) {dict(dropped)}")
    return kept


# =============================================================================
# Eksekusi: bersihkan label, hasilkan data formula kontras, slide-style, dan susun kurikulum
# =============================================================================
print("=" * 78)
print("[1/5] PEMBERSIHAN LABEL TRAINING DAN PENYERAGAMAN FORMAT PERTANYAAN")
print("=" * 78)
for rows in (train_general, val_general, test_general):
    for row in rows:
        row["question"] = clean_question(row["question"])
# Hanya train yang dibersihkan dari label buntung. Val dan test dibiarkan agar angka tetap jujur.
train_general_clean = filter_training_rows(train_general, "train_general")

print("\n" + "=" * 78)
print("[2/5] CONTRASTIVE FORMULA DATASET")
print("=" * 78)
formula_data = generate_formula_dataset(
    FORMULA_TRAIN_PER_LANG, FORMULA_EVAL_SEEN_PER_LANG, FORMULA_EVAL_UNSEEN_PER_LANG, SEED + 7
)
formula_train_rows = formula_data["train"]["id"] + formula_data["train"]["en"]
raw_formula_eval_seen = formula_data["eval_seen"]["id"] + formula_data["eval_seen"]["en"]
raw_formula_eval_unseen = formula_data["eval_unseen"]["id"] + formula_data["eval_unseen"]["en"]
for row in raw_formula_eval_seen:
    row["eval_split"] = "seen"
for row in raw_formula_eval_unseen:
    row["eval_split"] = "unseen"
raw_formula_eval = raw_formula_eval_seen + raw_formula_eval_unseen

formula_contexts_train = {r["context"] for r in formula_train_rows}
formula_contexts_eval = {r["context"] for r in raw_formula_eval}
assert formula_contexts_train.isdisjoint(formula_contexts_eval), "Konteks formula train dan eval tidak disjoint."
train_templates = {r["spec_key"] for r in formula_train_rows}
unseen_templates = {r["spec_key"] for r in raw_formula_eval_unseen}
assert train_templates.isdisjoint(unseen_templates), "Template unseen bocor ke training."
for lang in ("id", "en"):
    assert sum(r["lang"] == lang for r in formula_train_rows) >= 900, f"Formula train {lang} kurang dari 900."
    assert sum(r["lang"] == lang for r in raw_formula_eval) >= 180, f"Formula eval {lang} kurang dari 180."

formula_summary = pd.DataFrame([
    {"split": name, "lang": lang, "baris": sum(r["lang"] == lang for r in rows),
     "konteks_unik": len({r["context"] for r in rows if r["lang"] == lang}),
     "template": len({r["spec_key"] for r in rows if r["lang"] == lang})}
    for name, rows in (("train", formula_train_rows), ("eval_seen", raw_formula_eval_seen), ("eval_unseen", raw_formula_eval_unseen))
    for lang in ("id", "en")
])
display(formula_summary)
role_table = pd.crosstab(pd.Series([r["role"] for r in formula_train_rows], name="role"),
                         pd.Series([r["answer_type"] for r in formula_train_rows], name="answer_type"))
print("\nPeran target pada formula train (setiap konteks memiliki minimal 3 target berbeda):")
display(role_table)
print(f"Template train: {len(train_templates)} | template unseen untuk uji generalisasi: {sorted(unseen_templates)}")
variety = pd.DataFrame([
    {"role": role, "pertanyaan_unik": len({r["question"] for r in formula_train_rows if r["role"] == role}),
     "baris": sum(r["role"] == role for r in formula_train_rows)}
    for role in ("formula", "result", "variable", "law")
])
display(variety)

print("\nContoh Contrastive Triplet (satu konteks, tiga target berbeda):")
for lang in ("id", "en"):
    first_group = next(r["group_id"] for r in formula_train_rows if r["lang"] == lang and r["role"] == "law") if any(
        r["lang"] == lang and r["role"] == "law" for r in formula_train_rows) else formula_train_rows[0]["group_id"]
    group_rows = [r for r in formula_train_rows if r["group_id"] == first_group]
    print(f"\n  KONTEKS [{lang}]: {group_rows[0]['context']}")
    for row in group_rows:
        print(f"    ({row['role']:<8}|{row['answer_type']:<7}) {row['question']}  <-  {row['answer']!r}")

print("\n" + "=" * 78)
print("[3/5] SAFE SLIDE-STYLE AUGMENTATION (NON-DESTRUKTIF)")
print("=" * 78)
n_slide_general = int(SLIDE_FROM_GENERAL_RATIO * len(train_general_clean))
n_slide_formula = int(SLIDE_FROM_FORMULA_RATIO * len(formula_train_rows))
slide_from_general = make_slide_rows(train_general_clean, n_slide_general, SEED + 11)
slide_from_formula = make_slide_rows(formula_train_rows, n_slide_formula, SEED + 12)
slide_train_rows = slide_from_general + slide_from_formula
print(f"Slide dari prosa umum : {len(slide_from_general):,} dari target {n_slide_general:,}")
print(f"Slide dari formula    : {len(slide_from_formula):,} dari target {n_slide_formula:,}")
print(f"Baris prosa asli tetap utuh di train_general_clean: {len(train_general_clean):,}")

raw_slide_eval = []
for lang in ("id", "en"):
    lang_val = [r for r in val_general if r["lang"] == lang]
    raw_slide_eval.extend(make_slide_rows(lang_val, SLIDE_EVAL_PER_LANG, SEED + 21))
annotate_rows(slide_train_rows)
for row in slide_train_rows + raw_slide_eval:
    row["ctx_format"] = "slide"
    row["answer_type"] = answer_type(row["answer"])
print(f"raw_slide_eval: {len(raw_slide_eval)} baris ({Counter(r['lang'] for r in raw_slide_eval)})")
print("\nContoh slide-style (id dan en):")
for lang in ("id", "en"):
    sample = next(r for r in slide_from_general if r["lang"] == lang)
    print(f"  [{lang}] {sample['context'][:420]}")
    print(f"       Q: {sample['question']} | A: {sample['answer']!r}")
    assert sample["context"].split(":")[0] in {h[0] for h in SLIDE_HEADERS[lang]}

print("\n" + "=" * 78)
print("[4/5] KURIKULUM: STAGE 1 DAN STAGE 2")
print("=" * 78)
rng_stage = random.Random(SEED + 31)
replay_rows = rng_stage.sample(train_general_clean, int(REPLAY_RATIO * len(train_general_clean)))
stage1_train_rows = list(train_general_clean)
stage2_train_rows = list(formula_train_rows) + list(slide_train_rows) + list(replay_rows)
rng_stage.shuffle(stage1_train_rows)
rng_stage.shuffle(stage2_train_rows)

composition = pd.DataFrame({
    "stage1_train": {"prosa umum": len(stage1_train_rows)},
    "stage2_train": {"formula kontras": len(formula_train_rows), "slide-style": len(slide_train_rows), "replay prosa (35%)": len(replay_rows)},
}).fillna(0).astype(int)
display(composition)
print(f"Stage 1: {len(stage1_train_rows):,} baris | Stage 2: {len(stage2_train_rows):,} baris")

print("\n" + "=" * 78)
print("[5/5] SANITY CHECK")
print("=" * 78)
for row in stage1_train_rows + stage2_train_rows + raw_formula_eval + raw_slide_eval + val_general + test_general:
    assert row["context"][row["answer_start"]:row["answer_start"] + len(row["answer"])] == row["answer"], row
print("Seluruh answer_start valid pada seluruh split.")
prose_contexts_stage = {r["context"] for r in stage1_train_rows}
assert prose_contexts_stage.isdisjoint({r["context"] for r in val_general}), "Bocor train->val."
assert prose_contexts_stage.isdisjoint({r["context"] for r in test_general}), "Bocor train->test."
print("Konteks prosa train tidak beririsan dengan val dan test.")
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())


## Cell 7. Tokenisasi, special token, dan EDA pasca-preprocessing

Definisi `mark_formulas`, `mark_answer`, `mark_answer_and_formulas`, dan `make_length_safe_prompt` (maksimum 512 token input dan 64 token target, dengan jendela berpusat pada jawaban). Special token diuji agar masing-masing menjadi tepat satu id. Cell ini juga memeriksa simbol matematika yang berubah menjadi `<unk>` pada tokenizer mT5, lalu menampilkan komposisi Stage 1 versus Stage 2 dan contoh prompt yang sudah didecode.

In [ ]:
# =============================================================================
# Special token, marking answer/formula, dan prompt yang aman panjang
# =============================================================================
SPECIAL_TOKENS = {"additional_special_tokens": ["<answer>", "</answer>", "<formula>", "</formula>"]}
NEW_TOKEN_LIST = SPECIAL_TOKENS["additional_special_tokens"]
MAX_FORMULA_MARKS = 16


def _apply_marks(text, marks):
    """Sisipkan tag di sekitar span. marks = [(start, end, tag_buka, tag_tutup)] yang tidak saling tumpang tindih.
    Tag diberi spasi di kedua sisinya agar token pertama answer identik dengan token pada baris 'answer: ...'."""
    pieces, cursor = [], 0
    for start, end, open_tag, close_tag in sorted(marks):
        if start < cursor:
            continue
        pieces.append(text[cursor:start])
        pieces.append(f" {open_tag} {text[start:end]} {close_tag} ")
        cursor = end
    pieces.append(text[cursor:])
    return normalize_space("".join(pieces))


def _formula_marks(context, answer_span):
    marks = []
    for start, end, _ in detect_formula_spans(context)[:MAX_FORMULA_MARKS]:
        if answer_span and start < answer_span[1] and end > answer_span[0]:
            continue  # span yang beririsan dengan answer tidak dibungkus <formula>
        marks.append((start, end, "<formula>", "</formula>"))
    return marks


def mark_formulas(context, answer_span=None):
    """Bungkus ekspresi matematika dengan <formula>...</formula>, kecuali yang sedang menjadi <answer>."""
    context = normalize_space(context)
    return _apply_marks(context, _formula_marks(context, answer_span))


def mark_answer(context, answer, answer_start=-1):
    """Bungkus answer dengan <answer>...</answer>. Teks setelah answer SELALU dipertahankan."""
    context = normalize_space(context)
    span = locate_answer(context, normalize_space(answer), answer_start)
    if span is None:
        return context
    return _apply_marks(context, [(span[0], span[1], "<answer>", "</answer>")])


def mark_answer_and_formulas(context, answer, answer_start=-1):
    """Tandai answer dan formula sekaligus dengan offset dari konteks asli, sehingga tidak ada pergeseran posisi."""
    context = normalize_space(context)
    span = locate_answer(context, normalize_space(answer), answer_start)
    marks = _formula_marks(context, span)
    if span is not None:
        marks.append((span[0], span[1], "<answer>", "</answer>"))
    return _apply_marks(context, marks)


def make_prompt(lang, answer, kind, has_formula, marked_context):
    return (f"generate question {lang}: answer_type: {kind}: formula_context: {'yes' if has_formula else 'no'}: "
            f"answer: {answer} context: {marked_context}")


def _count_tokens(tok, text):
    return len(tok(text, truncation=False)["input_ids"])


def make_length_safe_prompt(lang, context, answer, answer_kind, has_formula, answer_start, tok=None, max_length=None):
    """Prompt dengan jendela konteks yang berpusat pada answer bila melebihi batas token."""
    tok = tok or tokenizer
    max_length = max_length or MAX_INPUT_LENGTH
    context = normalize_space(context)
    answer = normalize_space(answer)
    span = locate_answer(context, answer, answer_start)

    def render(low, high):
        piece = context[low:high]
        local = (span[0] - low, span[1] - low) if span and low <= span[0] and span[1] <= high else None
        marks = _formula_marks(piece, local)
        if local:
            marks.append((local[0], local[1], "<answer>", "</answer>"))
        return make_prompt(lang, answer, answer_kind, has_formula, _apply_marks(piece, marks))

    prompt = render(0, len(context))
    if _count_tokens(tok, prompt) <= max_length:
        return prompt
    center = (span[0] + span[1]) // 2 if span else len(context) // 2
    half = max(len(context) // 2, 400)
    for _ in range(16):
        half = int(half * 0.8)
        low, high = max(0, center - half), min(len(context), center + half)
        if span:
            low, high = min(low, span[0]), max(high, span[1])
        if low > 0:
            boundary = context.find(" ", low)
            if 0 <= boundary < (span[0] if span else high):
                low = boundary + 1
        if high < len(context):
            boundary = context.rfind(" ", 0, high)
            if boundary > (span[1] if span else low):
                high = boundary
        prompt = render(low, high)
        if _count_tokens(tok, prompt) <= max_length:
            return prompt
    return prompt


SCHEMA_DEFAULTS = {
    "lang": "", "context": "", "answer": "", "question": "", "answer_start": -1, "answer_type": "text",
    "has_formula": False, "formula_spans": "[]", "source": "", "ctx_format": "prose", "role": "",
    "spec_key": "", "group_id": "", "eval_split": "",
}


def to_schema(rows):
    """Seragamkan kolom antar baris agar Dataset.from_list tidak menjatuhkan atau menggeser kolom."""
    normalized = []
    for row in rows:
        item = {key: row.get(key, default) for key, default in SCHEMA_DEFAULTS.items()}
        item["answer_start"] = int(item["answer_start"])
        item["has_formula"] = bool(item["has_formula"])
        normalized.append(item)
    return normalized


# =============================================================================
# Tokenizer, penambahan special token, dan pemeriksaan cakupan simbol matematika
# =============================================================================
print("=" * 78)
print("[1/4] TOKENIZER DAN SPECIAL TOKEN")
print("=" * 78)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
BASE_VOCAB_SIZE = len(tokenizer)
added_count = tokenizer.add_special_tokens(SPECIAL_TOKENS)
NEW_TOKEN_IDS = tokenizer.convert_tokens_to_ids(NEW_TOKEN_LIST)
print(f"Vocab awal: {BASE_VOCAB_SIZE} | token ditambah: {added_count} | vocab sekarang: {len(tokenizer)}")
print("ID token baru:", dict(zip(NEW_TOKEN_LIST, NEW_TOKEN_IDS)))
assert len(set(NEW_TOKEN_IDS)) == 4 and tokenizer.unk_token_id not in NEW_TOKEN_IDS, "Special token tidak terdaftar benar."

# Simpan tokenizer lebih awal ke STAGE1_DIR, STAGE2_DIR, dan FINAL_DIR
# Ini menjamin spiece.model dan tokenizer.json selalu tersedia bahkan jika training terputus
for _target_dir in (STAGE1_DIR, STAGE2_DIR, FINAL_DIR):
    os.makedirs(_target_dir, exist_ok=True)
    tokenizer.save_pretrained(_target_dir, legacy_format=True)
print("[OK] Tokenizer dengan special tokens disimpan ke STAGE1_DIR, STAGE2_DIR, dan FINAL_DIR.")
probe_ids = tokenizer("<answer> Dinasti </answer> <formula> A = p × l </formula>")["input_ids"]
for token_id in NEW_TOKEN_IDS:
    assert probe_ids.count(token_id) == 1, "Special token harus menjadi tepat satu id per kemunculan."
print("Special token tokenisasi sebagai satu id per kemunculan: OK")

MATH_SYMBOLS = list("×÷²³√∑∏∫πρσμλΔθΩ±≤≥≠≈→°^_{}[]()|/\\%+-·∞∂")
unk_symbols = [ch for ch in MATH_SYMBOLS if tokenizer.unk_token_id in tokenizer(ch, add_special_tokens=False)["input_ids"]]
print("Simbol matematika yang menjadi <unk> pada tokenizer mT5:", unk_symbols if unk_symbols else "tidak ada")

print("\n" + "=" * 78)
print("[2/4] PEMBENTUKAN DATASET DAN FILTER PANJANG TOKEN")
print("=" * 78)
def filter_rows_by_token_length(rows, name):
    if not rows:
        return rows
    answer_lengths = [len(ids) for ids in tokenizer([r["answer"] for r in rows], add_special_tokens=False)["input_ids"]]
    question_lengths = [len(ids) for ids in tokenizer([r["question"] for r in rows], add_special_tokens=False)["input_ids"]]
    kept = [r for r, a, q in zip(rows, answer_lengths, question_lengths) if a <= MAX_ANSWER_TOKENS and q <= MAX_TARGET_LENGTH - 1]
    print(f"  {name:<22}: {len(rows):>6,} -> {len(kept):>6,}")
    return kept


stage1_rows = filter_rows_by_token_length(stage1_train_rows, "stage1_train")
stage2_rows = filter_rows_by_token_length(stage2_train_rows, "stage2_train")
val_general_rows = filter_rows_by_token_length(val_general, "val_general")
test_general_rows = filter_rows_by_token_length(test_general, "test_general")
formula_seen_rows = filter_rows_by_token_length(raw_formula_eval_seen, "val_formula_seen")
formula_unseen_rows = filter_rows_by_token_length(raw_formula_eval_unseen, "val_formula_unseen")
slide_eval_rows = filter_rows_by_token_length(raw_slide_eval, "val_slide")
formula_eval_rows = formula_seen_rows + formula_unseen_rows


def tokenize_batch(batch):
    prompts = [
        make_length_safe_prompt(lang, context, answer, kind, bool(has_formula), start)
        for lang, context, answer, kind, has_formula, start in zip(
            batch["lang"], batch["context"], batch["answer"], batch["answer_type"], batch["has_formula"], batch["answer_start"])
    ]
    model_inputs = tokenizer(prompts, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["question"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def tokenize_rows(rows, description):
    dataset = Dataset.from_list(to_schema(rows))
    return dataset.map(tokenize_batch, batched=True, batch_size=256, remove_columns=dataset.column_names, desc=description)


tok_stage1_train = tokenize_rows(stage1_rows, "Tokenisasi stage1_train")
tok_stage2_train = tokenize_rows(stage2_rows, "Tokenisasi stage2_train")
tok_val_general = tokenize_rows(val_general_rows, "Tokenisasi val_general")
tok_test_general = tokenize_rows(test_general_rows, "Tokenisasi test_general")
tok_val_formula_seen = tokenize_rows(formula_seen_rows, "Tokenisasi val_formula_seen")
tok_val_formula_unseen = tokenize_rows(formula_unseen_rows, "Tokenisasi val_formula_unseen")
tok_val_formula = concatenate_datasets([tok_val_formula_seen, tok_val_formula_unseen])
tok_val_slide = tokenize_rows(slide_eval_rows, "Tokenisasi val_slide")

# Dev-set pemilih checkpoint Stage 2. Memilih checkpoint hanya dengan val umum akan selalu memihak checkpoint
# yang paling dekat dengan Stage 1, sehingga tujuan Stage 2 (rumus dan slide) tidak pernah terpilih.
# Template formula unseen sengaja TIDAK dimasukkan agar tetap menjadi ukuran generalisasi yang jujur.
selection_general = random.Random(SEED).sample(val_general_rows, min(SELECTION_GENERAL_ROWS, len(val_general_rows)))
tok_val_stage2_mix = concatenate_datasets([
    tokenize_rows(selection_general, "Tokenisasi dev general"), tok_val_formula_seen, tok_val_slide,
])

dataset_sizes = pd.DataFrame({
    "baris": {name: len(ds) for name, ds in (
        ("tok_stage1_train", tok_stage1_train), ("tok_stage2_train", tok_stage2_train), ("tok_val_general", tok_val_general),
        ("tok_test_general", tok_test_general), ("tok_val_formula (seen+unseen)", tok_val_formula),
        ("tok_val_formula_unseen", tok_val_formula_unseen), ("tok_val_slide", tok_val_slide),
        ("tok_val_stage2_mix (pemilih checkpoint)", tok_val_stage2_mix))}
})
display(dataset_sizes)

unk_id = tokenizer.unk_token_id
total_tokens = sum(len(ids) for ids in tok_stage2_train["input_ids"])
unk_tokens = sum(ids.count(unk_id) for ids in tok_stage2_train["input_ids"])
print(f"\nProporsi <unk> pada input Stage 2: {unk_tokens:,}/{total_tokens:,} = {unk_tokens / max(1, total_tokens):.4%}")
if unk_tokens / max(1, total_tokens) > 0.005:
    print("PERINGATAN: proporsi <unk> tinggi. Periksa simbol pada daftar unk_symbols di atas.")

print("\n" + "=" * 78)
print("[3/4] POST-PREPROCESSING EDA: KOMPOSISI STAGE 1 vs STAGE 2")
print("=" * 78)
composition_records = []
for stage_name, rows in (("Stage 1", stage1_rows), ("Stage 2", stage2_rows)):
    for row in rows:
        composition_records.append({"stage": stage_name, "answer_type": row["answer_type"], "lang": row["lang"], "ctx_format": row["ctx_format"]})
composition_df = pd.DataFrame(composition_records)
type_composition = pd.crosstab(composition_df["stage"], composition_df["answer_type"])
lang_composition = pd.crosstab(composition_df["stage"], composition_df["lang"])
format_composition = pd.crosstab(composition_df["stage"], composition_df["ctx_format"])
for title, table in (("answer_type", type_composition), ("bahasa", lang_composition), ("format konteks", format_composition)):
    print(f"\nKomposisi {title}:")
    display(table)
figure, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, (title, table) in zip(axes, (("answer_type", type_composition), ("Bahasa", lang_composition), ("Format konteks", format_composition))):
    (table.div(table.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axis, rot=0)
    axis.set_title(f"Komposisi {title} (%)")
    axis.set_ylabel("persen")
figure.suptitle("Komposisi data Stage 1 vs Stage 2")
figure.savefig(os.path.join(EDA_DIR, "stage_composition.png"), dpi=150, bbox_inches="tight")
plt.show()
composition_df.groupby(["stage", "answer_type", "lang"]).size().rename("baris").reset_index().to_csv(
    os.path.join(EDA_DIR, "stage_composition.csv"), index=False)

print("\n" + "=" * 78)
print("[4/4] CONTOH PROMPT TERDECODE (special token terlihat)")
print("=" * 78)
def first_row(rows, **conditions):
    for row in rows:
        if all(row.get(key) == value for key, value in conditions.items()):
            return row
    return None


examples = [
    ("(a) General ID", first_row(stage1_rows, lang="id")),
    ("(b) General EN", first_row(stage1_rows, lang="en")),
    ("(c) Formula target", first_row(stage2_rows, role="formula", lang="id")),
    ("(d) Number dalam konteks formula", first_row(stage2_rows, role="result", lang="en")),
    ("(e) Slide-style", first_row(stage2_rows, ctx_format="slide", lang="id")),
]
for title, row in examples:
    prompt = make_length_safe_prompt(row["lang"], row["context"], row["answer"], row["answer_type"], row["has_formula"], row["answer_start"])
    decoded = tokenizer.decode(tokenizer(prompt, max_length=MAX_INPUT_LENGTH, truncation=True)["input_ids"], skip_special_tokens=False)
    print(f"\n{title}")
    print("  INPUT :", decoded[:560])
    print("  TARGET:", row["question"])
print("\n[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())

## Cell 8. Modelling: two-stage curriculum fine-tuning

`train_worker()` menjalankan dua tahap berurutan dalam satu peluncuran DDP.

Stage 1 melatih mt5-base pada prosa bersih selama 2 epoch dengan lr 3e-4, jadwal cosine, evaluasi tiap 250 step, dan menyimpan model terbaik ke `STAGE1_DIR`. Stage 2 memuat model itu ulang, lalu melatih pada formula kontras, slide-style, dan replay selama 2 epoch dengan lr 1e-4 dan jadwal cosine. Model terbaik disimpan ke `FINAL_DIR` beserta tokenizer, log CSV, dan `training_metrics.json`.

Pemilih checkpoint Stage 2 memakai dev-set campuran, bukan validasi umum saja. Evaluasi akhir dijalankan pada validasi umum, test umum, formula (seen dan unseen), dan slide.

In [ ]:
# =============================================================================
# CELL 8. Modelling: Two-Stage Curriculum Fine-Tuning (DDP T4 x 2)
# =============================================================================
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_predictions = [normalize_space(t) for t in tokenizer.batch_decode(predictions, skip_special_tokens=True)]
    decoded_labels = [normalize_space(t) for t in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    rouge = rouge_metric.compute(predictions=decoded_predictions, references=decoded_labels, use_stemmer=False)
    bleu = bleu_metric.compute(predictions=decoded_predictions, references=[[label] for label in decoded_labels])
    generated_lengths = [int(np.count_nonzero(row != tokenizer.pad_token_id)) for row in predictions]
    return {
        "rouge1": round(rouge["rouge1"] * 100, 4),
        "rouge2": round(rouge["rouge2"] * 100, 4),
        "rougeL": round(rouge["rougeL"] * 100, 4),
        "bleu4": round(float(bleu["score"]), 4),
        "gen_len": round(float(np.mean(generated_lengths)), 4),
    }


def init_new_token_embeddings(model):
    """Inisialisasi embedding token baru dari rata-rata embedding kosakata asli ditambah derau kecil.
    Baris embedding tambahan milik mT5 asli tidak pernah dilatih, sehingga tidak boleh dipakai apa adanya."""
    assert min(NEW_TOKEN_IDS) >= BASE_VOCAB_SIZE, "Token baru diasumsikan berada di akhir kosakata."
    generator = torch.Generator(device="cpu").manual_seed(SEED)
    with torch.no_grad():
        for module in (model.get_input_embeddings(), model.get_output_embeddings()):
            if module is None:
                continue
            weight = module.weight
            reference = weight[:BASE_VOCAB_SIZE]
            mean_vector = reference.mean(dim=0)
            scale = float(reference.float().std()) * 0.1
            for token_id in NEW_TOKEN_IDS:
                noise = torch.randn(weight.shape[1], generator=generator) * scale
                weight[token_id] = mean_vector + noise.to(device=weight.device, dtype=weight.dtype)


def configure_generation(model):
    config = model.generation_config
    config.num_beams = 4
    config.length_penalty = DECODE_CONFIG["length_penalty"]
    config.no_repeat_ngram_size = DECODE_CONFIG["no_repeat_ngram_size"]
    config.min_new_tokens = 4
    config.max_length = MAX_TARGET_LENGTH
    config.early_stopping = True
    config.use_cache = True   # use_cache=False pada fase 2 membuat evaluasi generatif jauh lebih lambat


def make_training_args(output_dir, epochs, learning_rate, eval_steps, n_train_rows):
    steps_per_epoch = math.ceil(n_train_rows / TARGET_EFFECTIVE_BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(round(total_steps * WARMUP_RATIO)))   # setara warmup_ratio=0.10
    requested = dict(
        output_dir=output_dir, num_train_epochs=epochs, learning_rate=learning_rate, lr_scheduler_type="cosine",
        warmup_steps=warmup_steps, weight_decay=WEIGHT_DECAY, label_smoothing_factor=LABEL_SMOOTHING,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, eval_strategy="steps", eval_steps=eval_steps,
        save_strategy="steps", save_steps=eval_steps, save_total_limit=2, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="rougeL", greater_is_better=True, logging_steps=25,
        predict_with_generate=True, generation_max_length=MAX_TARGET_LENGTH, generation_num_beams=4,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        fp16=False, bf16=False, optim="adamw_torch", max_grad_norm=1.0, ddp_find_unused_parameters=False,
        ddp_timeout=7200, dataloader_num_workers=2, dataloader_pin_memory=True, report_to="none",
        seed=SEED, data_seed=SEED,
    )
    valid_fields = {field.name for field in dataclasses.fields(Seq2SeqTrainingArguments)}
    dropped = sorted(set(requested) - valid_fields)
    if dropped:
        print(f"[PERINGATAN] argumen tidak dikenali versi transformers ini dan dilewati: {dropped}", flush=True)
    return Seq2SeqTrainingArguments(**{k: v for k, v in requested.items() if k in valid_fields}), total_steps, warmup_steps


def build_trainer(model, args, train_dataset, eval_dataset):
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)
    return Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset,
        processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics,
    )


def release_trainer(trainer):
    try:
        trainer.accelerator.free_memory()
    except Exception:
        pass
    del trainer
    gc.collect()
    torch.cuda.empty_cache()


def clean_metric_dict(metrics):
    return {k: (round(float(v), 5) if isinstance(v, (int, float, np.floating)) else v) for k, v in metrics.items()}


def train_worker():
    """Dijalankan oleh setiap proses DDP. Stage 1 lalu Stage 2 berurutan dalam satu peluncuran."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    for directory in (STAGE1_DIR, STAGE2_DIR, FINAL_DIR):
        os.makedirs(directory, exist_ok=True)
    summary = {"config": {
        "model": MODEL_NAME, "stage1_epochs": STAGE1_EPOCHS, "stage1_lr": STAGE1_LR, "stage2_epochs": STAGE2_EPOCHS,
        "stage2_lr": STAGE2_LR, "effective_batch": TARGET_EFFECTIVE_BATCH_SIZE, "label_smoothing": LABEL_SMOOTHING,
        "stage1_rows": len(tok_stage1_train), "stage2_rows": len(tok_stage2_train),
    }}

    # ------------------------------ STAGE 1 ------------------------------
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.config.tie_word_embeddings = False
    model.resize_token_embeddings(len(tokenizer))
    init_new_token_embeddings(model)
    configure_generation(model)
    args1, total1, warmup1 = make_training_args(CHECKPOINT_DIR_STAGE1, STAGE1_EPOCHS, STAGE1_LR, STAGE1_EVAL_STEPS, len(tok_stage1_train))
    trainer1 = build_trainer(model, args1, tok_stage1_train, tok_val_general)
    is_main = trainer1.is_world_process_zero()
    if is_main:
        print(f"[STAGE 1] {len(tok_stage1_train):,} baris | total step {total1} | warmup {warmup1} | lr {STAGE1_LR}", flush=True)
    train_result1 = trainer1.train()
    stage1_val = trainer1.evaluate(eval_dataset=tok_val_general, metric_key_prefix="val_general")
    stage1_log = list(trainer1.state.log_history)
    summary["stage1"] = {
        "train": clean_metric_dict(train_result1.metrics), "val_general": clean_metric_dict(stage1_val),
        "best_checkpoint": trainer1.state.best_model_checkpoint, "best_metric_rougeL": trainer1.state.best_metric,
    }
    if is_main:
        trainer1.save_model(STAGE1_DIR)
        tokenizer.save_pretrained(STAGE1_DIR, legacy_format=True)
        pd.DataFrame(stage1_log).to_csv(os.path.join(FINAL_DIR, "stage1_log_history.csv"), index=False)
        print(f"[STAGE 1] selesai. rougeL val_general={stage1_val.get('val_general_rougeL')}", flush=True)
    trainer1.accelerator.wait_for_everyone()
    release_trainer(trainer1)
    if is_main:
        shutil.rmtree(CHECKPOINT_DIR_STAGE1, ignore_errors=True)   # hemat penyimpanan Kaggle
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # ------------------------------ STAGE 2 ------------------------------
    model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)
    model.config.tie_word_embeddings = False
    assert model.get_input_embeddings().num_embeddings == len(tokenizer), "Ukuran embedding Stage 1 tidak cocok dengan tokenizer."
    configure_generation(model)
    args2, total2, warmup2 = make_training_args(CHECKPOINT_DIR_STAGE2, STAGE2_EPOCHS, STAGE2_LR, STAGE2_EVAL_STEPS, len(tok_stage2_train))
    trainer2 = build_trainer(model, args2, tok_stage2_train, tok_val_stage2_mix)
    if is_main:
        print(f"[STAGE 2] {len(tok_stage2_train):,} baris | total step {total2} | warmup {warmup2} | lr {STAGE2_LR}", flush=True)
    train_result2 = trainer2.train()
    final_metrics = {}
    for name, dataset_obj in (
        ("val_general", tok_val_general), ("test_general", tok_test_general), ("val_formula", tok_val_formula),
        ("val_formula_seen", tok_val_formula_seen), ("val_formula_unseen", tok_val_formula_unseen), ("val_slide", tok_val_slide),
    ):
        final_metrics[name] = clean_metric_dict(trainer2.evaluate(eval_dataset=dataset_obj, metric_key_prefix=name))
    stage2_log = list(trainer2.state.log_history)
    summary["stage2"] = {
        "train": clean_metric_dict(train_result2.metrics),
        "best_checkpoint": trainer2.state.best_model_checkpoint, "best_metric_rougeL_mix": trainer2.state.best_metric,
    }
    summary["final_eval"] = final_metrics
    if is_main:
        trainer2.save_model(STAGE2_DIR)
        trainer2.save_model(FINAL_DIR)
        tokenizer.save_pretrained(STAGE2_DIR, legacy_format=True)
        tokenizer.save_pretrained(FINAL_DIR, legacy_format=True)
        pd.DataFrame(stage2_log).to_csv(os.path.join(FINAL_DIR, "stage2_log_history.csv"), index=False)
        with open(os.path.join(FINAL_DIR, "training_metrics.json"), "w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2, ensure_ascii=False, default=str)
        shutil.rmtree(CHECKPOINT_DIR_STAGE2, ignore_errors=True)
        print("[STAGE 2] selesai. Ringkasan:", json.dumps(final_metrics, indent=1)[:1500], flush=True)
    trainer2.accelerator.wait_for_everyone()


print("train_worker siap. Dua tahap akan dijalankan berurutan dalam satu peluncuran DDP pada Cell 9.")
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized())

## Cell 9. Peluncuran DDP dan visualisasi kurva training

Menjalankan `notebook_launcher` dengan dua proses, lalu membaca log CSV dan membuat grafik training loss versus validation loss, ROUGE-1, ROUGE-2, ROUGE-L, dan BLEU-4 lintas dua tahap. Garis putus-putus vertikal menandai batas Stage 1 ke Stage 2. Dev-set kedua tahap berbeda, jadi angka validasi tidak boleh dibandingkan langsung lintas batas itu.

In [ ]:
# =============================================================================
# CELL 9. Peluncuran DDP dan visualisasi kurva training
# =============================================================================
from accelerate import notebook_launcher

assert not torch.cuda.is_initialized(), "CUDA sudah aktif di proses utama. Restart session lalu jalankan ulang dari Cell 2."
print(f"Meluncurkan training pada {NUM_GPUS} GPU. Estimasi total beberapa jam; jangan tutup tab Kaggle.")
launch_started = time.time()
notebook_launcher(train_worker, args=(), num_processes=NUM_GPUS, mixed_precision="no", use_port="29525")
print(f"Training selesai dalam {(time.time() - launch_started) / 60:.1f} menit.")
print("[cek] CUDA aktif proses utama setelah launcher:", torch.cuda.is_initialized())

stage1_log_path = os.path.join(FINAL_DIR, "stage1_log_history.csv")
stage2_log_path = os.path.join(FINAL_DIR, "stage2_log_history.csv")
metrics_path = os.path.join(FINAL_DIR, "training_metrics.json")
for path in (stage1_log_path, stage2_log_path, metrics_path):
    assert os.path.exists(path), f"File hasil training tidak ditemukan: {path}"
stage1_logs = pd.read_csv(stage1_log_path)
stage2_logs = pd.read_csv(stage2_log_path)
with open(metrics_path, encoding="utf-8") as handle:
    training_metrics = json.load(handle)

stage1_last_step = int(stage1_logs["step"].max())
stage1_logs["global_step"] = stage1_logs["step"]
stage2_logs["global_step"] = stage2_logs["step"] + stage1_last_step


def series(frame, column):
    subset = frame.dropna(subset=[column])
    return subset["global_step"], subset[column]


figure, axes = plt.subplots(2, 2, figsize=(16, 10))
for frame, label, color in ((stage1_logs, "Stage 1", "#3b7dd8"), (stage2_logs, "Stage 2", "#d8643b")):
    if "loss" in frame:
        axes[0, 0].plot(*series(frame, "loss"), color=color, alpha=0.8, label=f"train loss ({label})")
    if "eval_loss" in frame:
        axes[0, 0].plot(*series(frame, "eval_loss"), color=color, linestyle="--", marker="o", label=f"validation loss ({label})")
    for column, axis_index, title in (("eval_rouge1", (0, 1), "ROUGE-1"), ("eval_rouge2", (0, 1), "ROUGE-2"), ("eval_rougeL", (0, 1), "ROUGE-L")):
        if column in frame:
            axes[axis_index].plot(*series(frame, column), marker="o", label=f"{title} ({label})", alpha=0.85)
    if "eval_bleu4" in frame:
        axes[1, 0].plot(*series(frame, "eval_bleu4"), marker="o", color=color, label=f"BLEU-4 ({label})")
    if "eval_gen_len" in frame:
        axes[1, 1].plot(*series(frame, "eval_gen_len"), marker="o", color=color, label=f"panjang token hasil ({label})")
for axis, title in zip(axes.flat, ("Training loss vs validation loss", "ROUGE-1, ROUGE-2, ROUGE-L", "BLEU-4", "Panjang rata-rata hasil generasi")):
    axis.axvline(stage1_last_step, color="gray", linestyle=":", label="batas Stage 1 -> Stage 2")
    axis.set_title(title)
    axis.set_xlabel("global step")
    axis.legend(fontsize=7)
figure.suptitle("Kurva training dua tahap. Catatan: dev-set Stage 1 berbeda dengan Stage 2 (campuran umum, formula, slide).")
figure.savefig(os.path.join(EDA_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nRingkasan metrik akhir (model terbaik Stage 2):")
display(pd.DataFrame(training_metrics["final_eval"]).T)


## Cell 10. Evaluasi mendalam, breakdown per kategori, dan contrastive audit

Memuat model dari `FINAL_DIR` ke `cuda:0`. Berisi: tabel metrik per bahasa, tipe jawaban, dan format konteks (ROUGE, BLEU-4, Type Alignment, Leakage, Quality pass), audit sensitivitas jawaban pada 30 konteks, audit kontras rumus versus angka pada template yang tidak dilihat saat training, analisis kualitatif, dan tabel ablasi dekode.

Tabel ablasi menjawab pertanyaan apakah perubahan dekode dan reranking benar-benar membantu. Jika tidak, jangan dipertahankan.

In [ ]:
# =============================================================================
# Mesin inferensi: SATU-SATUNYA bagian yang menyentuh model dan GPU
# =============================================================================
def generate_texts(prompts, batch_size=16, num_return_sequences=1, config=None):
    """Kembalikan list berisi list kandidat hasil generasi untuk setiap prompt (urutan input dipertahankan)."""
    settings = dict(DECODE_CONFIG if config is None else config)
    settings["num_beams"] = max(int(settings.get("num_beams", 1)), num_return_sequences)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)
    with torch.inference_mode():
        for offset in range(0, len(order), batch_size):
            indices = order[offset:offset + batch_size]
            encoded = infer_tokenizer(
                [prompts[i] for i in indices], max_length=MAX_INPUT_LENGTH, truncation=True, padding=True, return_tensors="pt"
            ).to(inference_device)
            generated = infer_model.generate(**encoded, num_return_sequences=num_return_sequences, **settings)
            decoded = infer_tokenizer.batch_decode(generated, skip_special_tokens=True)
            for position, index in enumerate(indices):
                outputs[index] = decoded[position * num_return_sequences:(position + 1) * num_return_sequences]
    return outputs


# =============================================================================
# Logika di atas mesin inferensi (dapat diuji tanpa model)
# =============================================================================
def prompt_for_row(row):
    return make_length_safe_prompt(row["lang"], row["context"], row["answer"], row["answer_type"],
                                   bool(row["has_formula"]), row["answer_start"], tok=infer_tokenizer)


def predict_rows(rows, batch_size=24, config=None):
    outputs = generate_texts([prompt_for_row(r) for r in rows], batch_size=batch_size, config=config)
    return [clean_question(o[0]) for o in outputs]


def generate_best_question(context, answer, lang, answer_start=-1, n_best=6):
    """Hasilkan n-best lalu pilih kandidat pertama yang lolos filter kualitas, grounding, dan tipe jawaban."""
    context, answer = normalize_space(context), normalize_space(answer)
    kind = answer_type(answer)
    prompt = make_length_safe_prompt(lang, context, answer, kind, detect_formula_content(context)[0], answer_start, tok=infer_tokenizer)
    raw = generate_texts([prompt], num_return_sequences=n_best, config={**DECODE_CONFIG, "num_beams": max(8, n_best)})[0]
    candidates = []
    for text in raw:
        cleaned = clean_question(text)
        if cleaned and cleaned not in candidates:
            candidates.append(cleaned)
    evaluated = []
    for cleaned in candidates:
        ok, reasons = question_quality(cleaned, context, answer, lang, strict=True)
        if type_alignment(cleaned, answer, lang, kind) is False:
            ok, reasons = False, reasons + ["type_mismatch"]
        evaluated.append((cleaned, ok, reasons))
    for cleaned, ok, reasons in evaluated:
        if ok:
            return {"question": cleaned, "ok": True, "reasons": [], "low_confidence": False, "candidates": candidates}
    if not evaluated:
        return {"question": None, "ok": False, "reasons": ["no_candidate"], "low_confidence": True, "candidates": []}
    best = min(evaluated, key=lambda item: len(item[2]))
    return {"question": best[0], "ok": False, "reasons": best[2], "low_confidence": True, "candidates": candidates}


def _nan_if_none(value):
    return np.nan if value is None else float(value)


def build_eval_frame(rows, predictions, set_name, format_name):
    frame = pd.DataFrame(to_schema(rows))[["lang", "context", "answer", "answer_type", "question", "role", "group_id", "eval_split"]].copy()
    frame["pred"] = predictions
    frame["set"] = set_name
    frame["format"] = format_name
    aligned, leak, quality, reasons = [], [], [], []
    for row, pred in zip(frame.to_dict("records"), predictions):
        aligned.append(_nan_if_none(type_alignment(pred, row["answer"], row["lang"], row["answer_type"])))
        leak.append(float(answer_leaks(pred, row["answer"])))
        ok, why = question_quality(pred, row["context"], row["answer"], row["lang"], strict=True)
        quality.append(float(ok))
        reasons.append(",".join(why))
    frame["aligned"], frame["leak"], frame["quality_ok"], frame["reasons"] = aligned, leak, quality, reasons
    frame["short_or_dangling"] = frame["reasons"].str.contains("too_short|dangling_end|no_grounding").astype(float)
    return frame


def score_group(frame):
    if len(frame) == 0:
        return {}
    rouge = rouge_metric.compute(predictions=list(frame["pred"]), references=list(frame["question"]), use_stemmer=False)
    bleu = bleu_metric.compute(predictions=list(frame["pred"]), references=[[q] for q in frame["question"]])
    applicable = frame["aligned"].dropna()
    return {
        "n": len(frame), "ROUGE-1": round(rouge["rouge1"] * 100, 2), "ROUGE-2": round(rouge["rouge2"] * 100, 2),
        "ROUGE-L": round(rouge["rougeL"] * 100, 2), "BLEU-4": round(float(bleu["score"]), 2),
        "Type Alignment (%)": round(100 * applicable.mean(), 1) if len(applicable) else np.nan,
        "Leakage (%)": round(100 * frame["leak"].mean(), 1), "Quality pass (%)": round(100 * frame["quality_ok"].mean(), 1),
        "Pendek/buntung/tak berpijak (%)": round(100 * frame["short_or_dangling"].mean(), 1),
    }


def breakdown_table(frame, by):
    records = []
    for key, group in frame.groupby(by):
        records.append({**({by: key} if isinstance(by, str) else dict(zip(by, key))), **score_group(group)})
    return pd.DataFrame(records)


def run_answer_sensitivity_audit(rows, n_contexts=30, seed=SEED, threshold=95.0):
    by_context = defaultdict(list)
    for row in rows:
        by_context[row["context"]].append(row)
    eligible = [group for group in by_context.values() if len({r["answer"] for r in group}) >= 2]
    random.Random(seed).shuffle(eligible)
    selected = eligible[:n_contexts]
    records = []
    for group in selected:
        first = group[0]
        second = next(r for r in group if r["answer"] != first["answer"])
        question_a, question_b = predict_rows([first, second])
        records.append({
            "lang": first["lang"], "answer_a": first["answer"], "question_a": question_a,
            "answer_b": second["answer"], "question_b": question_b,
            "changed": normalize_space(question_a).casefold() != normalize_space(question_b).casefold(),
        })
    rate = 100 * float(np.mean([r["changed"] for r in records])) if records else float("nan")
    return {"n_contexts": len(records), "changed_output_rate": round(rate, 2), "threshold": threshold,
            "passed": bool(records) and rate > threshold, "details": records}


_LAW_CUES = ("hukum", "teorema", "aturan", "law", "theorem", "rule")


def run_contrastive_audit(rows, n_per_lang=5, seed=SEED):
    groups = defaultdict(list)
    for row in rows:
        if row.get("group_id"):
            groups[row["group_id"]].append(row)
    eligible = [g for g in groups.values() if {"formula", "result"} <= {r["role"] for r in g} and ({"variable", "law"} & {r["role"] for r in g})]
    chosen = []
    for lang in ("id", "en"):
        lang_groups = [g for g in eligible if g[0]["lang"] == lang]
        random.Random(seed).shuffle(lang_groups)
        chosen.extend(lang_groups[:n_per_lang])
    records = []
    for group_index, group in enumerate(chosen, start=1):
        by_role = {}
        for row in group:
            by_role.setdefault(row["role"], row)
        concept_row = by_role.get("law") or by_role["variable"]
        trio = [by_role["formula"], by_role["result"], concept_row]
        predictions = predict_rows(trio)
        lang = trio[0]["lang"]
        formula_ok = type_alignment(predictions[0], trio[0]["answer"], lang, "formula") is True
        result_ok = (type_alignment(predictions[1], trio[1]["answer"], lang, "number") is True
                     and type_alignment(predictions[1], "x", lang, "formula") is False)
        if concept_row["role"] == "law":
            concept_ok = any(cue in predictions[2].casefold() for cue in _LAW_CUES)
        else:
            concept_ok = type_alignment(predictions[2], concept_row["answer"], lang, "number") is True
        distinct_ok = len({normalize_space(p).casefold() for p in predictions}) == 3
        for role_name, row, pred, ok in (("formula", trio[0], predictions[0], formula_ok), ("result", trio[1], predictions[1], result_ok),
                                         (concept_row["role"], concept_row, predictions[2], concept_ok)):
            records.append({"konteks": group_index, "lang": lang, "role": role_name, "answer": row["answer"], "pertanyaan_model": pred,
                            "sesuai_peran": ok, "ketiganya_berbeda": distinct_ok})
    table = pd.DataFrame(records)
    per_context = table.groupby("konteks").agg(sesuai=("sesuai_peran", "all"), berbeda=("ketiganya_berbeda", "first"))
    pass_rate = 100 * float((per_context["sesuai"] & per_context["berbeda"]).mean()) if len(per_context) else float("nan")
    identical_rate = 100 * float((~per_context["berbeda"]).mean()) if len(per_context) else float("nan")
    return table, {"n_contexts": int(len(per_context)), "contrastive_pass_rate": round(pass_rate, 1), "identical_question_rate": round(identical_rate, 1)}


# =============================================================================
# Eksekusi evaluasi mendalam
# =============================================================================
inference_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Validasi keberadaan file bobot model hasil training sebelum inferensi
config_path = os.path.join(FINAL_DIR, "config.json")
if not os.path.exists(config_path):
    raise FileNotFoundError(
        f"Direktori '{FINAL_DIR}' belum memiliki berkas model ('config.json' tidak ditemukan). "
        "Pastikan Cell 9 (DDP Training) sudah dijalankan hingga selesai 100% sebelum mengeksekusi Cell 10!"
    )

try:
    infer_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR, use_fast=True)
except Exception as _tok_err:
    print(f"[INFO] Fast tokenizer gagal ({_tok_err}), beralih ke use_fast=False (SentencePiece slow)...")
    infer_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR, use_fast=False)

infer_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(inference_device).eval()
assert len(infer_tokenizer) == len(tokenizer), "Tokenizer hasil training tidak cocok dengan tokenizer notebook."
print("Model final dimuat dari", FINAL_DIR, "| parameter:", f"{sum(p.numel() for p in infer_model.parameters()) / 1e6:.0f} juta")

print("\n" + "=" * 78)
print("1. GRANULAR METRIC BREAKDOWN")
print("=" * 78)
EVAL_SETS = [
    ("val", "Prose General", val_general_rows),
    ("test", "Prose General", test_general_rows),
    ("formula_seen", "Formula Context", formula_seen_rows),
    ("formula_unseen", "Formula Context", formula_unseen_rows),
    ("slide", "Slide-Style PPT", slide_eval_rows),
]
frames = []
for set_name, format_name, rows in EVAL_SETS:
    started = time.time()
    predictions = predict_rows(rows)
    frames.append(build_eval_frame(rows, predictions, set_name, format_name))
    print(f"  {set_name:<15} {len(rows):>5} baris dalam {time.time() - started:.0f} detik")
eval_frame = pd.concat(frames, ignore_index=True)

tables = {
    "per_set": breakdown_table(eval_frame, "set"),
    "per_bahasa": breakdown_table(eval_frame, "lang"),
    "per_tipe_jawaban": breakdown_table(eval_frame, "answer_type"),
    "per_format_konteks": breakdown_table(eval_frame, "format"),
    "per_format_dan_bahasa": breakdown_table(eval_frame, ["format", "lang"]),
    "per_format_dan_tipe": breakdown_table(eval_frame, ["format", "answer_type"]),
}
for name, table in tables.items():
    print(f"\n--- {name} ---")
    display(table)
    table.to_csv(os.path.join(EDA_DIR, f"eval_breakdown_{name}.csv"), index=False)
    table.to_csv(os.path.join(FINAL_DIR, f"eval_breakdown_{name}.csv"), index=False)
eval_frame.to_csv(os.path.join(FINAL_DIR, "eval_predictions_full.csv"), index=False)

figure, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.barplot(data=tables["per_format_dan_bahasa"], x="format", y="ROUGE-L", hue="lang", ax=axes[0])
axes[0].set_title("ROUGE-L per format konteks dan bahasa")
sns.barplot(data=tables["per_format_dan_tipe"], x="format", y="Quality pass (%)", hue="answer_type", ax=axes[1])
axes[1].set_title("Persentase pertanyaan lolos filter kualitas")
figure.savefig(os.path.join(EDA_DIR, "eval_breakdown.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n" + "=" * 78)
print("2. GENERAL ANSWER-SENSITIVITY AUDIT (30 konteks, answer berbeda pada konteks yang sama)")
print("=" * 78)
sensitivity = run_answer_sensitivity_audit(val_general_rows, n_contexts=30)
print(f"Changed Output Rate: {sensitivity['changed_output_rate']}% dari {sensitivity['n_contexts']} konteks | target > 95% | "
      f"{'LULUS' if sensitivity['passed'] else 'GAGAL'}")
print("Catatan: dengan 30 konteks, satu kegagalan saja menurunkan angka sebesar 3,3 poin. Perlakukan sebagai indikator kasar.")
display(pd.DataFrame(sensitivity["details"]).head(10))
with open(os.path.join(FINAL_DIR, "answer_sensitivity_audit.json"), "w", encoding="utf-8") as handle:
    json.dump(sensitivity, handle, indent=2, ensure_ascii=False)

print("\n" + "=" * 78)
print("3. CONTRASTIVE FORMULA-vs-NUMBER AUDIT (pengujian langsung blind spot Fase 2)")
print("=" * 78)
unseen_table, unseen_summary = run_contrastive_audit(formula_unseen_rows, n_per_lang=5)
seen_table, seen_summary = run_contrastive_audit(formula_seen_rows, n_per_lang=15)
print(f"Template TIDAK DILIHAT saat training : {unseen_summary}")
print(f"Template sama dengan training         : {seen_summary}")
pd.set_option("display.max_colwidth", 140)
display(unseen_table)

canonical_cases = [
    ("id", "Luas persegi panjang dihitung dengan rumus A = p × l. Jika panjang p = 8 cm dan lebar l = 5 cm, maka luasnya adalah 40 cm².",
     ["A = p × l", "40 cm²", "8 cm"]),
    ("id", "Hukum Ohm menyatakan bahwa tegangan listrik dinyatakan dengan persamaan V = I × R. Jika kuat arus I = 2 A dan hambatan R = 6 Ω, maka tegangan adalah 12 V.",
     ["V = I × R", "12 V", "Hukum Ohm"]),
    ("en", "Newton's second law states that force is given by F = m × a. If mass m = 5 kg and acceleration a = 3 m/s², then the force is 15 N.",
     ["F = m × a", "15 N", "Newton's second law"]),
]
canonical_records = []
for lang, context, answers in canonical_cases:
    for answer in answers:
        start = locate_answer(context, answer, -1)[0]
        row = {"lang": lang, "context": context, "answer": answer, "answer_start": start, "answer_type": answer_type(answer),
               "has_formula": True, "question": "", "role": "", "group_id": "", "eval_split": ""}
        canonical_records.append({"lang": lang, "answer": answer, "tipe": row["answer_type"], "pertanyaan_model": predict_rows([row])[0]})
canonical_table = pd.DataFrame(canonical_records)
print("\nKasus kanonik dari laporan Fase 2 (pertanyaan harus berbeda untuk jawaban rumus vs angka):")
display(canonical_table)
for lang, context, answers in canonical_cases:
    subset = canonical_table[(canonical_table["lang"] == lang) & (canonical_table["answer"].isin(answers))]
    print(f"  [{lang}] pertanyaan unik: {subset['pertanyaan_model'].nunique()} dari {len(subset)}")
contrastive_report = {"unseen": unseen_summary, "seen": seen_summary, "canonical": canonical_records,
                      "detail_unseen": unseen_table.to_dict("records")}
with open(os.path.join(FINAL_DIR, "contrastive_audit.json"), "w", encoding="utf-8") as handle:
    json.dump(contrastive_report, handle, indent=2, ensure_ascii=False, default=str)

print("\n" + "=" * 78)
print("4. QUALITATIVE ERROR ANALYSIS DAN ANSWER LEAKAGE AUDIT")
print("=" * 78)
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
eval_frame["rougeL_row"] = [scorer.score(ref, pred)["rougeL"].fmeasure for ref, pred in zip(eval_frame["question"], eval_frame["pred"])]
leak_rate = 100 * eval_frame["leak"].mean()
print(f"Answer Leakage Rate keseluruhan: {leak_rate:.2f}%")
display(eval_frame.groupby("set")["leak"].mean().mul(100).round(2).rename("Leakage (%)").to_frame())
show_columns = ["lang", "set", "answer", "question", "pred", "rougeL_row"]
print("\n5 prediksi terbaik (ROUGE-L tertinggi):")
display(eval_frame.sort_values("rougeL_row", ascending=False).head(5)[show_columns])
print("5 prediksi terburuk (ROUGE-L terendah):")
display(eval_frame.sort_values("rougeL_row", ascending=True).head(5)[show_columns])
reason_counts = Counter(r for value in eval_frame["reasons"] for r in str(value).split(",") if r and r != "nan")
print("Alasan kegagalan filter kualitas pada prediksi:", dict(reason_counts))
with open(os.path.join(FINAL_DIR, "error_analysis.json"), "w", encoding="utf-8") as handle:
    json.dump({"leakage_rate_percent": round(float(leak_rate), 3), "failure_reasons": dict(reason_counts),
               "worst": eval_frame.sort_values("rougeL_row").head(5)[show_columns].to_dict("records"),
               "best": eval_frame.sort_values("rougeL_row", ascending=False).head(5)[show_columns].to_dict("records")},
              handle, indent=2, ensure_ascii=False, default=str)

print("\n" + "=" * 78)
print("5. ABLASI DEKODE: apakah perubahan dekode dan reranking benar-benar membantu?")
print("=" * 78)
ablation_rows = (random.Random(SEED).sample(val_general_rows, min(120, len(val_general_rows)))
                 + random.Random(SEED).sample(slide_eval_rows, min(60, len(slide_eval_rows)))
                 + random.Random(SEED).sample(formula_unseen_rows, min(60, len(formula_unseen_rows))))
strategies = {}
strategies["A. dekode Fase 2 (beam4, no_repeat=2)"] = predict_rows(ablation_rows, config=PHASE2_DECODE_CONFIG)
strategies["B. dekode baru (beam5, length_penalty 1.3, min 5 token)"] = predict_rows(ablation_rows)
rerank_predictions = []
for row in tqdm(ablation_rows, desc="Rerank n-best"):
    result = generate_best_question(row["context"], row["answer"], row["lang"], row["answer_start"])
    rerank_predictions.append(result["question"] or "")
strategies["C. dekode baru + rerank n-best"] = rerank_predictions
ablation_records = []
for name, predictions in strategies.items():
    frame = build_eval_frame(ablation_rows, predictions, "ablation", "mixed")
    ablation_records.append({"strategi": name, **score_group(frame)})
ablation_table = pd.DataFrame(ablation_records)
display(ablation_table)
ablation_table.to_csv(os.path.join(FINAL_DIR, "decoding_ablation.csv"), index=False)
ablation_table.to_csv(os.path.join(EDA_DIR, "decoding_ablation.csv"), index=False)
print("Baca tabel ini dengan jujur: bila strategi C tidak lebih baik dari A pada 'Quality pass (%)', rerank tidak layak dipertahankan.")
print("[INFO] Evaluasi selesai. Model tetap di cuda:0 untuk Cell 11.")

## Cell 11. Pipeline prediksi dan pembuatan kuis otomatis

Fungsi `extract_candidate_answers` mengekstrak kandidat jawaban dari rumus, angka bersatuan, tanggal, definisi, dan istilah kunci, lalu memilih yang tersebar merata antar kalimat. `generate_distractors` membuat tiga pengecoh per tipe jawaban (mutasi operator dan variabel untuk rumus, kelipatan untuk angka, geser tahun untuk tanggal, frasa lain untuk teks). Rumus komutatif yang sebenarnya setara dengan kunci jawaban otomatis disaring. `build_quiz_from_material` menerima paragraf atau poin slide.

Tiga kasus uji dijalankan, termasuk Candi Borobudur untuk membuktikan apakah bug Fase 2 benar-benar hilang. Tabel regresi memperlihatkan keluaran beam biasa dan keluaran setelah rerank berdampingan supaya terlihat bagian mana yang menyembuhkan.

In [ ]:
# =============================================================================
# BAGIAN A. Deteksi bahasa ringan dan penyusunan konteks dari paragraf atau slide
# =============================================================================
_LANG_STOPWORDS = {
    "id": set("""yang dan di ke dari untuk pada dengan adalah ini itu dalam tidak akan atau juga oleh sebagai karena
        dapat bisa merupakan jika maka saat yaitu sebuah para antara lebih telah sudah ada mereka serta tetapi namun
        hanya setiap suatu dapat tersebut sehingga dimana bahwa""".split()),
    "en": set("""the of and to in is are was were that for with as by on at from this which be or an it its has have
        had can will not if then when where than their these those because also such into""".split()),
}


def guess_language(text, default="id"):
    tokens = re.findall(r"[a-zA-ZÀ-ÿ']+", str(text).lower())
    if not tokens:
        return default
    scores = {lang: sum(token in words for token in tokens) for lang, words in _LANG_STOPWORDS.items()}
    if scores["id"] == scores["en"]:
        return default
    return "id" if scores["id"] > scores["en"] else "en"


_BULLET_RE = re.compile(r"^\s*(?:[-•*▪►●]|\d+[.)])\s+")
_CONTINUATION_RE = re.compile(r"\s*[\(\[]?\s*(?:lanjutan|cont\.?|cont'd|continued|bagian\s*\d+|part\s*\d+|\d+\s*/\s*\d+)\s*[\)\]]?\s*$", re.IGNORECASE)
_NON_DECLARATIVE_LABELS = {"contoh", "catatan", "note", "example", "keterangan", "hint", "tips", "rumus", "formula", "info"}
_CONNECTORS = {
    "id": ["Selain itu,", "Kemudian,", "Selanjutnya,", "Berikutnya,"],
    "en": ["In addition,", "Furthermore,", "Next,", "Moreover,"],
}


def _ensure_period(text):
    text = normalize_space(text)
    if not text:
        return text
    return text if text[-1] in ".!?" else text + "."


def _declarative_bullet(point, lang):
    """Ubah 'Istilah: penjelasan' menjadi kalimat deklaratif bila istilahnya pendek dan bukan label umum."""
    point = normalize_space(re.sub(_BULLET_RE, "", point))
    match = re.match(r"^([^:]{2,40}):\s+(.{4,})$", point)
    if match:
        label, rest = match.group(1).strip(), match.group(2).strip()
        if len(label.split()) <= 4 and label.casefold() not in _NON_DECLARATIVE_LABELS and not re.search(r"\d", label):
            verb = "adalah" if lang == "id" else "is"
            return f"{label} {verb} {rest}"
    return point


def stitch_slide(title, points, notes, lang):
    """Gabungkan judul, semua poin, dan catatan pembicara menjadi satu paragraf kohesif."""
    title = normalize_space(title)
    cleaned = [_declarative_bullet(p, lang) for p in points if normalize_space(p)]
    if not cleaned and not normalize_space(notes):
        return ""
    lead = f"Pada materi {title}, dijelaskan bahwa:" if lang == "id" else f"In the topic of {title}, it is explained that:"
    if not title:
        lead = "Dijelaskan bahwa:" if lang == "id" else "It is explained that:"
    sentences = []
    for index, point in enumerate(cleaned):
        sentence = _ensure_period(point)
        if index == 0:
            sentences.append(f"{lead} {sentence}")
        else:
            sentences.append(f"{_CONNECTORS[lang][(index - 1) % len(_CONNECTORS[lang])]} {sentence}")
    if normalize_space(notes):
        label = "Catatan tambahan:" if lang == "id" else "Additional note:"
        sentences.append(f"{label} {_ensure_period(notes)}")
    return normalize_space(" ".join(sentences))


def _normalize_title(title):
    return normalize_space(_CONTINUATION_RE.sub("", normalize_space(title))).casefold()


def _parse_slide_text(block):
    lines = [line for line in str(block).splitlines() if line.strip()]
    title, points = "", []
    for line in lines:
        if _BULLET_RE.match(line):
            points.append(re.sub(_BULLET_RE, "", line).strip())
        elif not title:
            title = re.sub(r"^\s*slide\s*\d+\s*[:.\-]?\s*", "", line.strip(), flags=re.IGNORECASE)
        elif points:
            points[-1] = points[-1] + " " + line.strip()
        else:
            points.append(line.strip())
    return {"title": title, "points": points, "notes": ""}


def material_to_contexts(raw, lang=None, min_slide_words=25):
    """Ubah teks paragraf atau daftar slide menjadi konteks siap pakai. Slide pendek dengan judul sama digabung."""
    slides = None
    if isinstance(raw, (list, tuple)):
        slides = [item if isinstance(item, dict) else _parse_slide_text(item) for item in raw]
    elif isinstance(raw, str):
        bullet_lines = sum(bool(_BULLET_RE.match(line)) for line in raw.splitlines())
        if bullet_lines >= 2:
            blocks = [b for b in re.split(r"\n\s*\n|^\s*-{3,}\s*$", raw, flags=re.MULTILINE) if b.strip()]
            slides = [_parse_slide_text(b) for b in blocks]
    contexts = []
    if slides is not None:
        slides = [dict(title=s.get("title", ""), points=list(s.get("points", [])), notes=s.get("notes", "")) for s in slides]
        merged = []
        for slide in slides:
            words = len(" ".join(slide["points"]).split())
            if merged:
                previous = merged[-1]
                previous_words = len(" ".join(previous["points"]).split())
                same_title = _normalize_title(slide["title"]) == _normalize_title(previous["title"]) or not normalize_space(slide["title"])
                if same_title and (previous_words < min_slide_words or words < min_slide_words):
                    previous["points"].extend(slide["points"])
                    previous["notes"] = normalize_space(previous["notes"] + " " + slide["notes"])
                    continue
            merged.append(slide)
        for slide in merged:
            sample = " ".join([slide["title"]] + slide["points"] + [slide["notes"]])
            slide_lang = lang or guess_language(sample)
            text = stitch_slide(re.sub(_CONTINUATION_RE, "", slide["title"]), slide["points"], slide["notes"], slide_lang)
            if len(text.split()) >= 6:
                contexts.append({"context": text, "lang": slide_lang, "title": normalize_space(slide["title"]), "kind": "slide"})
        return contexts

    paragraphs = [normalize_space(p) for p in re.split(r"\n\s*\n", str(raw)) if normalize_space(p)]
    buffer = ""
    for paragraph in paragraphs:
        buffer = (buffer + " " + paragraph).strip()
        if len(buffer.split()) >= 40:
            contexts.append(buffer)
            buffer = ""
    if buffer:
        contexts.append(buffer)
    results = []
    for text in contexts:
        sentences = split_sentences(text)
        chunk, chunks = [], []
        for sentence in sentences:
            chunk.append(sentence)
            if len(" ".join(chunk).split()) >= 150:
                chunks.append(" ".join(chunk))
                chunk = []
        if chunk:
            chunks.append(" ".join(chunk))
        for piece in chunks:
            if len(piece.split()) >= 6:
                results.append({"context": piece, "lang": lang or guess_language(piece), "title": "", "kind": "paragraph"})
    return results


# =============================================================================
# BAGIAN B. Ekstraksi kandidat jawaban otomatis (tanpa input manual)
# =============================================================================
_NUMBER_UNIT_RE = re.compile(
    r"(?<![\w.,])[-+−]?\d+(?:[.,]\d+)*(?:\s?(?:%|°[CFK]?|" + _UNIT_BASE + r"[²³]?(?:[/·]" + _UNIT_BASE + r"[²³]?)?))"
    r"(?![A-Za-zΑ-Ωα-ω0-9_])"
)
_COUNT_NUMBER_RE = re.compile(r"(?<![\w.,/])(\d{1,4})\s+([A-Za-zÀ-ÿ]{4,})")
_YEAR_SEARCH_RE = re.compile(r"(?<![\d.,])(?:1[0-9]\d{2}|20\d{2})(?!\d)")
_YEAR_CONTEXT_RE = re.compile(r"(?:tahun|year|pada|in|sejak|since|dari|from|until|sampai|hingga|by|circa|sekitar|around|\()\s*$", re.IGNORECASE)
_DATE_SEARCH_RES = [
    re.compile(r"(?<![\w])(?:\d{1,2}\s+)?(?:" + _MONTHS + r")\.?\s+(?:\d{1,2},\s*)?" + _YEAR + r"(?![\w])", re.IGNORECASE),
    re.compile(r"(?<![\w])(?:0?[1-9]|[12]\d|3[01])[-/.](?:0?[1-9]|1[0-2])[-/.]" + _YEAR + r"(?![\w])"),
    re.compile(r"(?<![\w])abad\s+ke-?\s*(?:[IVXLC]+|\d+)", re.IGNORECASE),
    re.compile(r"(?<![\w])\d{1,2}(?:st|nd|rd|th)\s+century", re.IGNORECASE),
]
_DEFINITION_RULES = {
    "id": [
        (re.compile(r"(?P<term>[^.!?;:]{3,90}?)\s+(?:adalah|merupakan|ialah|yaitu|didefinisikan sebagai|diartikan sebagai|berarti)\s+(?P<body>[^.!?;]{8,220})"), "term_first"),
        (re.compile(r"(?P<body>[^.!?;:]{8,160}?)\s+(?:disebut sebagai|disebut|dinamakan|dinamai|dikenal sebagai|dikenal dengan istilah|dikenal dengan)\s+(?P<term>[^.!?;,]{3,60})"), "body_first"),
    ],
    "en": [
        (re.compile(r"(?P<term>[^.!?;:]{3,90}?)\s+(?:is defined as|are defined as|refers to|refer to|means)\s+(?P<body>[^.!?;]{8,220})"), "term_first"),
        (re.compile(r"(?P<term>[A-Z][^.!?;:]{2,50}?)\s+(?:is an?|is the|are the)\s+(?P<body>[^.!?;]{14,220})"), "term_first_weak"),
        (re.compile(r"(?P<body>[^.!?;:]{8,160}?)\s+(?:is called|are called|is known as|are known as|is termed|is named|is referred to as)\s+(?P<term>[^.!?;,]{3,60})"), "body_first"),
    ],
}
_TERM_STOP_START = {"ini", "itu", "dia", "ia", "mereka", "yang", "hal", "kita", "kami", "saya", "this", "that", "it", "he", "she",
                    "they", "which", "there", "these", "those", "menurut", "according", "namun", "tetapi", "but", "however"}
_TERM_LEADING_DETERMINERS = {"dalam", "pada", "secara", "di", "untuk", "sebuah", "suatu", "para", "sang", "the", "a", "an", "in", "for", "on", "by", "of"}
_CAPITALIZED_SEQ_RE = re.compile(
    r"\b[A-Z][\w'’-]+(?:\s+(?:(?:of|de|van|von|bin|al)\s+)?[A-Z][\w'’-]+){0,3}"
)
_ACRONYM_RE = re.compile(r"\b[A-Z]{2,6}(?:/[A-Z]{2,6})?\b")
_EXPANSION_RE = re.compile(r"((?:[A-Z][\w-]+\s+){1,5}[A-Z][\w-]+)\s*\(([A-Z]{2,8})\)")
_NOT_CONCEPT = {m.casefold() for m in _MONTHS.split("|")} | {
    "senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu", "monday", "tuesday", "wednesday", "thursday", "friday",
    "saturday", "sunday", "jika", "pada", "dalam", "untuk", "dengan", "selain", "kemudian", "selanjutnya", "berikutnya",
    "catatan", "contoh", "if", "in", "for", "with", "the", "this", "that", "however", "therefore", "moreover", "furthermore",
}
_KIND_GROUP = {"formula": "formula", "number": "number", "count": "number", "date": "date",
               "definition_term": "definition", "definition_body": "definition", "emphasis": "concept",
               "capitalized": "concept", "acronym": "concept", "expansion": "concept", "forced": "forced"}
_BASE_SCORE = {"formula": 1.0, "definition_term": 0.9, "emphasis": 0.85, "number": 0.8, "date": 0.8,
               "definition_body": 0.6, "expansion": 0.65, "capitalized": 0.5, "acronym": 0.45, "count": 0.4,
               "assigned_value": 0.7, "forced": 3.0}


def _normalize_key(text):
    return re.sub(r"[^\w²³%°]+", "", str(text).casefold())


def _clean_term(text, start, lang):
    """Rapikan span istilah: buang klausa pembuka sebelum koma terakhir dan kata penentu awal."""
    end = start + len(text)
    cut = text.rfind(", ")
    if cut >= 0:
        start += cut + 2
        text = text[cut + 2:]
    words = text.split()
    while len(words) > 1 and words[0].casefold() in _TERM_LEADING_DETERMINERS:
        offset = text.find(words[0]) + len(words[0])
        trimmed = len(text) - len(text[offset:].lstrip())
        start += trimmed
        text = text[trimmed:]
        words = text.split()
    stripped = text.strip(" ,;:()\"'“”")
    start += len(text) - len(text.lstrip(" ,;:()\"'“”"))
    words = stripped.split()
    if not words or len(words) > 6:
        return None
    if words[0].casefold() in _TERM_STOP_START or not (words[0][0].isalnum()):
        return None
    if len(stripped) < 3 or not re.search(r"[A-Za-zÀ-ÿ]", stripped):
        return None
    return start, start + len(stripped), stripped


def _trim_body(text, start):
    """Batasi definisi maksimal 14 kata, dipotong di tanda koma bila memungkinkan."""
    words = text.split()
    if len(words) > 14:
        cut_positions = [m.start() for m in re.finditer(r",\s", text) if 4 <= len(text[:m.start()].split()) <= 14]
        if cut_positions:
            text = text[:cut_positions[-1]]
        else:
            text = " ".join(words[:14])
    stripped = text.strip(" ,;:")
    if len(stripped.split()) < 3:
        return None
    return start, start + len(stripped), stripped


def _overlaps(a_start, a_end, b_start, b_end):
    return a_start < b_end and a_end > b_start


def extract_candidate_answers(context, lang, max_candidates=4, emphasized_terms=None, min_sentence_words=5):
    """Ekstrak kandidat answer terbaik dari satu context dengan pendekatan hibrida dan skor."""
    text = normalize_space(context)
    sentence_list = sentence_spans(text)

    def sentence_index(position):
        for i, (s, e) in enumerate(sentence_list):
            if s <= position < e + 1:
                return i
        return max(0, len(sentence_list) - 1)

    formula_spans = detect_formula_spans(text)
    candidates = []

    def add(kind, start, end, rule, bonus=0.0):
        value = text[start:end].strip()
        if not value:
            return
        start += len(text[start:end]) - len(text[start:end].lstrip())
        end = start + len(value)
        index = sentence_index(start)
        s_start, s_end = sentence_list[index] if sentence_list else (0, len(text))
        if len(text[s_start:s_end].split()) < min_sentence_words:
            return
        score = _BASE_SCORE.get("assigned_value" if rule == "assigned_value" else kind, 0.4) + bonus
        words = len(value.split())
        if words > 12:
            score *= 0.7
        candidates.append({"text": value, "start": start, "end": end, "kind": kind, "rule": rule, "score": score,
                           "sentence": index, "group": _KIND_GROUP.get(kind, "concept")})

    # 1. Rumus
    for start, end, value in formula_spans:
        inner_start, inner_end = start, end
        if value.startswith("$") and value.endswith("$") and len(value) > 2:
            inner_start, inner_end = start + 1, end - 1
        inner = text[inner_start:inner_end]
        if len(inner.strip()) < 3:
            continue
        assignment = re.fullmatch(r"([^=≈≤≥≠]+[=≈≤≥≠]\s*)([-+−]?\d[\d.,]*\s*\S{0,12})", inner.strip())
        if assignment:
            # Pemberian nilai variabel ("p = 8 cm"): kandidat yang berguna adalah nilainya, bukan "rumus" p = 8 cm.
            value_start = inner_start + (len(inner) - len(inner.lstrip())) + len(assignment.group(1))
            add("number", value_start, value_start + len(assignment.group(2)), "assigned_value")
        else:
            add("formula", inner_start, inner_end, "equation")

    def inside_formula(start, end):
        return any(_overlaps(start, end, fs, fe) for fs, fe, _ in formula_spans)

    # 2. Angka bersatuan, tanggal, tahun
    for match in _NUMBER_UNIT_RE.finditer(text):
        if not inside_formula(match.start(), match.end()):
            add("number", match.start(), match.end(), "number_unit")
    date_spans = []
    for pattern in _DATE_SEARCH_RES:
        for match in pattern.finditer(text):
            date_spans.append((match.start(), match.end()))
            add("date", match.start(), match.end(), "date_full")
    for match in _YEAR_SEARCH_RE.finditer(text):
        before = text[max(0, match.start() - 14):match.start()]
        if any(_overlaps(match.start(), match.end(), ds, de) for ds, de in date_spans):
            continue
        if _YEAR_CONTEXT_RE.search(before) and not inside_formula(match.start(), match.end()):
            add("date", match.start(), match.end(), "year")
    for match in _COUNT_NUMBER_RE.finditer(text):
        following = match.group(2).casefold()
        if following in _NOT_CONCEPT or inside_formula(match.start(), match.end()):
            continue
        if re.fullmatch(_YEAR, match.group(1)):
            continue
        if _NUMBER_UNIT_RE.match(text, match.start()):
            continue
        add("count", match.start(1), match.end(2), "count_noun")

    # 3. Definisi dan istilah kunci
    for index, (s_start, s_end) in enumerate(sentence_list):
        sentence = text[s_start:s_end]
        for pattern, mode in _DEFINITION_RULES.get(lang, []):
            match = pattern.search(sentence)
            if not match:
                continue
            term = _clean_term(match.group("term"), s_start + match.start("term"), lang)
            body = _trim_body(match.group("body"), s_start + match.start("body"))
            term_weight = 0.0 if mode == "term_first_weak" else 0.0
            if term and not inside_formula(term[0], term[1]):
                if mode == "term_first_weak" and (len(term[2].split()) > 4 or not term[2][0].isupper()):
                    pass
                else:
                    add("definition_term", term[0], term[1], mode, bonus=-0.25 if mode == "term_first_weak" else term_weight)
            if body and not inside_formula(body[0], body[1]):
                add("definition_body", body[0], body[1], mode)
            break

    for term in (emphasized_terms or []):
        term = normalize_space(term)
        if len(term) < 3:
            continue
        for match in re.finditer(re.escape(term), text):
            if not inside_formula(match.start(), match.end()):
                add("emphasis", match.start(), match.end(), "emphasis")
                break
    for match in _EXPANSION_RE.finditer(text):
        add("expansion", match.start(1), match.end(1), "expansion")
    sentence_starts = {s for s, _ in sentence_list}
    for match in _CAPITALIZED_SEQ_RE.finditer(text):
        value = match.group(0)
        words = value.split()
        if inside_formula(match.start(), match.end()):
            continue
        if all(w.casefold() in _NOT_CONCEPT for w in words):
            continue
        if len(words) == 1 and (match.start() in sentence_starts or value.casefold() in _NOT_CONCEPT or len(value) < 4):
            continue
        if words[0].casefold() in _NOT_CONCEPT and len(words) > 1:
            offset = len(words[0]) + 1
            add("capitalized", match.start() + offset, match.end(), "capitalized", bonus=0.1 * (len(words) - 2))
        else:
            add("capitalized", match.start(), match.end(), "capitalized", bonus=0.1 * (len(words) - 1))
    for match in _ACRONYM_RE.finditer(text):
        if not inside_formula(match.start(), match.end()):
            add("acronym", match.start(), match.end(), "acronym")

    # Skor dan penyaringan duplikat teks
    best_by_key = {}
    for cand in candidates:
        key = _normalize_key(cand["text"])
        if len(key) < 2:
            continue
        if key not in best_by_key or cand["score"] > best_by_key[key]["score"]:
            best_by_key[key] = cand
    pool = sorted(best_by_key.values(), key=lambda c: -c["score"])

    def select(per_sentence_cap, already=None):
        chosen = list(already or [])
        used_sentence = Counter(c["sentence"] for c in chosen)
        used_group = Counter(c["group"] for c in chosen)
        remaining = [c for c in pool if c not in chosen]
        while remaining and len(chosen) < max_candidates:
            best = None
            for cand in remaining:
                if any(_overlaps(cand["start"], cand["end"], c["start"], c["end"]) for c in chosen):
                    continue
                if used_sentence[cand["sentence"]] >= per_sentence_cap:
                    continue
                adjusted = cand["score"] + (0.15 if used_group[cand["group"]] == 0 else 0.0) - 0.3 * used_sentence[cand["sentence"]]
                if best is None or adjusted > best[0]:
                    best = (adjusted, cand)
            if best is None:
                break
            chosen.append(best[1])
            used_sentence[best[1]["sentence"]] += 1
            used_group[best[1]["group"]] += 1
            remaining.remove(best[1])
        return chosen

    chosen = []
    for per_sentence_cap in (1, 2, 3, 4):
        chosen = select(per_sentence_cap, chosen)
        if len(chosen) >= max_candidates:
            break
    chosen.sort(key=lambda c: c["start"])
    for cand in chosen:
        cand["answer_type"] = answer_type(cand["text"])
    return chosen


# =============================================================================
# BAGIAN C. Generator distractor (pengecoh) per tipe jawaban
# =============================================================================
_MONTH_LIST = {
    "id": ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli", "Agustus", "September", "Oktober", "November", "Desember"],
    "en": ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
}
_NUMBER_PARTS_RE = re.compile(r"^(?P<pre>[^\d\-+−]*?)(?P<sign>[-+−]?)(?P<num>\d[\d.,]*)(?P<post>.*)$", re.DOTALL)


def _parse_number_token(token, lang):
    """Urai '1.000,5' atau '1,000.5' menjadi (nilai, jumlah_desimal, pemisah_ribuan, pemisah_desimal)."""
    dots, commas = token.count("."), token.count(",")
    grouping, decimal_sep = "", ""
    if dots and commas:
        decimal_sep = "." if token.rfind(".") > token.rfind(",") else ","
        grouping = "," if decimal_sep == "." else "."
    elif dots or commas:
        sep = "." if dots else ","
        count = dots or commas
        index = token.rfind(sep)
        digits_after = len(token) - index - 1
        integer_part = token[:index].replace(".", "").replace(",", "")
        if count >= 2:
            grouping = sep
        elif digits_after == 3 and integer_part not in ("0", "") and len(integer_part) <= 3:
            grouping = sep
        else:
            decimal_sep = sep
    integer_text = token.replace(grouping, "") if grouping else token
    if decimal_sep:
        integer_text = integer_text.replace(decimal_sep, ".")
    decimals = len(integer_text.split(".")[1]) if "." in integer_text else 0
    return float(integer_text), decimals, grouping, decimal_sep


def _format_number_like(value, decimals, grouping, decimal_sep, lang):
    if decimals == 0 and abs(value - round(value)) > 1e-9:
        decimals = 1 if abs(value) < 10 else 0
    value = round(value, decimals)
    text = f"{abs(value):.{decimals}f}"
    integer_text, _, fraction = text.partition(".")
    if grouping:
        groups = []
        while len(integer_text) > 3:
            groups.insert(0, integer_text[-3:])
            integer_text = integer_text[:-3]
        groups.insert(0, integer_text)
        integer_text = grouping.join(groups)
    if fraction:
        integer_text += (decimal_sep or ("," if lang == "id" else ".")) + fraction
    return ("-" if value < 0 else "") + integer_text


def _number_distractors(answer, lang, rng, k):
    match = _NUMBER_PARTS_RE.match(answer.strip())
    if not match:
        return []
    value, decimals, grouping, decimal_sep = _parse_number_token(match.group("num"), lang)
    if match.group("sign") in ("-", "−"):
        value = -value
    prefix, post = match.group("pre"), match.group("post")
    step = 10 ** math.floor(math.log10(abs(value))) if value else 1
    ordered = [value * 2, value / 2, value + step, value - step, value * 1.5, value * 10, value / 10, value * 0.8, value * 1.25, value + 2 * step]
    seen = {answer.strip()}
    integer_original = decimals == 0 and abs(value - round(value)) < 1e-9

    def collect(allow_fraction):
        found = []
        for candidate in ordered:
            if value > 0 and candidate <= 0:
                continue
            if integer_original and not allow_fraction and abs(candidate - round(candidate)) > 1e-9:
                continue
            formatted = _format_number_like(candidate, decimals, grouping, decimal_sep, lang)
            if value != 0 and float(formatted.replace(grouping, "").replace(decimal_sep or ".", ".").replace("−", "-") or 0) == 0:
                continue
            text = prefix + formatted + post
            if text not in seen and text not in found:
                found.append(text)
        return found

    results = collect(False)
    if len(results) < k:
        results = results + [t for t in collect(True) if t not in results]
    return results[:k]


def _date_distractors(answer, lang, rng, k):
    value = answer.strip()
    if re.fullmatch(_YEAR, value):
        year = int(value)
        shifts = []
        for magnitude in (1, 2, 5, 3, 10):
            sign = rng.choice([-1, 1])
            for signed in (sign * magnitude, -sign * magnitude):
                if 1000 <= year + signed <= 2100 and signed not in shifts:
                    shifts.append(signed)
                    break
        return [str(year + s) for s in shifts[:k]]
    results = []
    year_match = re.search(_YEAR, value)
    month_match = re.search(r"(?:" + _MONTHS + r")", value, flags=re.IGNORECASE)
    if year_match:
        for delta in (1, -2, 5):
            candidate = value[:year_match.start()] + str(int(year_match.group(0)) + delta) + value[year_match.end():]
            if candidate != value:
                results.append(candidate)
    if month_match:
        months = _MONTH_LIST["id" if lang == "id" else "en"]
        current = month_match.group(0).casefold()
        choices = [m for m in months if not m.casefold().startswith(current[:3])]
        candidate = value[:month_match.start()] + rng.choice(choices) + value[month_match.end():]
        results.insert(1, candidate)
    day_match = re.match(r"^(\d{1,2})(\s+[A-Za-z])", value)
    if day_match:
        day = int(day_match.group(1))
        new_day = day + 5 if day <= 20 else day - 5
        results.append(str(new_day) + value[day_match.end(1):])
    century = re.search(r"(abad\s+ke-?\s*)(\d+)|(\d{1,2})((?:st|nd|rd|th)\s+century)", value, flags=re.IGNORECASE)
    if century and not results:
        number = int(century.group(2) or century.group(3))
        for delta in (1, -1, 2):
            if number + delta >= 1:
                results.append(value.replace(str(number), str(number + delta), 1))
    unique = []
    for item in results:
        if item != value and item not in unique:
            unique.append(item)
    return unique[:k]


_FORMULA_OP_SWAPS = {"+": ["-", "×"], "-": ["+"], "−": ["+"], "×": ["/", "+"], "*": ["/", "+"], "/": ["×", "-"], "÷": ["×", "+"], "·": ["/", "+"]}
_FORMULA_OPERATOR_RE = re.compile(r"(?<=[\w)\]}²³])\s*([+\-−×*/÷·])\s*(?=[\w(\[{√∑\\])")
_FORMULA_VARIABLE_RE = re.compile(r"(?<![\\A-Za-z])(?:[A-Z]{2,3}|[A-Za-zΑ-Ωα-ω](?:_\{?\w{1,3}\}?)?)(?![A-Za-z])")


def _split_relation(formula):
    depth = 0
    for index, ch in enumerate(formula):
        if ch in "({[":
            depth += 1
        elif ch in ")}]":
            depth -= 1
        elif ch == "=" and depth == 0:
            return formula[:index], formula[index + 1:]
    return None, formula


def _formula_canonical(formula):
    """Bentuk kanonik untuk mendeteksi rumus yang secara aljabar identik (komutatif)."""
    left, right = _split_relation(formula)
    operators = set(re.findall(r"[+\-−×*/÷·^]", re.sub(r"\d+/\d+", "N", right)))
    tokens = re.findall(r"\d+/\d+|[A-Z]{2,3}|[A-Za-zΑ-Ωα-ω](?:_\{?\w{1,3}\}?)?[²³]?|\d+(?:[.,]\d+)?", right)
    if operators and (operators <= {"×", "*", "·"} or operators <= {"+"}):
        return (normalize_space(left or ""), "".join(sorted(tokens)), "".join(sorted(operators)))
    return None


def _balanced(text):
    stack = []
    pairs = {")": "(", "]": "[", "}": "{"}
    for ch in text:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack


def _formula_distractors(answer, rng, k):
    original = normalize_space(answer)
    canon = _formula_canonical(original)
    left, right = _split_relation(original)
    offset = len(original) - len(right)
    tiers = {"op": [], "var": [], "pow": [], "fn": []}

    for match in _FORMULA_OPERATOR_RE.finditer(original):
        if match.start(1) < offset:
            continue
        for replacement in _FORMULA_OP_SWAPS.get(match.group(1), []):
            tiers["op"].append(original[:match.start(1)] + replacement + original[match.end(1):])

    names = []
    for match in _FORMULA_VARIABLE_RE.finditer(original):
        if match.group(0) not in names and match.group(0) != "d":
            names.append(match.group(0))
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            placeholder = "\u0000"
            swapped = re.sub(_FORMULA_VARIABLE_RE, lambda m: b if m.group(0) == a else (a if m.group(0) == b else m.group(0)), original)
            tiers["var"].append(swapped)

    for old, new in (("²", "³"), ("³", "²"), ("^2", "^3"), ("^{2}", "^{3}"), ("^3", "^2"), ("^{3}", "^{2}")):
        if old in original:
            tiers["pow"].append(original.replace(old, new, 1))
    if "²" in original:
        tiers["pow"].append(original.replace("²", "", 1))
    if "1/2 × " in original:
        tiers["pow"].append(original.replace("1/2 × ", "", 1))
        tiers["pow"].append(original.replace("1/2", "2", 1))
    for old, new in (("= -log", "= log"), ("log", "ln"), ("= -ln", "= ln"), ("sin", "cos"), ("cos", "sin"), ("√", "")):
        if old in original:
            tiers["fn"].append(original.replace(old, new, 1))
    frac = re.search(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", original)
    if frac:
        tiers["pow"].append(original[:frac.start()] + "\\frac{" + frac.group(2) + "}{" + frac.group(1) + "}" + original[frac.end():])

    def acceptable(candidate):
        candidate = normalize_space(candidate)
        if candidate == original or not _balanced(candidate):
            return False
        if re.search(r"[+\-−×*/÷·]\s*[×*/÷·]", candidate) or re.search(r"/\s*[\w.]+\s*/", candidate):
            return False
        if canon is not None and _formula_canonical(candidate) == canon:
            return False
        return True

    pools = {}
    for tier, items in tiers.items():
        unique = []
        for item in items:
            item = normalize_space(item)
            if acceptable(item) and item not in unique:
                unique.append(item)
        rng.shuffle(unique)
        pools[tier] = unique
    results = []
    while len(results) < k and any(pools.values()):
        for tier in ("op", "var", "pow", "fn"):
            if pools[tier] and len(results) < k:
                candidate = pools[tier].pop(0)
                if candidate not in results:
                    results.append(candidate)
    return results


def _match_case(candidate, reference):
    if not candidate or not reference:
        return candidate
    if reference[0].islower() and candidate[0].isupper() and not candidate.isupper() and candidate[1:2].islower():
        return candidate[0].lower() + candidate[1:]
    if reference[0].isupper() and candidate[0].islower():
        return candidate[0].upper() + candidate[1:]
    return candidate


def _text_distractors(answer, context, lang, pool, rng, k, answer_kind=None):
    answer_tokens = set(word_tokens(answer))
    answer_words = len(answer.split())
    answer_key = _normalize_key(answer)
    texts = []
    answer_is_body = answer_kind == "definition_body"

    def same_shape(kind):
        # Frasa definisi panjang tidak boleh bercampur dengan istilah pendek: bentuknya membocorkan kunci.
        return answer_kind is None or kind is None or (kind == "definition_body") == answer_is_body

    for item in (pool or []):
        if isinstance(item, dict):
            if item.get("answer_type", "text") != "text" or not same_shape(item.get("kind")):
                continue
            texts.append((item["text"], item.get("context_id")))
        else:
            texts.append((str(item), None))
    for local in extract_candidate_answers(context, lang, max_candidates=12):
        if local["answer_type"] == "text" and same_shape(local["kind"]):
            texts.append((local["text"], "same_context"))
    scored, seen = [], set()
    for text_value, origin in texts:
        key = _normalize_key(text_value)
        if not key or key == answer_key or key in seen or key in answer_key or answer_key in key:
            continue
        tokens = set(word_tokens(text_value))
        union = answer_tokens | tokens
        if union and len(answer_tokens & tokens) / len(union) >= 0.34:
            continue
        words = len(text_value.split())
        if abs(words - answer_words) > max(1, math.ceil(0.5 * answer_words)):
            continue
        seen.add(key)
        penalty = abs(words - answer_words) * 0.3 + (0.0 if text_value[:1].isupper() == answer[:1].isupper() else 0.6)
        scored.append((penalty + rng.random() * 0.4, text_value))
    scored.sort()
    return [_match_case(text_value, answer) for _, text_value in scored[:k]]


def generate_distractors(answer, kind, context, lang, pool=None, rng=None, k=3, answer_kind=None):
    """Hasilkan hingga k pengecoh yang meyakinkan untuk jawaban bertipe formula, number, date, atau text."""
    rng = rng or random.Random(0)
    answer = normalize_space(answer)
    if kind == "formula":
        options = _formula_distractors(answer, rng, k)
    elif kind == "number":
        options = _number_distractors(answer, lang, rng, k)
    elif kind == "date":
        options = _date_distractors(answer, lang, rng, k)
        if len(options) < k:
            options = options + [o for o in _text_distractors(answer, context, lang, pool, rng, k, answer_kind) if o not in options]
    else:
        options = _text_distractors(answer, context, lang, pool, rng, k, answer_kind)
    unique = []
    for option in options:
        option = normalize_space(option)
        if option and option.casefold() != answer.casefold() and option not in unique:
            unique.append(option)
    return unique[:k]


# =============================================================================
# BAGIAN D. Perakit kuis end-to-end (pilihan ganda dan esai)
# =============================================================================
def _jaccard(a, b):
    sa, sb = set(word_tokens(a)), set(word_tokens(b))
    return len(sa & sb) / max(1, len(sa | sb))


def build_quiz_from_material(raw_text_or_slides, lang=None, max_questions=None, max_per_context=4,
                             quiz_type="mixed", forced_answers=None, question_fn=None, seed=42):
    """
    Bangun kuis dari paragraf atau poin slide tanpa answer manual.
    quiz_type: "mixed" (otomatis), "mcq" (hanya pilihan ganda), atau "essay".
    forced_answers: daftar string answer yang harus dicoba bila ada di context (untuk uji regresi).
    question_fn: fungsi (context, answer, lang, answer_start) -> dict hasil rerank; default generate_best_question.
    """
    question_fn = question_fn or generate_best_question
    rng = random.Random(seed)
    contexts = material_to_contexts(raw_text_or_slides, lang)
    prepared = []
    for index, item in enumerate(contexts):
        context = item["context"]
        candidates = extract_candidate_answers(context, item["lang"], max_candidates=max_per_context)
        for forced in (forced_answers or []):
            span = locate_answer(context, forced, -1)
            if span and not any(c["text"] == context[span[0]:span[1]] for c in candidates):
                text = context[span[0]:span[1]]
                candidates.append({"text": text, "start": span[0], "end": span[1], "kind": "forced", "rule": "forced",
                                   "score": 3.0, "sentence": 0, "group": "forced", "answer_type": answer_type(text)})
        prepared.append((index, item, candidates))

    pool = []
    for index, item, candidates in prepared:
        for cand in candidates:
            pool.append({"text": cand["text"], "answer_type": cand["answer_type"], "context_id": index, "kind": cand["kind"]})

    items, rejected, seen_questions = [], [], []
    for index, context_item, candidates in prepared:
        context, context_lang = context_item["context"], context_item["lang"]
        for cand in candidates:
            result = question_fn(context, cand["text"], context_lang, cand["start"])
            question = result.get("question")
            if not question or not result.get("ok", False):
                rejected.append({"answer": cand["text"], "context": context[:90], "reasons": result.get("reasons", ["no_question"]),
                                 "best_rejected_question": question})
                continue
            if any(question.casefold() == prev.casefold() or _jaccard(question, prev) >= 0.85 for prev in seen_questions):
                rejected.append({"answer": cand["text"], "context": context[:90], "reasons": ["duplicate_question"],
                                 "best_rejected_question": question})
                continue
            kind = cand["answer_type"]
            other_pool = [p for p in pool if p["context_id"] != index or p["text"] != cand["text"]]
            distractors = generate_distractors(cand["text"], kind, context, context_lang, pool=other_pool, rng=rng, k=3, answer_kind=cand["kind"])
            distractors = [d for d in distractors if d.casefold() not in question.casefold()]
            wants_mcq = quiz_type == "mcq" or (quiz_type == "mixed" and len(cand["text"].split()) <= 8)
            entry = {
                "id": f"Q{len(items) + 1:03d}", "language": context_lang, "question": question, "answer": cand["text"],
                "answer_type": kind, "candidate_rule": cand["rule"], "context": context, "context_title": context_item["title"],
                "low_confidence": bool(result.get("low_confidence", False)),
            }
            if wants_mcq and len(distractors) >= 3:
                options = [cand["text"]] + distractors[:3]
                rng.shuffle(options)
                letters = ["A", "B", "C", "D"]
                entry.update({
                    "type": "pilihan_ganda",
                    "options": [{"label": letters[i], "text": options[i]} for i in range(4)],
                    "correct_option": letters[options.index(cand["text"])],
                })
            else:
                if quiz_type == "mcq":
                    rejected.append({"answer": cand["text"], "context": context[:90], "reasons": ["distractor_kurang"],
                                     "best_rejected_question": question})
                    continue
                entry.update({"type": "esai", "options": None, "correct_option": None, "reference_answer": cand["text"]})
            items.append(entry)
            seen_questions.append(question)
            if max_questions and len(items) >= max_questions:
                break
        if max_questions and len(items) >= max_questions:
            break
    return {
        "meta": {"n_contexts": len(prepared), "n_questions": len(items), "n_rejected": len(rejected),
                 "quiz_type": quiz_type, "languages": sorted({i["language"] for i in items})},
        "items": items, "rejected": rejected,
    }


def print_quiz(quiz, show_context=False):
    meta = quiz["meta"]
    print(f"Konteks: {meta['n_contexts']} | Soal: {meta['n_questions']} | Ditolak filter: {meta['n_rejected']}")
    for item in quiz["items"]:
        flag = "  [kepercayaan rendah]" if item.get("low_confidence") else ""
        print(f"\n{item['id']} [{item['type']}|{item['answer_type']}|{item['language']}]{flag}")
        if show_context:
            print("   Konteks:", item["context"])
        print("   ", item["question"])
        if item["options"]:
            for option in item["options"]:
                marker = "  <- kunci" if option["label"] == item["correct_option"] else ""
                print(f"      {option['label']}. {option['text']}{marker}")
        else:
            print("      Jawaban acuan:", item["reference_answer"])


# =============================================================================
# Demonstrasi tiga kasus uji (model dimuat dari FINAL_DIR pada Cell 10)
# =============================================================================
BLIND_SPOT_QUESTIONS = {"siapa pendiri ?", "siapa pendiri?", "kapankah lahir ?", "kapankah lahir?"}


def regression_report(context, lang, answers):
    """Bandingkan keluaran beam biasa dengan keluaran setelah rerank untuk answer yang menjadi bukti bug Fase 2."""
    records = []
    for answer in answers:
        span = locate_answer(context, answer, -1)
        start = span[0] if span else -1
        row = {"lang": lang, "context": context, "answer": answer, "answer_start": start, "answer_type": answer_type(answer),
               "has_formula": detect_formula_content(context)[0], "question": "", "role": "", "group_id": "", "eval_split": ""}
        plain = predict_rows([row])[0]
        best = generate_best_question(context, answer, lang, start)
        records.append({
            "answer": answer, "pertanyaan_beam_biasa": plain, "lolos_filter_biasa": question_quality(plain, context, answer, lang)[0],
            "pertanyaan_setelah_rerank": best["question"], "lolos_filter_rerank": best["ok"],
        })
    return pd.DataFrame(records)


pd.set_option("display.max_colwidth", 160)

print("=" * 78)
print("KASUS UJI 1. Paragraf umum: Candi Borobudur (bukti bug Fase 2)")
print("=" * 78)
borobudur_text = (
    "Candi Borobudur adalah candi Buddha di Magelang, Jawa Tengah. Candi ini dibangun pada abad ke-9 pada masa "
    "pemerintahan Dinasti Syailendra dan ditetapkan sebagai Situs Warisan Dunia UNESCO pada tahun 1991. "
    "Candi ini memiliki 504 arca Buddha dan 72 stupa berlubang."
)
case1_report = regression_report(normalize_space(borobudur_text), "id", ["Dinasti Syailendra", "1991"])
display(case1_report)
for _, row in case1_report.iterrows():
    buntung = normalize_space(str(row["pertanyaan_beam_biasa"])).casefold() in BLIND_SPOT_QUESTIONS
    print(f"  {row['answer']!r}: pertanyaan buntung persis seperti Fase 2? {'YA' if buntung else 'tidak'} | "
          f"lolos filter (beam biasa)={row['lolos_filter_biasa']} | lolos filter (rerank)={row['lolos_filter_rerank']}")
quiz_case1 = build_quiz_from_material(borobudur_text, lang="id", forced_answers=["Dinasti Syailendra", "1991"], max_per_context=5)
print_quiz(quiz_case1)

print("\n" + "=" * 78)
print("KASUS UJI 2. Materi rumus (Indonesia dan Inggris): rumus vs angka pada konteks yang sama")
print("=" * 78)
formula_materials = [
    ("id", "Luas persegi panjang dihitung dengan rumus A = p × l. Jika panjang p = 8 cm dan lebar l = 5 cm, maka luasnya adalah 40 cm².",
     ["A = p × l", "40 cm²"]),
    ("en", "Ohm's law states that voltage is given by V = I × R. If current I = 2 A and resistance R = 6 Ω, then the voltage is 12 V.",
     ["V = I × R", "12 V"]),
    ("id", "Hukum Ohm menyatakan bahwa tegangan dinyatakan dengan persamaan V = I × R. Jika kuat arus I = 2 A dan hambatan R = 6 Ω, maka tegangan adalah 12 V.",
     ["V = I × R", "12 V"]),
]
quiz_case2_parts = []
for lang, text, answers in formula_materials:
    report = regression_report(normalize_space(text), lang, answers)
    display(report[["answer", "pertanyaan_beam_biasa", "pertanyaan_setelah_rerank", "lolos_filter_rerank"]])
    distinct = report["pertanyaan_beam_biasa"].nunique() == len(report)
    print(f"  Pertanyaan rumus dan angka berbeda (beam biasa)? {'YA' if distinct else 'TIDAK, mode collapse masih ada'}")
    quiz_case2_parts.append(build_quiz_from_material(text, lang=lang, forced_answers=answers, max_per_context=4))
for part in quiz_case2_parts:
    print_quiz(part)

print("\n" + "=" * 78)
print("KASUS UJI 3. Poin-poin slide PPT kuliah (bahasa campuran per slide)")
print("=" * 78)
slide_material = """Hukum Newton
- Gaya: dorongan atau tarikan pada benda
- Hukum II Newton menyatakan F = m × a
- Benda 5 kg dengan percepatan 2 m/s² mengalami gaya 10 N

Hukum Newton (lanjutan)
- Hukum III Newton: aksi sama dengan reaksi

Photosynthesis
- Photosynthesis is the process by which green plants make food
- It occurs in the chloroplasts and releases oxygen
- First described in 1779 by Jan Ingenhousz
"""
print("Konteks hasil penggabungan slide:")
for item in material_to_contexts(slide_material):
    print(f"  [{item['lang']}] {item['context']}")
quiz_case3 = build_quiz_from_material(slide_material, max_per_context=4)
print_quiz(quiz_case3)

sample_quiz = {"kasus_1_paragraf_borobudur": quiz_case1, "kasus_2_rumus": quiz_case2_parts, "kasus_3_slide": quiz_case3,
               "regresi_blind_spot": case1_report.to_dict("records")}
with open(os.path.join(FINAL_DIR, "sample_generated_quiz.json"), "w", encoding="utf-8") as handle:
    json.dump(sample_quiz, handle, indent=2, ensure_ascii=False, default=str)
print("\nKuis demo disimpan ke", os.path.join(FINAL_DIR, "sample_generated_quiz.json"))


## Cell 12. Auto-packaging menjadi ZIP siap unduh

Mengemas `FINAL_DIR` dan `EDA_DIR` ke satu ZIP (`ZIP_DEFLATED`), memverifikasi integritas dengan `testzip()`, menampilkan daftar isi beserta ukuran dalam MB, dan membuat tautan unduh. Berkas yang tidak dihasilkan oleh versi transformers yang dipakai ditandai `tidak ada`, bukan dianggap error.

In [ ]:
# =============================================================================
# CELL 12. Auto-packaging menjadi satu file ZIP siap unduh
# =============================================================================
try:
    del infer_model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

EXPECTED_FINAL_FILES = [
    "config.json", "generation_config.json", "tokenizer.json", "spiece.model", "special_tokens_map.json",
    "training_metrics.json", "answer_sensitivity_audit.json", "sample_generated_quiz.json",
    "stage1_log_history.csv", "stage2_log_history.csv",
]
present_files = set(os.listdir(FINAL_DIR))
weights_present = any(name.startswith("model") and (name.endswith(".safetensors") or name.endswith(".bin")) for name in present_files)
print("Pemeriksaan isi FINAL_DIR:")
print(f"  [{'OK' if weights_present else 'HILANG'}] bobot model (model*.safetensors)")
for name in EXPECTED_FINAL_FILES:
    print(f"  [{'OK' if name in present_files else 'tidak ada'}] {name}")
print("  (file yang 'tidak ada' dapat memang tidak dihasilkan oleh versi transformers yang dipakai)")


def build_zip(zip_path, sources):
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for source_dir, prefix in sources:
            for root, _, files in os.walk(source_dir):
                for name in sorted(files):
                    full_path = os.path.join(root, name)
                    archive.write(full_path, os.path.join(prefix, os.path.relpath(full_path, source_dir)))


started = time.time()
build_zip(ZIP_PATH, [(FINAL_DIR, "mt5-base-aqg-final"), (EDA_DIR, "eda_and_eval_plots")])
print(f"\nZIP dibuat dalam {time.time() - started:.0f} detik: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH) as archive:
    corrupted = archive.testzip()
    assert corrupted is None, f"ZIP rusak pada berkas: {corrupted}"
    listing = pd.DataFrame(
        [{"file": info.filename, "ukuran_MB": round(info.file_size / 1024 ** 2, 3), "terkompresi_MB": round(info.compress_size / 1024 ** 2, 3)}
         for info in archive.infolist()]
    )
print("Integritas ZIP: OK (testzip tidak menemukan berkas rusak)")
display(listing)
print(f"Total {len(listing)} berkas | ukuran asli {listing['ukuran_MB'].sum():,.1f} MB | ukuran ZIP {os.path.getsize(ZIP_PATH) / 1024 ** 2:,.1f} MB")
if FileLink is not None:
    display(FileLink(ZIP_PATH))
